# PILCO — apprendre une politique avec une poignée d'essais

> **PILCO** = *Probabilistic Inference for Learning COntrol* ([Deisenroth & Rasmussen, 2011](https://mlg.eng.cam.ac.uk/pub/pdf/DeiRas11.pdf)).
>
> C'est l'algorithme de RL **le plus efficace en données** que nous verrons : il apprend à
> balancer un pendule en **quelques dizaines de secondes** d'interaction réelle, là où un
> algorithme model-free (DQN, SAC) en demande des heures.

Ce notebook est le **premier** d'un diptyque :

| Notebook | Sujet | Modèle du monde |
|----------|-------|-----------------|
| **10 — PILCO** (celui-ci) | Fondations + méthode analytique | **Gaussian Process** |
| 10b — Deep PILCO | Passage à l'échelle | Bayesian Neural Network |

PILCO empile beaucoup de briques mathématiques (processus gaussiens, kernels, propagation
analytique d'incertitude, politique intégrable). Chacune sera **introduite comme un petit cours** :
intuition d'abord, puis l'équation expliquée terme par terme, puis du code, une visualisation, et
une vérification. À la fin, vous devriez pouvoir réexpliquer *pourquoi* chaque pièce existe.

## 1. Pourquoi un modèle *probabiliste* du monde ?

Reprenons la grande idée du **model-based RL** (vue avec Dyna au notebook [09](./09_dyna_q_dyna_q_plus_deep_dyna_walkthrough.ipynb)) : au lieu de jeter
chaque transition après usage, on **apprend un modèle de la dynamique** $$f : (x_t, u_t) \mapsto
x_{t+1}$$ puis on *planifie* dans ce modèle pour améliorer la politique sans toucher au monde réel.

Dyna utilisait un modèle **déterministe** (une table, puis un réseau). PILCO fait un choix
différent et crucial : son modèle ne dit pas seulement *« voici ce qui va arriver »*, il dit aussi
**« voici à quel point j'en suis sûr »**.

**Analogie du pilote dans le brouillard.** Un modèle déterministe affirme : *« si tu tires le
manche de 3°, l'avion monte de 2 m »*. PILCO dit plutôt : *« je pense qu'il montera de 2 m, mais
ça pourrait être 1,2 ou 2,8 »*. Cette hésitation honnête est ce qui change tout : une petite erreur
de modèle, répétée sur 30 pas de planification, peut diverger énormément. Si l'agent **sait** où
son modèle est fiable et où il extrapole à l'aveugle, il évite de bâtir un plan sur du sable.

C'est toute l'architecture de PILCO qui découle de cette idée : *propager l'incertitude*, et
*optimiser une politique qui reste bonne en moyenne sur cette incertitude*.

## 2. La boucle PILCO en trois verbes

```mermaid
flowchart TB
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    env["Environnement réel"]

    subgraph loop["Boucle PILCO"]
        observe["1. OBSERVER<br/>Collecter quelques rollouts réels<br/>$$~(x, u) \to x'$$"]
        imagine["2. IMAGINER<br/>Apprendre un GP probabiliste<br/>$$~p(x' | x, u)$$<br/>Propager l'incertitude analytiquement"]
        adjust["3. AJUSTER<br/>Minimiser $$~J(\pi)$$<br/>Mettre à jour la politique"]
    end

    env -->|"rollouts réels"| observe
    observe -->|"dataset de transitions"| imagine
    imagine -->|"coût espéré $$~J(\pi)$$"| adjust
    adjust -->|"nouvelle politique$$~\pi$$"| env
```

1. **Observer** quelques transitions réelles.
2. **Imaginer** l'avenir *avec son incertitude* à travers le modèle.
3. **Ajuster** la politique pour réduire le coût futur prévu.

Si une équation paraît obscure dans la suite, la bonne question est toujours : **à quoi sert-elle
dans cette boucle ?**

Le plan du notebook suit la logique de construction : on bâtit d'abord le **modèle probabiliste**
(le Gaussian Process, sections 1 à 4 du cours), puis la **propagation d'incertitude** et la
**politique** (section 5), et enfin on **assemble la boucle** (section 6).

In [ ]:
import math
from pathlib import Path

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video, display
from matplotlib import cm
from matplotlib.patches import Ellipse
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import gaussian_kde, norm
from tqdm.auto import tqdm

# PILCO est numériquement sensible : on travaille en float64 partout.
torch.set_default_dtype(torch.float64)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

print("Environnement prêt — torch", torch.__version__, "| dtype", torch.get_default_dtype())


---
# Partie I — Le Gaussian Process, brique par brique

Tout PILCO repose sur le **Gaussian Process (GP)**. Mais on ne peut pas comprendre un GP sans
d'abord être parfaitement à l'aise avec **la loi normale** — d'abord en 1D, puis en plusieurs
dimensions, et surtout avec l'opération de **conditionnement**. On construit donc l'échelle
barreau par barreau, exactement dans l'ordre où les idées s'emboîtent.

## I.1 — La gaussienne en une dimension

Une variable $X$ suit une loi normale $\mathcal{N}(\mu, \sigma^2)$ si sa densité est :

$$p(x) = \frac{1}{\sqrt{2\pi}\,\sigma}\,\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

| Terme | Rôle |
|-------|------|
| $\mu$ | **moyenne** — le centre de la cloche, la valeur la plus probable |
| $\sigma^2$ | **variance** — l'étalement ; $\sigma$ est l'écart-type |
| $\frac{1}{\sqrt{2\pi}\sigma}$ | constante de **normalisation** (l'aire totale vaut 1) |
| $\exp(-(x-\mu)^2/2\sigma^2)$ | décroissance en cloche : plus on s'éloigne de $\mu$, plus c'est rare |

C'est l'objet le plus simple qui capture une idée fondamentale : **une croyance avec son
incertitude**. « Je pense $\mu$, à $\pm\sigma$ près ».

In [ ]:
def gaussian_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (np.sqrt(2 * np.pi) * sigma)

xs = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

# Gauche : la densité pour trois écarts-types
for sigma in [0.7, 1.0, 1.5, 2.5]:
    axes[0].plot(xs, gaussian_pdf(xs, 0.0, sigma), label=f"σ = {sigma}")
axes[0].set_title("Densité $\\mathcal{N}(0, \\sigma^2)$"); axes[0].set_xlabel("x"); axes[0].legend()

# Droite : 2000 échantillons sous la cloche
samples = np.random.normal(0.0, 1.5, size=2000)
axes[1].hist(samples, bins=40, density=True, alpha=0.5, color="steelblue", label="échantillons")
axes[1].plot(xs, gaussian_pdf(xs, 0.0, 1.5), "r", lw=2, label="densité exacte")
axes[1].set_title("2000 tirages de $\\mathcal{N}(0, 1.5^2)$"); axes[1].set_xlabel("x"); axes[1].legend()
plt.tight_layout(); plt.show()

**Lecture.** À gauche, $\sigma$ contrôle la largeur : petite variance = croyance « pointue »
(confiante), grande variance = croyance étalée (incertaine). À droite, l'histogramme de tirages
épouse presque exactement la densité — la formule *décrit* bien le processus aléatoire. Retenez cette
dualité **densité ↔ échantillons** : on s'en servira sans cesse pour *vérifier* visuellement nos
calculs analytiques (les fameux « MC vs analytique »).

## I.2 — D'une variable à un vecteur : pourquoi la corrélation ?

Notre but final est de modéliser une **fonction** $f(x)$. Une fonction, c'est une valeur en
chaque point d'entrée : $f(x_1), f(x_2), \dots, f(x_n)$ — autrement dit un **vecteur** de valeurs.
On va donc modéliser ce vecteur par une loi normale… mais laquelle ?

Il faut faire un petit changement de regard.

Pour une variable scalaire, on tire un point sur l'axe vertical :

$$
y \sim \mathcal N(0,1).
$$

Mais pour une fonction, les entrées $x_1,\dots,x_n$ sont fixées sur l'axe horizontal, et on tire
simultanément les valeurs verticales :

$$
\mathbf f =
\begin{bmatrix}
f(x_1) \\
f(x_2) \\
\vdots \\
f(x_n)
\end{bmatrix}.
$$

Autrement dit, l'axe $x$ ne représente pas une variable aléatoire ici : il représente les
**positions où l'on évalue la fonction**. Ce qui est aléatoire, ce sont les hauteurs
$f(x_i)$ au-dessus de ces positions.

**Première tentative naïve : des gaussiennes indépendantes.** Si chaque $f(x_i)$ est tiré
indépendamment selon $\mathcal{N}(0,1)$, on obtient :

$$
f(x_i) \sim \mathcal N(0,1)
\quad\text{indépendamment pour chaque } i.
$$

<!-- Cela revient à dire que le vecteur de valeurs suit une gaussienne multivariée avec une matrice
de covariance identité :

$$
\mathbf f \sim \mathcal N(\mathbf 0, I).
$$ -->

Cela revient à dire que le vecteur de valeurs suit une **gaussienne multivariée** avec une
matrice de covariance identité :

$$
\mathbf f \sim \mathcal N(\mathbf 0, I).
$$

Ici, $\mathbf f$ n'est plus un simple nombre aléatoire : c'est un vecteur complet,

$$
\mathbf f =
\begin{bmatrix}
f(x_1) \\
f(x_2) \\
\vdots \\
f(x_n)
\end{bmatrix}.
$$

La gaussienne multivariée décrit donc une distribution sur des **vecteurs entiers**. Chaque
échantillon de cette loi donne une valeur possible pour $f(x_1)$, une valeur possible pour
$f(x_2)$, etc. Si on relie ces valeurs dans l'ordre des $x_i$, on obtient une fonction candidate.

La matrice de covariance indique comment les composantes du vecteur bougent les unes par
rapport aux autres. Avec la matrice identité,

$$
I =
\begin{bmatrix}
1 & 0 & \cdots & 0 \\
0 & 1 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & 1
\end{bmatrix},
$$

chaque diagonale vaut $1$ : chaque $f(x_i)$ a une variance de $1$. Mais tous les termes hors
diagonale valent $0$ : cela signifie que $f(x_i)$ et $f(x_j)$ ne sont pas corrélés quand
$i \neq j$.

Autrement dit, la loi dit :

$$
\operatorname{cov}(f(x_i), f(x_j)) = 0
\quad \text{pour } i \neq j.
$$

Donc deux points très proches sur l'axe des entrées, par exemple $x_i = 0.50$ et
$x_j = 0.51$, ne se parlent pas du tout. La valeur tirée en $0.50$ peut être très haute, et
celle tirée en $0.51$ très basse, sans que la loi trouve cela improbable. C'est exactement ce
qui produit les courbes hérissées de bruit blanc.

Le problème saute aux yeux dès qu'on relie les points : **rien ne dit que deux entrées proches
doivent donner des sorties proches**. On obtient des « fonctions » hérissées, inexploitables.

Il manque la pièce maîtresse : la **corrélation** entre les composantes. On veut que si $x_1$ et
$x_2$ sont proches, alors $f(x_1)$ et $f(x_2)$ soient *corrélés* : ils doivent avoir tendance à
monter ou descendre ensemble. C'est précisément ce qu'encode la **matrice de covariance** d'une
gaussienne **multivariée** :

$$
\mathbf f \sim \mathcal N(\mathbf 0, K),
\qquad
K_{ij} = \operatorname{cov}(f(x_i), f(x_j)).
$$

Le changement de repère est donc le suivant :

- avant : on tire **un scalaire** sur l'axe vertical ;
- maintenant : on fixe plusieurs entrées sur l'axe horizontal ;
- puis on tire **tout un vecteur de hauteurs** au-dessus de ces entrées ;
- la covariance décide si ces hauteurs bougent indépendamment ou ensemble.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
rng = np.random.default_rng(0)

# 1) Une variable scalaire : un tirage sur l'axe y
y_samples = rng.normal(0, 1, size=120)
axes[0].scatter(np.zeros_like(y_samples), y_samples, s=18, alpha=0.45)
axes[0].axhline(0, color="black", lw=1)
axes[0].set_xlim(-0.6, 0.6)
axes[0].set_title("Une variable aléatoire\nun tirage sur l'axe y")
axes[0].set_xlabel("pas d'entrée x")
axes[0].set_ylabel("y")
axes[0].set_xticks([0])
axes[0].set_xticklabels([""])

# 2) Plusieurs entrées fixes, valeurs indépendantes
grid = np.linspace(0, 1, 25)
for _ in range(4):
    values = rng.normal(0, 1, size=grid.size)
    axes[1].plot(grid, values, marker="o", ms=3, alpha=0.8)
axes[1].set_title("Vecteur de valeurs indépendantes\nK = I")
axes[1].set_xlabel("entrée x")
axes[1].set_ylabel("f(x)")

# 3) Plusieurs entrées fixes, valeurs corrélées par un kernel RBF
def smooth_sample(grid, ell=0.15):
    d = grid[:, None] - grid[None, :]
    K = np.exp(-0.5 * (d / ell) ** 2) + 1e-9 * np.eye(grid.size)
    L = np.linalg.cholesky(K)
    return L @ rng.normal(size=grid.size)

for _ in range(4):
    axes[2].plot(grid, smooth_sample(grid), lw=2, alpha=0.8)
axes[2].set_title("Vecteur de valeurs corrélées\npoints proches → valeurs proches")
axes[2].set_xlabel("entrée x")
axes[2].set_ylabel("f(x)")

plt.tight_layout()
plt.show()

**Lecture.** À gauche, une gaussienne scalaire tire seulement une hauteur $y$. Au milieu, on a
fixé plusieurs entrées $x_i$, puis tiré indépendamment les hauteurs $f(x_i)$ : relier ces points
donne du bruit blanc, pas une fonction plausible. À droite, les mêmes positions $x_i$ sont
utilisées, mais les hauteurs sont tirées ensemble avec une covariance corrélée : les points
proches ont tendance à produire des valeurs proches.

C'est exactement le saut conceptuel des processus gaussiens : on ne place pas une loi normale
sur un seul nombre, mais sur un **vecteur de valeurs de fonction**. Toute la magie du GP tiendra
dans **comment on choisit la covariance** entre deux entrées $x_i$ et $x_j$. Ce sera le rôle du
*kernel* dans la partie I.4.

## I.3 — La gaussienne multivariée

Un vecteur $\mathbf{x} \in \mathbb{R}^D$ suit une loi normale multivariée
$\mathcal{N}(\boldsymbol{\mu}, \Sigma)$ de densité :

$$p(\mathbf{x}) = \frac{1}{(2\pi)^{D/2}\,|\Sigma|^{1/2}}\,
\exp\!\left(-\tfrac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \Sigma^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

| Terme | Rôle |
|-------|------|
| $\boldsymbol{\mu} \in \mathbb{R}^D$ | vecteur **moyenne** (un centre par dimension) |
| $\Sigma \in \mathbb{R}^{D\times D}$ | matrice de **covariance**, symétrique définie-positive |
| $\Sigma_{ii}$ | variance de la composante $i$ (sur la diagonale) |
| $\Sigma_{ij}$ | **covariance** entre composantes $i$ et $j$ (hors diagonale) — le lien recherché |
| $(\mathbf{x}-\boldsymbol{\mu})^\top \Sigma^{-1}(\mathbf{x}-\boldsymbol{\mu})$ | **distance de Mahalanobis** au carré : une distance qui tient compte des corrélations |

En 2D, la densité est une **cloche** au-dessus du plan, et ses lignes de niveau sont des
**ellipses**. L'orientation et l'aplatissement de ces ellipses *sont* la corrélation : une ellipse
inclinée signifie que les deux variables montent et descendent ensemble.

In [ ]:
def gaussian_2d_pdf(x, y, mean, cov):
    pos = np.dstack((x, y))
    inv_cov = np.linalg.inv(cov)
    det_cov = np.linalg.det(cov)
    diff = pos - mean

    exponent = np.einsum("...i,ij,...j->...", diff, inv_cov, diff)
    norm = 1.0 / (2.0 * np.pi * np.sqrt(det_cov))
    return norm * np.exp(-0.5 * exponent)

# On regarde seulement deux composantes du vecteur :
# f1 = f(x1) et f2 = f(x2).
f1 = np.linspace(-3, 3, 140)
f2 = np.linspace(-3, 3, 140)
F1, F2 = np.meshgrid(f1, f2)

mean = np.array([0.0, 0.0])

# Cas 1 : covariance identité -> f(x1) et f(x2) indépendants.
cov_independent = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
])

# Cas 2 : covariance corrélée -> f(x1) et f(x2) ont tendance à bouger ensemble.
rho = 0.85
cov_correlated = np.array([
    [1.0, rho],
    [rho, 1.0],
])

Z_independent = gaussian_2d_pdf(F1, F2, mean, cov_independent)
Z_correlated = gaussian_2d_pdf(F1, F2, mean, cov_correlated)

fig = plt.figure(figsize=(13, 9))

# 1) Vue du dessus : non corrélé
ax1 = fig.add_subplot(2, 2, 1)
contour1 = ax1.contourf(F1, F2, Z_independent, levels=30, cmap=cm.Blues)
ax1.contour(F1, F2, Z_independent, levels=8, colors="black", alpha=0.35, linewidths=0.8)
ax1.set_title("Vue du dessus — non corrélé\ncov(f(x₁), f(x₂)) = 0")
ax1.set_xlabel("f(x₁)")
ax1.set_ylabel("f(x₂)")
ax1.set_aspect("equal")
fig.colorbar(contour1, ax=ax1, fraction=0.046, pad=0.04)

# 2) Vue du dessus : corrélé
ax2 = fig.add_subplot(2, 2, 2)
contour2 = ax2.contourf(F1, F2, Z_correlated, levels=30, cmap=cm.Oranges)
ax2.contour(F1, F2, Z_correlated, levels=8, colors="black", alpha=0.35, linewidths=0.8)
ax2.set_title("Vue du dessus — corrélé\ncov(f(x₁), f(x₂)) > 0")
ax2.set_xlabel("f(x₁)")
ax2.set_ylabel("f(x₂)")
ax2.set_aspect("equal")
fig.colorbar(contour2, ax=ax2, fraction=0.046, pad=0.04)

# 3) Vue 3D : non corrélé
ax3 = fig.add_subplot(2, 2, 3, projection="3d")
ax3.plot_surface(F1, F2, Z_independent, cmap=cm.Blues, linewidth=0, alpha=0.9)
ax3.set_title("Cloche 3D — non corrélé")
ax3.set_xlabel("f(x₁)")
ax3.set_ylabel("f(x₂)")
ax3.set_zlabel("densité")
ax3.view_init(elev=28, azim=-55)

# 4) Vue 3D : corrélé
ax4 = fig.add_subplot(2, 2, 4, projection="3d")
ax4.plot_surface(F1, F2, Z_correlated, cmap=cm.Oranges, linewidth=0, alpha=0.9)
ax4.set_title("Cloche 3D — corrélé")
ax4.set_xlabel("f(x₁)")
ax4.set_ylabel("f(x₂)")
ax4.set_zlabel("densité")
ax4.view_init(elev=28, azim=-55)

plt.tight_layout()
plt.show()


**Lecture.** Les deux colonnes représentent la même loi normale sur deux valeurs de fonction :
$f(x_1)$ et $f(x_2)$.

En haut a gauche,la vue 2D montre la densité vue du dessus, covariance nulle : les ellipses sont des cercles, $x_1$ et $x_2$ sont
indépendants (connaître l'un ne dit rien sur l'autre). À droite, covariance $+0.8$ : l'ellipse est
inclinée à 45°: les deux valeurs ont tendance à monter ou descendre ensemble, quand $x_1$ est grand, $x_2$ tend à l'être aussi

En bas, la vue 3D montre la même information sous forme de cloche. Dans le cas non corrélé, la
cloche est droite et symétrique. Dans le cas corrélé, elle est inclinée et allongée dans la
direction diagonale. Cette inclinaison représente exactement la covariance positive entre
$f(x_1)$ et $f(x_2)$.

Dans un processus gaussien, on généralise cette idée à un grand nombre de points d'entrée :
la matrice de covariance $K$ indique, pour chaque paire $(x_i, x_j)$, à quel point les valeurs
$f(x_i)$ et $f(x_j)$ doivent bouger ensemble.

**C'est exactement ce levier
qu'on va utiliser** : en remplissant $\Sigma$ avec « les entrées proches sont corrélées », on
forcera les fonctions tirées à être lisses. Reste une opération à maîtriser, la plus importante de
toutes : le **conditionnement**.

<!-- ## I.4 — Le conditionnement : observer pour mettre à jour

Voici le théorème qui **fonde toute la régression GP**. Découpons un vecteur gaussien en deux
blocs : $\mathbf{a}$ (ce qu'on *observe*) et $\mathbf{b}$ (ce qu'on veut *prédire*) :

$$\begin{bmatrix} \mathbf{a} \\ \mathbf{b} \end{bmatrix} \sim \mathcal{N}\!\left(
\begin{bmatrix} \boldsymbol{\mu}_a \\ \boldsymbol{\mu}_b \end{bmatrix},\;
\begin{bmatrix} \Sigma_{aa} & \Sigma_{ab} \\ \Sigma_{ba} & \Sigma_{bb} \end{bmatrix}\right)$$

Si l'on **observe** $\mathbf{a}$, la loi de $\mathbf{b}$ *sachant* $\mathbf{a}$ reste gaussienne,
avec une moyenne et une covariance **mises à jour** :

$$\boxed{\;\boldsymbol{\mu}_{b\mid a} = \boldsymbol{\mu}_b + \Sigma_{ba}\Sigma_{aa}^{-1}(\mathbf{a}-\boldsymbol{\mu}_a)\;}
\qquad
\boxed{\;\Sigma_{b\mid a} = \Sigma_{bb} - \Sigma_{ba}\Sigma_{aa}^{-1}\Sigma_{ab}\;}$$

Décortiquons :

- $\Sigma_{ba}\Sigma_{aa}^{-1}$ est un **gain** : il traduit « de combien révise-t-on $\mathbf{b}$
  par unité d'écart observé sur $\mathbf{a}$ ». S'il n'y a aucune corrélation ($\Sigma_{ba}=0$),
  le gain est nul : observer $\mathbf{a}$ n'apprend rien sur $\mathbf{b}$.
- La moyenne se déplace proportionnellement à la **surprise** $(\mathbf{a}-\boldsymbol{\mu}_a)$.
- La covariance, elle, ne dépend **pas** de la valeur observée : elle ne fait que **diminuer**
  (on retranche un terme positif). **Observer réduit toujours l'incertitude.**

Cette unique formule, appliquée à un vecteur de valeurs de fonction, *est* la prédiction GP. -->

## I.4 — Le conditionnement : observer pour mettre à jour

Voici le théorème qui **fonde toute la régression GP**. Jusqu'ici, on a appris à mettre une loi
normale multivariée sur un vecteur de valeurs de fonction. Mais une régression ne consiste pas
seulement à dire « voici les fonctions plausibles avant les données ». Elle consiste à dire :

> maintenant que j'ai observé quelques points, quelles fonctions restent plausibles ?

C'est exactement le rôle du **conditionnement gaussien**.

On découpe un vecteur gaussien en deux blocs :

- $\mathbf{a}$ : les variables que l'on **observe** ;
- $\mathbf{b}$ : les variables que l'on veut **prédire**.

Dans une régression GP, $\mathbf{a}$ correspondra typiquement aux valeurs observées sur les
points d'entraînement, et $\mathbf{b}$ aux valeurs inconnues sur les points de test.

On part de la loi jointe :

$$
\begin{bmatrix}
\mathbf{a} \\
\mathbf{b}
\end{bmatrix}
\sim
\mathcal{N}\!\left(
\begin{bmatrix}
\boldsymbol{\mu}_a \\
\boldsymbol{\mu}_b
\end{bmatrix},
\begin{bmatrix}
\Sigma_{aa} & \Sigma_{ab} \\
\Sigma_{ba} & \Sigma_{bb}
\end{bmatrix}
\right).
$$

Cette écriture dit trois choses.

D'abord, $\Sigma_{aa}$ décrit comment les composantes observées $\mathbf{a}$ sont corrélées entre
elles. Ensuite, $\Sigma_{bb}$ décrit notre incertitude sur les composantes à prédire
$\mathbf{b}$. Enfin, les blocs croisés $\Sigma_{ab}$ et $\Sigma_{ba}$ disent comment les
observations et les prédictions sont reliées.

Ce sont précisément ces blocs croisés qui rendent l'observation utile. Si $\mathbf{a}$ et
$\mathbf{b}$ sont corrélés, alors voir $\mathbf{a}$ nous apprend quelque chose sur $\mathbf{b}$.
S'ils ne sont pas corrélés, observer $\mathbf{a}$ ne change rien.

Après avoir observé une valeur concrète de $\mathbf{a}$, la loi de $\mathbf{b}$ **sachant**
$\mathbf{a}$ reste gaussienne :

$$
\mathbf{b}\mid \mathbf{a}
\sim
\mathcal{N}(
\boldsymbol{\mu}_{b\mid a},
\Sigma_{b\mid a}
).
$$

La moyenne conditionnelle vaut :

$$
\boxed{
\boldsymbol{\mu}_{b\mid a}
=
\boldsymbol{\mu}_b
+
\Sigma_{ba}\Sigma_{aa}^{-1}
(\mathbf{a}-\boldsymbol{\mu}_a)
}
$$

et la covariance conditionnelle vaut :

$$
\boxed{
\Sigma_{b\mid a}
=
\Sigma_{bb}
-
\Sigma_{ba}\Sigma_{aa}^{-1}\Sigma_{ab}
}.
$$

La première formule explique **comment la prédiction se déplace**. Avant d'observer, la meilleure
prédiction moyenne pour $\mathbf{b}$ est $\boldsymbol{\mu}_b$. Après observation, on ajoute une
correction.

Cette correction dépend de deux éléments :

$$
(\mathbf{a}-\boldsymbol{\mu}_a)
$$

et

$$
\Sigma_{ba}\Sigma_{aa}^{-1}.
$$

Le premier terme est la **surprise observée** : ce que l'on a vu moins ce que le modèle attendait.
Si $\mathbf{a}$ est exactement égal à sa moyenne prior, il n'y a pas de surprise, donc la moyenne
de $\mathbf{b}$ ne bouge pas.

Le second terme est un **gain de propagation**. Il répond à la question :

> quand les observations changent d'une certaine manière, dans quelle direction faut-il déplacer
> les prédictions ?

Si $\mathbf{a}$ et $\mathbf{b}$ sont fortement corrélés, ce gain est grand : une surprise sur
$\mathbf{a}$ se propage fortement vers $\mathbf{b}$. Si $\Sigma_{ba}=0$, alors :

$$
\Sigma_{ba}\Sigma_{aa}^{-1} = 0,
$$

et donc :

$$
\boldsymbol{\mu}_{b\mid a} = \boldsymbol{\mu}_b.
$$

Observer $\mathbf{a}$ ne change pas la moyenne de $\mathbf{b}$.

La seconde formule explique **comment l'incertitude diminue**. Avant observation, l'incertitude
sur $\mathbf{b}$ est $\Sigma_{bb}$. Après observation, on retranche :

$$
\Sigma_{ba}\Sigma_{aa}^{-1}\Sigma_{ab}.
$$

Ce terme représente la part de l'incertitude de $\mathbf{b}$ qui pouvait être expliquée par
$\mathbf{a}$. Une fois $\mathbf{a}$ observé, cette part n'est plus incertaine : elle est devenue
information.

Un détail très important : la covariance conditionnelle ne dépend pas de la valeur observée
$\mathbf{a}$, mais seulement de **l'endroit où l'on observe** et de la covariance du modèle. Dit
autrement :

- la **valeur** observée déplace la moyenne ;
- la **position** des observations réduit l'incertitude.

C'est exactement ce qu'on verra dans un GP. Si on observe un point très haut, la moyenne
prédictive autour de ce point monte. Mais l'incertitude diminue surtout parce qu'on a observé
à cet endroit, peu importe que la valeur observée soit haute ou basse.

Dans le cas GP, on remplacera simplement les blocs par des matrices de kernel :

$$
\mathbf{a} = \mathbf{y}_{train},
\qquad
\mathbf{b} = \mathbf{f}_{test},
$$

$$
\Sigma_{aa} = K(X_{train}, X_{train}) + \sigma_n^2 I,
$$

$$
\Sigma_{ba} = K(X_{test}, X_{train}),
\qquad
\Sigma_{bb} = K(X_{test}, X_{test}).
$$

La formule de conditionnement devient alors la formule de prédiction GP :

$$
\boldsymbol{\mu}_{test}
=
\boldsymbol{\mu}_{test}^{prior}
+
K(X_{test}, X_{train})
\left[
K(X_{train}, X_{train}) + \sigma_n^2 I
\right]^{-1}
(\mathbf{y}_{train}-\boldsymbol{\mu}_{train})
$$

et

$$
\Sigma_{test}
=
K(X_{test}, X_{test})
-
K(X_{test}, X_{train})
\left[
K(X_{train}, X_{train}) + \sigma_n^2 I
\right]^{-1}
K(X_{train}, X_{test}).
$$

Cette unique formule est la régression GP. Le kernel dit quelles valeurs de fonction sont
corrélées. Le conditionnement dit comment transformer cette corrélation en prédiction après
avoir vu des données.

In [ ]:
def gaussian_condition(mu, Sigma, idx_obs, a_values):
    # Loi de b | a pour une gaussienne jointe. idx_obs = indices observés (a).
    D = mu.size
    idx_pred = [i for i in range(D) if i not in idx_obs]
    mu_a, mu_b = mu[idx_obs], mu[idx_pred]
    Saa = Sigma[np.ix_(idx_obs, idx_obs)]
    Sbb = Sigma[np.ix_(idx_pred, idx_pred)]
    Sba = Sigma[np.ix_(idx_pred, idx_obs)]
    gain = Sba @ np.linalg.inv(Saa)
    mu_cond = mu_b + gain @ (a_values - mu_a)
    Sigma_cond = Sbb - gain @ Sba.T
    return idx_pred, mu_cond, Sigma_cond

# Jointe corrélée (x1, x2) ; on observe x1 = 1.5 et on conditionne x2
mu_j = np.array([0.0, 0.0])
S_j = np.array([[1.0, 0.8], [0.8, 1.0]])
_, mu_c, S_c = gaussian_condition(mu_j, S_j, idx_obs=[0], a_values=np.array([1.5]))
print(f"Avant : x2 ~ N(0.0, 1.00)")
print(f"Après avoir observé x1=1.5 : x2 ~ N({mu_c.item():.3f}, {S_c.item():.3f})")
print(f"-> la moyenne s'est déplacée vers +{mu_c.item():.3f} (corrélation +{S_j[0,1]:.1f}) et la variance a chuté de 1.0 à {S_c.item():.2f}")

In [ ]:
# Visualisation : jointe 2D + marginales sur les axes + conditionnelle
def mvn_pdf(points, mu, Sigma):
    # Densité gaussienne 2D évaluée sur une grille de points de forme (..., 2).
    Sinv = np.linalg.inv(Sigma)
    det = np.linalg.det(Sigma)
    d = points - mu
    quad = np.einsum("...i,ij,...j->...", d, Sinv, d)
    return np.exp(-0.5 * quad) / (2 * np.pi * np.sqrt(det))


x_obs = 1.5
mu_j_flat = np.asarray(mu_j).reshape(-1)
S_j_arr = np.asarray(S_j)

grid = np.linspace(-3, 3, 200)
XX, YY = np.meshgrid(grid, grid)
pts = np.stack([XX, YY], axis=-1)

Z = mvn_pdf(pts, mu_j_flat, S_j_arr)

fig = plt.figure(figsize=(12.5, 4.6))
outer = fig.add_gridspec(1, 2, width_ratios=[1.15, 1.0], wspace=0.35)

# Bloc gauche : contour 2D + marginales p(x1) et p(x2)
left = outer[0].subgridspec(
    2, 2,
    height_ratios=[0.28, 1.0],
    width_ratios=[1.0, 0.28],
    hspace=0.04,
    wspace=0.04,
)

ax_top = fig.add_subplot(left[0, 0])
ax_joint = fig.add_subplot(left[1, 0], sharex=ax_top)
ax_right = fig.add_subplot(left[1, 1], sharey=ax_joint)

# Jointe p(x1, x2)
ax_joint.contourf(XX, YY, Z, levels=24, cmap="viridis")
ax_joint.contour(XX, YY, Z, levels=8, colors="white", alpha=0.45, linewidths=0.8)
ax_joint.axvline(x_obs, color="red", lw=2, label=fr"on observe $x_1={x_obs}$")
ax_joint.set_title("Jointe $p(x_1, x_2)$ + marginales")
ax_joint.set_xlabel("$x_1$")
ax_joint.set_ylabel("$x_2$")
ax_joint.legend(loc="upper left")
ax_joint.set_aspect("equal")

# Marginale sur l'axe x : p(x1)
p_x1 = gaussian_pdf(grid, mu_j_flat[0], np.sqrt(S_j_arr[0, 0]))
ax_top.plot(grid, p_x1, color="steelblue", lw=2)
ax_top.fill_between(grid, 0, p_x1, color="steelblue", alpha=0.25)
ax_top.axvline(x_obs, color="red", lw=2)
ax_top.set_ylabel("$p(x_1)$")
ax_top.tick_params(axis="x", labelbottom=False)
ax_top.grid(True, alpha=0.2)

# Marginale sur l'axe y : p(x2)
p_x2 = gaussian_pdf(grid, mu_j_flat[1], np.sqrt(S_j_arr[1, 1]))
ax_right.plot(p_x2, grid, color="darkorange", lw=2)
ax_right.fill_betweenx(grid, 0, p_x2, color="darkorange", alpha=0.25)
ax_right.set_xlabel("$p(x_2)$")
ax_right.tick_params(axis="y", labelleft=False)
ax_right.grid(True, alpha=0.2)

# Bloc droit : prior marginal vs posterior conditionnel
ax_cond = fig.add_subplot(outer[1])
x2s = np.linspace(-3, 3, 300)
prior = gaussian_pdf(x2s, mu_j_flat[1], np.sqrt(S_j_arr[1, 1]))
post = gaussian_pdf(x2s, mu_c.item(), np.sqrt(S_c.item()))

ax_cond.plot(x2s, prior, "--", color="gray", lw=2, label="avant : $p(x_2)$")
ax_cond.plot(x2s, post, "red", lw=2.5, label=fr"après : $p(x_2 \mid x_1={x_obs})$")
ax_cond.fill_between(x2s, 0, post, color="red", alpha=0.18)
ax_cond.set_title("Conditionnement = tranche renormalisée")
ax_cond.set_xlabel("$x_2$")
ax_cond.set_ylabel("densité")
ax_cond.legend()
ax_cond.grid(True, alpha=0.25)

plt.show()

**Lecture.** Conditionner, c'est **trancher** la cloche jointe le long de $x_1 = 1.5$, puis
**renormaliser** cette tranche : on obtient une nouvelle gaussienne sur $x_2$, plus piquée
(incertitude réduite) et décalée vers les valeurs cohérentes avec l'observation. **Toute la
régression GP est cette image, répétée en haute dimension** : la jointe relie les valeurs de
fonction aux points connus *et* aux points à prédire ; on observe les premières, on tranche, et il
reste la prédiction avec sa barre d'erreur. Gardez cette image : c'est le cœur conceptuel.

In [ ]:
# Mini-test : le conditionnement préserve une covariance valide (symétrique, semi-définie positive)
mu_t = np.zeros(3)
A = np.random.randn(3, 3)
S_t = A @ A.T + 0.1 * np.eye(3)          # une covariance valide
idx, m_t, Sc_t = gaussian_condition(mu_t, S_t, idx_obs=[0], a_values=np.array([0.5]))
assert np.allclose(Sc_t, Sc_t.T, atol=1e-10), "covariance conditionnelle non symétrique"
assert np.linalg.eigvalsh(Sc_t).min() > -1e-10, "covariance conditionnelle non SDP"
assert np.linalg.eigvalsh(Sc_t).max() <= np.linalg.eigvalsh(S_t[np.ix_(idx, idx)]).max() + 1e-9
print("✓ conditionnement OK : sortie gaussienne valide, incertitude réduite")

## I.5 — Le kernel : fabriquer la corrélation

On sait maintenant que pour générer des fonctions lisses, il faut une matrice de covariance
$\Sigma$ qui dise :

> deux entrées proches doivent produire des valeurs de fonction corrélées.

Le problème est que, dans un GP, on ne veut pas écrire cette matrice à la main. Si on évalue la
fonction en $n$ points $x_1,\dots,x_n$, il faudrait remplir une matrice $n\times n$ :

$$
\Sigma =
\begin{bmatrix}
\operatorname{cov}(f(x_1), f(x_1)) & \operatorname{cov}(f(x_1), f(x_2)) & \cdots \\
\operatorname{cov}(f(x_2), f(x_1)) & \operatorname{cov}(f(x_2), f(x_2)) & \cdots \\
\vdots & \vdots & \ddots
\end{bmatrix}.
$$

Le **kernel** résout exactement ce problème. Un kernel est une fonction

$$
k(x, x')
$$

qui prend deux entrées et renvoie leur covariance :

$$
\operatorname{cov}(f(x), f(x')) = k(x, x').
$$

Une fois qu'on a choisi un kernel, on peut construire automatiquement la matrice de covariance
en appliquant cette même recette à toutes les paires de points :

$$
K_{ij} = k(x_i, x_j).
$$

Donc le kernel est le cœur du GP : il définit **ce que le modèle considère comme des fonctions
plausibles**. Il ne prédit pas directement les valeurs de la fonction. Il définit plutôt la
structure de corrélation qui dit comment une observation en un point doit influencer les autres.

---

### Pourquoi un kernel dans PILCO ?

Dans PILCO, le GP sert à apprendre la dynamique du système. Typiquement, on veut approximer une
fonction du type :

$$
f(s_t, a_t) \approx \Delta s_t = s_{t+1} - s_t.
$$

L'entrée du GP est donc un couple état-action :

$$
x_t =
\begin{bmatrix}
s_t \\
a_t
\end{bmatrix},
$$

et la sortie est le changement d'état.

L'hypothèse physique derrière PILCO est la suivante :

> si deux couples état-action $(s,a)$ et $(s',a')$ sont proches, alors les changements d'état
> $\Delta s$ et $\Delta s'$ devraient être proches aussi.

C'est exactement une hypothèse de corrélation. Le kernel encode donc notre croyance sur la
régularité de la dynamique. Sans kernel, le GP ne saurait pas comment généraliser une transition
observée à des transitions voisines.

Par exemple, si le pendule est presque au même angle, presque à la même vitesse, et reçoit presque
le même couple moteur, on s'attend à une transition similaire. Le kernel traduit cette idée en
nombre : une covariance élevée.

---

### Ce que doit respecter un kernel

Un kernel ne peut pas être n'importe quelle fonction de similarité. Pour être une covariance valide,
il doit produire, pour n'importe quel ensemble de points, une matrice $K$ symétrique et positive
semi-définie :

$$
K = K^\top,
\qquad
\mathbf{v}^\top K \mathbf{v} \ge 0
\quad \text{pour tout vecteur } \mathbf{v}.
$$

Intuitivement, cela garantit que la matrice obtenue peut réellement être la covariance d'une
gaussienne multivariée. Si cette condition n'est pas respectée, on ne peut plus échantillonner
correctement des fonctions, ni conditionner la loi gaussienne de manière stable.

C'est pour cela qu'on utilise des kernels connus et bien étudiés : RBF, Matérn, linéaire,
périodique, etc. Chaque kernel correspond à une hypothèse différente sur les fonctions possibles.

---

### Le kernel RBF / Squared Exponential

PILCO utilise généralement le **kernel exponentiel quadratique**, appelé aussi :

- **Squared Exponential** ;
- **SE kernel** ;
- **RBF kernel** ;
- kernel gaussien.

Sa forme avec ARD (*Automatic Relevance Determination*) est :

$$
k(x, x')
=
\sigma_f^2
\exp\!\left(
-\frac{1}{2}
\sum_{d=1}^{D}
\frac{(x_d - x'_d)^2}{\ell_d^2}
\right).
$$

On peut le lire en trois étapes.

D'abord, on mesure la distance entre $x$ et $x'$ dimension par dimension :

$$
(x_d - x'_d)^2.
$$

Ensuite, on remet chaque dimension à l'échelle avec une **length-scale** $\ell_d$ :

$$
\frac{(x_d - x'_d)^2}{\ell_d^2}.
$$

Enfin, on transforme cette distance en covariance avec une exponentielle décroissante :

$$
\exp\!\left(-\frac{1}{2}\text{distance normalisée}\right).
$$

Donc :

- si $x = x'$, la distance vaut $0$, donc $k(x,x') = \sigma_f^2$ ;
- si $x$ et $x'$ sont proches, la covariance reste élevée ;
- si $x$ et $x'$ sont loin, la covariance tend vers $0`.

---

### Rôle des hyperparamètres

| Terme | Rôle |
|-------|------|
| $\sigma_f^2$ | **variance du signal** : amplitude typique des variations de la fonction |
| $\ell_d$ | **length-scale** de la dimension $d$ : distance nécessaire pour que la fonction change notablement |
| $(x_d - x'_d)^2 / \ell_d^2$ | distance au carré, remise à l'échelle dimension par dimension |
| $\exp(-\tfrac12 \cdot)$ | transforme une distance en corrélation lisse et décroissante |

La length-scale est le paramètre le plus intuitif.

Si $\ell_d$ est petit, une petite différence dans la dimension $d$ suffit à faire chuter la
corrélation. Le modèle considère donc que la fonction peut varier rapidement selon cette dimension.

Si $\ell_d$ est grand, même deux valeurs assez éloignées sur cette dimension restent corrélées.
Le modèle considère donc que la fonction varie lentement selon cette dimension.

Dans PILCO, cette idée est essentielle. Certaines dimensions de l'état peuvent être très
importantes pour prédire la dynamique, d'autres beaucoup moins. Avec une length-scale différente
par dimension, le GP peut apprendre automatiquement quelles dimensions comptent.

C'est le sens de l'ARD :

$$
\text{ARD} = \text{Automatic Relevance Determination}.
$$

Une petite length-scale signifie : cette dimension est importante, car la fonction change vite
quand elle varie. Une grande length-scale signifie : cette dimension influence peu la sortie,
car même de grands changements ne modifient pas beaucoup la prédiction.

---

### Pourquoi cette forme est adaptée au monde physique

Le kernel RBF encode une hypothèse très forte : les fonctions sont lisses. Pas seulement continues,
mais très régulières. Deux points très proches ont presque la même valeur, et cette similarité
décroît doucement avec la distance.

C'est une bonne hypothèse pour beaucoup de systèmes physiques à petite échelle. La dynamique d'un
pendule, d'un chariot ou d'un bras robot ne change généralement pas brutalement si l'état et
l'action changent un tout petit peu.

Dans PILCO, cette hypothèse est ce qui permet d'être très sample-efficient. Avec peu de transitions,
le GP peut généraliser aux régions voisines de l'espace état-action. Une transition observée ne
sert pas seulement à ce point exact : elle informe toute une zone autour de lui.

Mais cette hypothèse a aussi une limite. Si le système a des discontinuités, des contacts durs, des
chocs ou des changements de régime très abrupts, un RBF peut être trop lisse. Le modèle risque alors
de lisser des phénomènes qui ne devraient pas l'être.

In [ ]:
def rbf_kernel(X1, X2, lengthscales, signal_var=1.0):
    """Kernel RBF/SE avec ARD.

    X1: [N, D]
    X2: [M, D]
    Retourne K: [N, M] avec
    k(x,x') = sigma_f^2 exp(-1/2 sum_d ((x_d - x'_d) / ell_d)^2)
    """
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    lengthscales = np.asarray(lengthscales, dtype=float)

    scaled_distance = (X1[:, None, :] - X2[None, :, :]) / lengthscales
    squared_distance = np.sum(scaled_distance ** 2, axis=-1)

    return signal_var * np.exp(-0.5 * squared_distance)


fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

# 1) Corrélation k(0, r) en fonction de la distance r
rs = np.linspace(0, 4, 250).reshape(-1, 1)
for ell in [0.3, 1.0, 2.0]:
    k = rbf_kernel(np.zeros((1, 1)), rs, [ell]).ravel()
    axes[0].plot(rs, k, lw=2, label=f"ℓ = {ell}")

axes[0].set_title("Corrélation $k(0, r)$ vs distance")
axes[0].set_xlabel("distance $r$")
axes[0].set_ylabel("covariance")
axes[0].legend()

# 2) Matrice K pour une petite length-scale
grid = np.linspace(0, 1, 50).reshape(-1, 1)
K_small = rbf_kernel(grid, grid, [0.08])
im1 = axes[1].imshow(K_small, cmap="viridis", origin="lower")
axes[1].set_title("Matrice $K$ — petite ℓ\ncorrélation locale")
axes[1].set_xlabel("index du point $x_j$")
axes[1].set_ylabel("index du point $x_i$")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

# 3) Matrice K pour une grande length-scale
K_large = rbf_kernel(grid, grid, [0.35])
im2 = axes[2].imshow(K_large, cmap="viridis", origin="lower")
axes[2].set_title("Matrice $K$ — grande ℓ\ncorrélation large")
axes[2].set_xlabel("index du point $x_j$")
axes[2].set_ylabel("index du point $x_i$")
fig.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

**Lecture.** À gauche, la length-scale contrôle la vitesse à laquelle la corrélation décroît
avec la distance. Pour $\ell=0.3$, la corrélation tombe très vite : seuls les points presque
identiques s'influencent. Pour $\ell=2.0$, la corrélation reste élevée beaucoup plus loin :
le modèle suppose une fonction très lisse.

Au centre, avec une petite length-scale, la matrice $K$ a une bande brillante très étroite autour
de la diagonale. Cela signifie que chaque point est surtout corrélé avec ses voisins immédiats.
Le GP peut alors produire des fonctions plus flexibles, mais il généralise moins loin.

À droite, avec une grande length-scale, la bande brillante est beaucoup plus large. Des points
éloignés restent corrélés. Le GP produit alors des fonctions plus rigides et très lisses.

Dans PILCO, ces hyperparamètres sont appris à partir des données. C'est crucial : on ne veut pas
choisir à la main à quel point le monde est rigide ou flexible. Le GP ajuste ses length-scales pour
expliquer les transitions observées avec la bonne quantité de lissage.

## I.6 — ARD : toutes les dimensions ne se valent pas

Dans le kernel ci-dessus, chaque dimension $d$ a **sa propre** length-scale $\ell_d$. C'est l'idée
**ARD** (*Automatic Relevance Determination*).

Pourquoi c'est important ? Parce que dans un état physique comme celui du pendule
$[\cos\theta, \sin\theta, \dot\theta]$, certaines dimensions influencent la dynamique beaucoup plus
que d'autres. Une length-scale **grande** sur une dimension signifie « la fonction varie lentement
le long de cet axe → cette dimension est peu pertinente » ; une length-scale **petite** signifie
« la fonction y est très sensible → dimension importante ». En apprenant un $\ell_d$ par dimension,
le GP **découvre tout seul** la pertinence de chaque variable d'entrée — d'où le nom.

In [ ]:
# Anisotropie : même point, deux length-scales différentes selon l'axe -> ellipses de corrélation
center = np.zeros((1, 2))
gx2 = np.linspace(-2, 2, 120); GX, GY = np.meshgrid(gx2, gx2)
P = np.stack([GX.ravel(), GY.ravel()], axis=1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, ls, name in zip(axes, [[1.0, 1.0], [1.2, 0.3]], ["ℓ = (1.0, 1.0) isotrope", "ℓ = (1.2, 0.3) ARD"]):
    K = rbf_kernel(center, P, ls).reshape(GX.shape)
    ax.contourf(GX, GY, K, levels=20, cmap="viridis"); ax.set_title(f"$k(0, x)$ — {name}")
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

**Lecture.** À gauche, length-scales égales : la corrélation autour de l'origine est un
**disque** — les deux dimensions comptent autant. À droite, $\ell_2 \ll \ell_1$ : la corrélation
s'étire en **ellipse** le long de l'axe 1 (peu sensible) et se resserre sur l'axe 2 (très
sensible). En pratique, après apprentissage, regarder les $\ell_d$ d'un GP de dynamique vous dit
*quelles variables d'état pilotent vraiment le système*.

## I.7 — Le prior : un GP est une distribution sur des fonctions

On a maintenant tout pour donner la définition exacte. Un **Gaussian Process** est une
distribution sur des **fonctions**, entièrement déterminée par deux objets :

- une fonction moyenne $m(x)$ ;
- un kernel $k(x,x')$.

On écrit :

$$
f \sim \mathcal{GP}\bigl(m(x),\; k(x, x')\bigr).
$$

Dans beaucoup d'exemples, on prend la moyenne nulle par convention :

$$
f \sim \mathcal{GP}\bigl(0,\; k(x, x')\bigr).
$$

Cela ne veut pas dire que la fonction réelle vaut forcément zéro. Cela veut dire qu'avant d'avoir
vu des données, notre prédiction moyenne est zéro partout. Toute la structure intéressante vient
alors du kernel, qui décrit quelles valeurs de fonction doivent bouger ensemble.

La définition importante est la suivante : pour **n'importe quel** ensemble fini de points

$$
X = \{x_1, x_2, \dots, x_n\},
$$

le vecteur des valeurs de fonction

$$
\mathbf f_X =
\begin{bmatrix}
f(x_1) \\
f(x_2) \\
\vdots \\
f(x_n)
\end{bmatrix}
$$

suit une gaussienne multivariée :

$$
\mathbf f_X \sim \mathcal N(\mathbf m_X, K),
$$

avec

$$
\mathbf m_X =
\begin{bmatrix}
m(x_1) \\
m(x_2) \\
\vdots \\
m(x_n)
\end{bmatrix},
\qquad
K_{ij}=k(x_i,x_j).
$$

C'est cette propriété qui fait du GP une distribution sur des fonctions. On ne définit pas une loi
sur un seul scalaire, mais une règle cohérente qui permet de construire une loi normale
multivariée pour n'importe quelle collection finie de points.

Échantillonner une fonction du prior, c'est donc :

1. choisir une grille de points $X$ ;
2. construire la matrice de covariance $K$ avec le kernel ;
3. tirer un vecteur :

$$
\mathbf f_X \sim \mathcal N(\mathbf 0, K).
$$

Chaque tirage donne une valeur pour $f(x_1)$, une valeur pour $f(x_2)$, etc. Si on relie ces valeurs
dans l'ordre des entrées, on obtient une **fonction plausible selon le prior**.

Le mot **prior** est important. À ce stade, le GP n'a encore vu aucune donnée. Les fonctions tirées
ne sont pas encore adaptées au système que l'on veut modéliser. Elles représentent seulement nos
hypothèses avant observation :

- amplitude typique des variations ;
- degré de lissage ;
- importance relative des dimensions d'entrée ;
- corrélation entre points proches.

On retrouve donc exactement l'expérience « corrélé vs indépendant » de la partie I.2, sauf que
maintenant on comprend d'où vient la corrélation. Elle n'est pas ajoutée à la main dans une matrice
arbitraire : elle est générée automatiquement par le kernel.

Dans PILCO, ce prior est notre croyance initiale sur la dynamique. Avant d'observer des transitions,
on suppose que la dynamique est lisse : deux couples état-action proches devraient produire des
changements d'état proches. Ensuite, dès que l'on observe des données, on conditionne ce prior pour
obtenir un **posterior** : une distribution sur les fonctions de dynamique compatibles avec les
transitions observées.

In [ ]:
def sample_prior(grid, lengthscales, signal_var=1.0, n_samples=5):
    K = rbf_kernel(grid, grid, lengthscales, signal_var) + 1e-9 * np.eye(len(grid))
    L = np.linalg.cholesky(K)                    # K = L L^T  (vu en I.9)
    return (L @ np.random.normal(size=(len(grid), n_samples))).T

g = np.linspace(-5, 5, 200).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, ell in zip(axes, [0.5, 2.0]):
    for f in sample_prior(g, [ell], n_samples=6):
        ax.plot(g, f, lw=1.6, alpha=0.85)
    ax.set_title(f"6 fonctions tirées du prior — ℓ = {ell}"); ax.set_xlabel("x")
axes[0].set_ylabel("f(x)")
plt.tight_layout(); plt.show()

**Lecture.** Chaque courbe est une fonction **tirée au hasard** de notre prior — aucune
donnée n'est encore observée, ce sont nos *croyances a priori*. Avec $\ell=0.5$ (gauche), les
fonctions ondulent vite ; avec $\ell=2.0$ (droite), elles sont amples et douces. Le prior n'est
donc pas « une » fonction mais **tout un nuage de fonctions** dont le kernel fixe le style.
Apprendre, ce sera **conditionner** ce nuage sur les données observées — exactement l'opération de
la partie I.4, en dimension « nombre de points ».

**Au passage — paramétrique vs non-paramétrique.** Un réseau de neurones est
*paramétrique* : il a un nombre **fixe** de poids, et une fois entraîné, il « oublie » les
données. Un GP est *non-paramétrique* : il **garde les données** et la complexité du modèle
**croît avec elles** (la matrice $K$ est de taille $n \times n$). C'est sa force — flexibilité
illimitée et incertitude calibrée — et, on le verra au notebook 10b, sa faiblesse : un coût en
$O(n^3)$ qui le condamne sur les gros jeux de données.

In [ ]:
# Mini-test du kernel : symétrie, définie-positivité, décroissance avec la distance
Xt = np.random.randn(10, 3)
K = rbf_kernel(Xt, Xt, [1.0, 1.0, 1.0], signal_var=2.0)
assert np.allclose(K, K.T, atol=1e-12), "kernel non symétrique"
assert np.linalg.eigvalsh(K).min() > -1e-9, "matrice de Gram non SDP"
assert np.allclose(np.diag(K), 2.0), "k(x,x) doit valoir sigma_f^2"
near = rbf_kernel(np.zeros((1, 1)), np.array([[0.1]]), [1.0]).item()
far = rbf_kernel(np.zeros((1, 1)), np.array([[3.0]]), [1.0]).item()
assert near > far, "la corrélation doit décroître avec la distance"
print(f"✓ kernel OK : symétrique, SDP, k(x,x)=σ_f²={2.0}, k proche={near:.3f} > k loin={far:.3f}")

## I.8 — De la croyance à la prédiction : le posterior GP

On a maintenant deux pièces :

1. un **prior GP**, c'est-à-dire une distribution sur les fonctions plausibles avant d'avoir vu les
   données ;
2. une règle de **conditionnement gaussien**, qui dit comment mettre à jour une partie d'un vecteur
   gaussien quand on en observe une autre.

Le posterior GP n'est rien de plus que l'application directe de cette règle aux valeurs d'une
fonction.

L'idée intuitive est la suivante. Avant les données, le GP imagine beaucoup de fonctions possibles.
Certaines passent très haut, d'autres très bas, certaines oscillent plus ou moins, mais toutes
respectent les hypothèses du kernel. Puis on observe des points. Toutes les fonctions du prior qui
ne sont pas compatibles avec ces observations deviennent peu plausibles. Il reste alors un nouveau
nuage de fonctions : c'est le **posterior**.

On peut le voir comme un filtre :

```text
prior GP = toutes les fonctions plausibles selon le kernel
données observées = contraintes
posterior GP = fonctions plausibles ET compatibles avec les données
```

Si le prior est une carte météo avant d'ouvrir la fenêtre, le posterior est la carte mise à jour
après avoir vu la vraie température à quelques endroits.

---

### Observations bruitées

En régression, on n'observe pas directement la vraie fonction $f(x_i)$. On observe une valeur
bruitée :

$$
y_i = f(x_i) + \varepsilon_i,
\qquad
\varepsilon_i \sim \mathcal N(0,\sigma_n^2).
$$

Le terme $\sigma_n^2$ représente le bruit d'observation. Dans un système physique, il peut venir
de capteurs imparfaits, d'une dynamique non modélisée, d'arrondis numériques, ou simplement du fait
que notre modèle GP n'est pas une description parfaite du monde.

En notation vectorielle :

$$
\mathbf y =
\begin{bmatrix}
y_1 \\
\vdots \\
y_n
\end{bmatrix}.
$$

Les valeurs que l'on veut prédire aux nouveaux points $X_*$ sont notées :

$$
f_* =
\begin{bmatrix}
f(x_*^{(1)}) \\
\vdots \\
f(x_*^{(m)})
\end{bmatrix}.
$$

On observe donc $\mathbf y$, et on veut prédire $f_*$.

---

### La loi jointe : mettre observations et prédictions dans le même vecteur

La définition d'un GP dit que n'importe quel ensemble fini de valeurs de fonction suit une
gaussienne multivariée. On peut donc mettre dans un même vecteur :

- les observations d'entraînement $\mathbf y$ ;
- les valeurs inconnues $f_*$ aux points de test.

On écrit :

$$
\begin{bmatrix}
\mathbf{y} \\
f_*
\end{bmatrix}
\sim
\mathcal{N}\!\left(
\mathbf{0},
\begin{bmatrix}
K + \sigma_n^2 I & K_* \\
K_*^\top & K_{**}
\end{bmatrix}
\right).
$$

Chaque bloc a un rôle précis :

$$
K = k(X, X)
$$

est la covariance entre les points d'entraînement.

$$
K_* = k(X, X_*)
$$

est la covariance entre les points d'entraînement et les points de test.

$$
K_{**} = k(X_*, X_*)
$$

est la covariance entre les points de test.

Enfin,

$$
\sigma_n^2 I
$$

ajoute le bruit d'observation sur les données d'entraînement. On ajoute ce bruit seulement au bloc
des observations, parce que $\mathbf y$ est bruité. Les valeurs $f_*$ que l'on prédit représentent
la fonction latente non bruitée.

---

### Pourquoi ajouter $\sigma_n^2 I$ ?

Ce terme est facile à sous-estimer, mais il est très important.

Sans bruit, le GP est forcé de passer exactement par chaque point observé. C'est parfois ce qu'on
veut, mais en pratique les données peuvent être bruitées. Si un point est légèrement faux, on ne
veut pas que toute la fonction se torde brutalement pour l'interpoler parfaitement.

Ajouter $\sigma_n^2 I$ revient à dire :

> les observations sont informatives, mais pas sacrées.

Plus $\sigma_n^2$ est grand, moins le modèle fait confiance aux points individuellement. Plus il est
petit, plus le GP colle aux données.

On peut aussi lire ce terme comme une protection numérique. La matrice $K$ peut être presque
singulière si deux points d'entraînement sont très proches. Ajouter un petit bruit sur la diagonale
rend l'inversion plus stable.

---

### Conditionner : exactement la formule de la partie I.4

Dans la partie I.4, on avait une loi jointe :

$$
\begin{bmatrix}
\mathbf a \\
\mathbf b
\end{bmatrix}.
$$

Ici, il suffit d'identifier :

$$
\mathbf a = \mathbf y,
\qquad
\mathbf b = f_*.
$$

Les blocs deviennent :

$$
\Sigma_{aa} = K + \sigma_n^2 I,
$$

$$
\Sigma_{ba} = K_*^\top,
$$

$$
\Sigma_{bb} = K_{**}.
$$

En appliquant directement la formule du conditionnement gaussien, on obtient :

$$
\boxed{
\bar{f}_*
=
K_*^\top (K + \sigma_n^2 I)^{-1}\mathbf y
}
$$

et

$$
\boxed{
\operatorname{cov}(f_*)
=
K_{**}
-
K_*^\top (K + \sigma_n^2 I)^{-1}K_*
}.
$$

Ces deux équations sont la prédiction GP.

---

### Lire la moyenne : une somme pondérée de souvenirs

On définit souvent :

$$
\boldsymbol{\alpha}
=
(K + \sigma_n^2 I)^{-1}\mathbf y.
$$

Alors la moyenne prédite devient :

$$
\bar f_*
=
K_*^\top \boldsymbol{\alpha}.
$$

Cette forme est très parlante.

Chaque point d'entraînement reçoit un poids $\alpha_i$. Ensuite, pour prédire en un nouveau point
$x_*$, on regarde à quel point $x_*$ est similaire à chaque point d'entraînement via le kernel.
La prédiction est une somme pondérée de ces similarités.

Pour un seul point de test, on peut l'écrire ainsi :

$$
\bar f(x_*)
=
\sum_{i=1}^{n}
\alpha_i\, k(x_i, x_*).
$$

Donc le GP prédit avec des **bumps de kernel** centrés sur les données. Chaque observation laisse
une sorte d'empreinte autour d'elle. Près d'un point observé, son influence est forte. Loin de lui,
son influence disparaît.

Analogie : imagine que chaque point de donnée plante une lampe sur l'axe des entrées. Le kernel
décrit la forme du halo lumineux autour de cette lampe. La prédiction en $x_*$ est la somme des
lumières reçues depuis toutes les lampes, avec des intensités positives ou négatives selon les
poids $\alpha_i$.

---

### Pourquoi les poids ne sont pas simplement les valeurs $y_i$ ?

On pourrait croire que chaque poids devrait être égal à $y_i$. Mais les points d'entraînement
peuvent être corrélés entre eux. Si deux points sont très proches, ils apportent presque la même
information. Il ne faut donc pas compter deux fois la même preuve.

C'est le rôle de l'inversion :

$$
(K + \sigma_n^2 I)^{-1}.
$$

Elle corrige les poids en tenant compte des corrélations entre les données. Les poids
$\boldsymbol{\alpha}$ ne disent pas seulement « quelle valeur ai-je observée ? », mais aussi :

> quelle information nouvelle ce point apporte-t-il par rapport aux autres points déjà observés ?

C'est pour cela que le GP est plus subtil qu'une simple moyenne pondérée par distance.

---

### Lire la covariance : enlever l'incertitude expliquée par les données

Avant observation, l'incertitude aux points de test est :

$$
K_{**}.
$$

Après observation, on retranche :

$$
K_*^\top (K + \sigma_n^2 I)^{-1}K_*.
$$

Ce terme mesure combien les points d'entraînement expliquent les points de test. Si un point de
test est très corrélé avec les données observées, alors $K_*$ est grand, et l'incertitude diminue
fortement. Si un point de test est loin de tout, $K_*$ est proche de zéro, et la réduction
d'incertitude est presque nulle.

Donc :

$$
\operatorname{cov}(f_*) \approx K_{**}
\quad \text{loin des données}.
$$

C'est une propriété très importante : **le GP sait qu'il ne sait pas**. Loin des observations, il
revient vers son incertitude prior au lieu d'inventer une certitude artificielle.

---

### Pourquoi la variance ne dépend pas des valeurs observées ?

La covariance posterior vaut :

$$
K_{**}
-
K_*^\top (K + \sigma_n^2 I)^{-1}K_*.
$$

On remarque que $\mathbf y$ n'apparaît pas dans cette formule. Cela peut surprendre.

Cela veut dire que l'incertitude dépend de **où** on a observé, pas de **quelle valeur** on a
observée. Si on mesure un point à $x=0.5$, on réduit l'incertitude autour de $0.5$ que la valeur
observée soit $10$, $-3$ ou $0.2$. La valeur change la moyenne, mais la position change la
confiance.

Analogie : si tu demandes la température à quelqu'un à Paris, tu réduis ton incertitude sur Paris
et ses environs, peu importe que la réponse soit 12°C ou 30°C. La réponse change ta carte moyenne
des températures ; le fait d'avoir une mesure à Paris change ta certitude locale.

---

### Résumé des termes

| Terme | Lecture |
|-------|---------|
| $\boldsymbol{\alpha} = (K + \sigma_n^2 I)^{-1}\mathbf{y}$ | poids appris, corrigés par les corrélations entre données |
| $\bar{f}_* = K_*^\top \boldsymbol{\alpha}$ | moyenne prédite comme somme pondérée de kernels centrés sur les données |
| $K_{**}$ | incertitude a priori aux points de test |
| $K_*^\top(K+\sigma_n^2 I)^{-1}K_*$ | réduction d'incertitude apportée par les données |
| $\sigma_n^2 I$ | bruit d'observation et stabilisation numérique |

---

### Le lien direct avec PILCO

Dans PILCO, on applique cette prédiction GP à la dynamique. Les entrées sont des couples
état-action :

$$
x_t =
\begin{bmatrix}
s_t \\
a_t
\end{bmatrix},
$$

et les sorties sont souvent des deltas d'état :

$$
y_t = \Delta s_t = s_{t+1} - s_t.
$$

Après avoir observé des transitions, le GP donne pour un nouveau couple $(s,a)$ :

- une moyenne de transition prévue ;
- une variance de transition prévue.

La moyenne dit « voici le changement d'état le plus probable ». La variance dit « voici à quel
point je suis incertain ». C'est cette incertitude qui rend PILCO très différent d'un simple réseau
de neurones de dynamique : le modèle ne donne pas seulement une prédiction, il donne aussi sa
confiance.

Et cette confiance est indispensable pour la suite de PILCO, parce qu'on ne va pas seulement prédire
un prochain état ponctuel. On va propager une **distribution d'états** à travers le modèle. Le GP
est donc le moteur probabiliste qui permet à PILCO de planifier avec très peu de données.


In [ ]:
class GP(torch.nn.Module):
    # GP régression à kernel SE-ARD (hyperparamètres en log pour rester positifs).
    def __init__(self, input_dim):
        super().__init__()
        self.log_ell = torch.nn.Parameter(torch.zeros(input_dim))   # length-scales
        self.log_sf  = torch.nn.Parameter(torch.zeros(()))          # log signal std
        self.log_sn  = torch.nn.Parameter(torch.log(torch.tensor(0.1)))  # log noise std

    def kernel(self, X1, X2):
        ell = torch.exp(self.log_ell)
        d = (X1[:, None, :] - X2[None, :, :]) / ell
        return torch.exp(2 * self.log_sf) * torch.exp(-0.5 * (d ** 2).sum(-1))

    def set_data(self, X, y):
        self.X, self.y = X, y

    def _chol(self):
        n = self.X.shape[0]
        K = self.kernel(self.X, self.X) + (torch.exp(2 * self.log_sn) + 1e-6) * torch.eye(n)
        return torch.linalg.cholesky(K)            # K = L Lᵀ (cf. I.9)

    def nlml(self):
        # -log p(y|X) = ½ yᵀα + ½ log|K| + n/2 log 2π
        n = self.X.shape[0]; L = self._chol()
        alpha = torch.cholesky_solve(self.y[:, None], L)[:, 0]
        return 0.5 * (self.y @ alpha) + torch.log(torch.diagonal(L)).sum() + 0.5 * n * math.log(2 * math.pi)

    def fit(self, steps=60):
        opt = torch.optim.LBFGS(self.parameters(), max_iter=steps, line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad(); loss = self.nlml(); loss.backward(); return loss
        opt.step(closure)
        return float(self.nlml())

    def predict(self, Xs):
        L = self._chol()
        alpha = torch.cholesky_solve(self.y[:, None], L)[:, 0]
        Ks = self.kernel(self.X, Xs)                          # [N, M]
        mean = Ks.t() @ alpha                                 # K_*ᵀ α
        v = torch.cholesky_solve(Ks, L)                       # (K+σ²I)⁻¹ K_*
        var = torch.exp(2 * self.log_sf) - (Ks * v).sum(0)    # K_** − K_*ᵀ(...)K_*
        return mean, var.clamp_min(0.0)

print("classe GP définie")


In [ ]:
# Mini-test : un GP qui apprend sin(x) doit interpoler les données et être incertain ailleurs
Xtr = torch.linspace(-3, 3, 12).unsqueeze(1)
ytr = torch.sin(Xtr).squeeze(1)
gp = GP(input_dim=1); gp.set_data(Xtr, ytr)
nlml0 = float(gp.nlml()); nlml1 = gp.fit(steps=80)
m_tr, _ = gp.predict(Xtr)
rmse = float(((m_tr - ytr) ** 2).mean().sqrt())
_, v_near = gp.predict(torch.zeros(1, 1)); _, v_far = gp.predict(torch.tensor([[9.0]]))
print(f"NLML : {nlml0:.2f} -> {nlml1:.2f}  (apprentissage des hyperparamètres)")
print(f"RMSE sur les données : {rmse:.4f}  (interpolation quasi parfaite)")
print(f"variance près des données : {float(v_near):.3f}  | loin (x=9) : {float(v_far):.3f}")
assert nlml1 < nlml0 and rmse < 1e-2 and float(v_far) > float(v_near)
print("✓ le GP interpole les données ET signale son ignorance ailleurs")

In [ ]:
# LA visualisation canonique du GP : moyenne ± 2σ qui se resserre sur les données
Xs = torch.linspace(-6, 6, 300).unsqueeze(1)
mean, var = gp.predict(Xs)
mean, std = mean.detach().numpy(), var.detach().sqrt().numpy()
xs_np = Xs.numpy().ravel()
plt.figure(figsize=(9, 4))
plt.fill_between(xs_np, mean - 2 * std, mean + 2 * std, alpha=0.25, color="steelblue", label="±2σ (incertitude)")
plt.plot(xs_np, mean, "b", lw=2, label="moyenne prédite $\\bar f_*$")
plt.plot(xs_np, np.sin(xs_np), "g--", lw=1, label="vraie fonction sin(x)")
plt.scatter(Xtr.numpy(), ytr.numpy(), c="red", zorder=5, label="données observées")
plt.title("Posterior GP : certain près des données, incertain loin"); plt.xlabel("x"); plt.legend(); plt.show()

**Lecture.** C'est l'image à retenir du GP. La bande bleue (±2σ) **pince** à zéro sur chaque
point rouge (on y connaît la valeur, à $\sigma_n$ près) et **s'évase** entre et au-delà des données
— là où le modèle extrapole. La moyenne suit la vraie fonction (verte) là où il y a de
l'information, puis **revient vers 0** (la moyenne du prior) faute de données. C'est cette bande
d'incertitude, propagée sur un horizon, que PILCO exploitera pour ne pas se faire piéger par un
modèle trop sûr de lui.

In [ ]:
# Effet du bruit d'observation σ_n : plus de bruit -> bande plus large, interpolation moins stricte
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, sn in zip(axes, [0.01, 0.2, 0.6]):
    g = GP(1); g.set_data(Xtr, ytr)
    with torch.no_grad(): g.log_sn.fill_(math.log(sn)); g.log_ell.fill_(0.0); g.log_sf.fill_(0.0)
    m, v = g.predict(Xs); m, s = m.detach().numpy(), v.detach().sqrt().numpy()
    ax.fill_between(xs_np, m - 2 * s, m + 2 * s, alpha=0.25, color="steelblue")
    ax.plot(xs_np, m, "b"); ax.scatter(Xtr.numpy(), ytr.numpy(), c="red", s=15)
    ax.set_title(f"σ_n = {sn}"); ax.set_xlabel("x")
plt.suptitle("Le bruit d'observation contrôle à quel point le GP « colle » aux données"); plt.tight_layout(); plt.show()

**Lecture.** À gauche ($\sigma_n$ minuscule), le GP *interpole* : il passe pile par chaque
point et la bande s'annule dessus. À droite ($\sigma_n$ grand), il *régularise* : il accepte de ne
pas passer exactement par les points (qu'il considère bruités) et garde une incertitude résiduelle
partout. $\sigma_n$ est donc un curseur **confiance dans les mesures** ↔ **lissage**. Mais comment
le régler, ainsi que $\ell$ et $\sigma_f$, sans tâtonner ? Réponse : la vraisemblance marginale.

## I.9 — Apprendre les hyperparamètres : la vraisemblance marginale

Les hyperparamètres $\theta = (\ell, \sigma_f, \sigma_n)$ pilotent tout le comportement du GP : la
length-scale $\ell$ règle la régularité, $\sigma_f$ l'amplitude du signal, $\sigma_n$ le bruit
d'observation. On ne les devine pas à la main : on **laisse les données les choisir**, en gardant
ceux qui rendent les observations les plus *probables* selon le modèle.

**« Vraisemblance marginale » : marginale par rapport à quoi ?** Le GP ne pose pas une seule
fonction $f$, mais une *distribution* sur les fonctions. Plutôt que de fixer une $f$ particulière,
on l'**intègre** — on la marginalise sur toutes ses possibilités, pondérées par le prior :

$$p(\mathbf{y}\mid X,\theta) = \int p(\mathbf{y}\mid f)\,p(f\mid X,\theta)\,\mathrm{d}f.$$

C'est le sens du mot *marginale* : on a **éliminé la fonction inconnue** par intégration. Cette
quantité répond à une question très concrète : *avant même de connaître $f$, quelle était la
plausibilité qu'une fonction tirée de ce prior, plus du bruit, produise exactement les $\mathbf{y}$
observés ?*

**Pourquoi le calcul est exact.** Le prior est gaussien et le bruit est gaussien ; l'intégrale se
résout donc en forme close. Les observations suivent une simple loi normale multivariée :

$$\mathbf{y}\sim\mathcal{N}\!\big(\mathbf{0},\,K+\sigma_n^2 I\big).$$

La log-vraisemblance marginale n'est alors **rien d'autre que la log-densité de cette gaussienne**
évaluée au vecteur observé $\mathbf{y}$, ce qui donne trois termes :

$$\log p(\mathbf{y}\mid X, \theta) =
\underbrace{-\tfrac{1}{2}\mathbf{y}^\top (K+\sigma_n^2 I)^{-1}\mathbf{y}}_{\text{ajustement aux données}}
\;\underbrace{-\tfrac{1}{2}\log|K+\sigma_n^2 I|}_{\text{pénalité de complexité}}
\;\underbrace{-\tfrac{n}{2}\log 2\pi}_{\text{constante}}$$

C'est le **rasoir d'Occam automatique**. Lisons les deux termes qui comptent :

- l'**ajustement** $-\tfrac12\mathbf{y}^\top(K+\sigma_n^2 I)^{-1}\mathbf{y}$ est une distance de
  Mahalanobis : il récompense les modèles qui expliquent bien $\mathbf{y}$, et s'améliore si on
  réduit $\ell$ pour coller aux données ;
- la **complexité** $-\tfrac12\log|K+\sigma_n^2 I|$ est le logarithme d'un **déterminant**,
  c'est-à-dire le produit des valeurs propres de la covariance — une sorte de **volume** des jeux
  de données que le modèle juge plausibles. Un modèle très souple ($\ell$ minuscule) peut expliquer
  une immense variété de signaux ; il doit donc **étaler** sa masse de probabilité, et chaque jeu de
  données particulier en reçoit moins. Ce terme pénalise exactement cet étalement.
- la **constante** $-\tfrac{n}{2}\log 2\pi$ n'est pas un terme qu'on *ajoute* par choix : elle
  **tombe mécaniquement** de la densité gaussienne. La loi normale multivariée s'écrit
  $$p(\mathbf{y})=(2\pi)^{-n/2}\,|K+\sigma_n^2 I|^{-1/2}\exp\!\big(-\tfrac12\mathbf{y}^\top(K+\sigma_n^2 I)^{-1}\mathbf{y}\big),$$
  et le facteur de normalisation $(2\pi)^{-n/2}$ — celui qui garantit que la densité **intègre à 1** —
  devient, une fois passé au logarithme, exactement $-\tfrac{n}{2}\log 2\pi$. Elle ne dépend que du
  **nombre de points** $n$, **jamais** des hyperparamètres $\theta$ : lors de l'optimisation sur
  $\theta$ elle est donc **constante** et n'influence pas le minimum (on pourrait l'ignorer pour
  *trouver* $\theta$). On la garde tout de même parce qu'elle est nécessaire pour que la valeur soit
  une **vraie log-probabilité** — par exemple si l'on veut comparer des modèles entraînés sur des
  jeux de tailles différentes, ou interpréter le nombre lui-même.

> Un modèle capable de tout expliquer n'explique rien en particulier : la pénalité de complexité
> est ce qui en tient compte.

L'optimum équilibre les deux : ni trop rigide (sous-apprentissage), ni trop souple
(sur-apprentissage). Point remarquable, **aucun jeu de validation n'est nécessaire** : ce compromis
est contenu dans un seul nombre calculé sur les données d'entraînement — c'est de la sélection de
modèle bayésienne. On minimise donc le **négatif** (`Negative Log Marginal Likelihood (NLML)`) par **L-BFGS**
(*Limited-memory Broyden–Fletcher–Goldfarb–Shanno*) — un algorithme de descente du second ordre
qui approxime la **matrice hessienne** (les courbures de la surface de perte) à partir des
gradients successifs, **sans jamais la stocker explicitement** (d'où le « Limited-memory » : on
ne garde qu'une poignée de vecteurs gradients récents au lieu d'une matrice $p\times p$ complète).
Il converge bien plus vite que la descente de gradient ordinaire sur des surfaces lisses comme la
NLML — c'est le choix standard pour les GP et pour PILCO. C'est exactement ce que fait `GP.fit`.

**Pourquoi cela compte pour PILCO.** Ces hyperparamètres seront ajustés sur les quelques
transitions réellement collectées. Or PILCO ne se sert pas seulement de la moyenne du GP, mais de
son **incertitude**, propagée sur des dizaines de pas. Une length-scale ou un bruit mal réglés
donneraient une confiance fausse — trop sûre, ou trop floue — et toute la planification reposerait
dessus. Bien lire $\theta$ dans les données, c'est rendre cette incertitude **digne de confiance**.
Visualisons ce paysage.

In [ ]:
# Surface 3D de la NLML en fonction de (length-scale, bruit) — on voit la vallée

ell_grid = np.linspace(0.2, 3.0, 50)
sn_grid = np.linspace(0.02, 1.0, 50)
Z = np.zeros((sn_grid.size, ell_grid.size))
probe = GP(1); probe.set_data(Xtr, ytr)
with torch.no_grad():
    probe.log_sf.fill_(0.0)
    for i, sn in enumerate(sn_grid):
        for j, el in enumerate(ell_grid):
            probe.log_ell.fill_(math.log(el)); probe.log_sn.fill_(math.log(sn))
            Z[i, j] = float(probe.nlml())

EL, SN = np.meshgrid(ell_grid, sn_grid)
jstar, istar = np.unravel_index(np.argmin(Z), Z.shape)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(EL, SN, Z, cmap="viridis_r", alpha=0.85, edgecolor="none")
ax.scatter([ell_grid[istar]], [sn_grid[jstar]], [Z[jstar, istar]],
           color="red", s=120, marker="*", zorder=10, label="minimum")
ax.set_xlabel("length-scale ℓ"); ax.set_ylabel("bruit σ_n"); ax.set_zlabel("NLML")
ax.set_title("Paysage de la vraisemblance marginale (négatif = mieux)")
ax.view_init(elev=30, azim=-50)
ax.legend(); plt.tight_layout(); plt.show()
print(f"optimum sur grille : ℓ ≈ {ell_grid[istar]:.2f}, σ_n ≈ {sn_grid[jstar]:.2f}")


**Lecture.** En 3D on **voit** la vallée : la surface de la NLML forme un **creux allongé** —
l'optimiseur doit descendre au fond de cette cuvette pour trouver le bon compromis. Les flancs
montent des deux côtés : à gauche (ℓ minuscule), le modèle colle au bruit et le terme de
complexité explose ; à droite (ℓ énorme), le modèle est trop rigide et le terme d'ajustement
se dégrade. L'étoile rouge au fond de la vallée est le compromis optimal, celui que `fit()`
trouve par L-BFGS. **On n'impose donc jamais la rigidité du monde : on la lit dans les
données.**

## I.10 — Pourquoi on n'inverse jamais $K$ : la décomposition de Cholesky

Les formules du GP — postérieure et NLML — font toutes apparaître $(K + \sigma_n^2 I)^{-1}$.
L'idée naïve serait de calculer cet inverse, de le stocker, et de le multiplier par $\mathbf{y}$
ou $\mathbf{k}_*$. **C'est une erreur**, pour deux raisons indépendantes :

1. **Coût.** Inverser une matrice $n \times n$ coûte $\mathcal{O}(n^3)$ — le même ordre qu'une
   décomposition — mais **produit une matrice dense** $n \times n$ qu'il faut stocker et
   re-multiplier. Si l'on a besoin de $(K+\sigma_n^2 I)^{-1}\mathbf{y}$ pour plusieurs
   $\mathbf{y}$, on refait le produit à chaque fois.
2. **Stabilité numérique.** La matrice de covariance $K$ est définie-positive, mais peut être
   **mal conditionnée** : si deux points d'entraînement sont proches, leurs lignes de $K$ se
   ressemblent, certaines valeurs propres deviennent minuscules, et l'inverse **amplifie**
   l'erreur d'arrondi. En flottants 64 bits, on peut perdre toute précision ; en 32 bits
   (courant en deep learning), c'est encore pire.

**La parade standard : la décomposition de Cholesky.** Toute matrice **symétrique
définie-positive** $A$ se factorise en $A = L L^\top$, où $L$ est **triangulaire inférieure**
(des zéros au-dessus de la diagonale). Pour un GP, on pose $A = K + \sigma_n^2 I$ et on calcule
$L$ une seule fois ($\mathcal{O}(n^3/3)$, deux fois moins cher qu'une inversion complète).

Ensuite, **on ne forme jamais l'inverse**. On **résout** deux systèmes triangulaires par simple
substitution avant/arrière :

$$\text{pour obtenir } \boldsymbol{\alpha} = (K+\sigma_n^2 I)^{-1}\mathbf{y} :\quad
L\,\mathbf{v} = \mathbf{y}\;\text{(substitution avant)},\quad
L^\top\boldsymbol{\alpha} = \mathbf{v}\;\text{(substitution arrière)}.$$

Résoudre un système triangulaire est $\mathcal{O}(n^2)$ et **numériquement stable** : pas
d'amplification des erreurs d'arrondi. On obtient $\boldsymbol\alpha$ sans jamais toucher à
$A^{-1}$.

**Bonus : le log-déterminant.** La NLML contient $\log|K+\sigma_n^2 I|$. Le déterminant d'un
produit est le produit des déterminants, et celui d'une matrice triangulaire est le produit de sa
diagonale :

$$\log|A| = \log|L L^\top| = 2\log|L| = 2\sum_{i=1}^{n}\log L_{ii}.$$

On calcule le log-déterminant **gratuitement** à partir de la diagonale de $L$, déjà disponible
— pas de valeurs propres, pas de déterminant explicite, une seule ligne de code.

**Jitter.** Quand la matrice est *presque* singulière malgré $\sigma_n^2$ (points quasi-dupliqués,
noyau presque plat), Cholesky peut échouer. On ajoute alors un minuscule **jitter** $\varepsilon I$
($\varepsilon \sim 10^{-6}$) sur la diagonale — assez pour régulariser la factorisation, trop petit
pour changer les prédictions. C'est ce que fait notre `GP._cholesky()`.

Démonstration du contraste inversion vs Cholesky :

In [ ]:
# Deux points presque identiques -> K quasi singulière. Inversion vs Cholesky+jitter.
x_dup = torch.tensor([[0.0], [1e-7], [1.0], [2.0]])
y_dup = torch.tensor([0.0, 0.0, 1.0, 0.5])
Kd = (torch.exp(-0.5 * ((x_dup[:, None, 0] - x_dup[None, :, 0]) ** 2))).double()
print(f"conditionnement de K : {np.linalg.cond(Kd.numpy()):.2e}  (énorme -> inversion dangereuse)")
try:
    inv = torch.linalg.inv(Kd)
    alpha_inv = inv @ y_dup
    print(f"via inversion directe   : α = {np.round(alpha_inv.numpy(), 2)}  (valeurs absurdes / instables)")
except Exception as e:
    print("inversion directe a échoué :", e)
# Cholesky avec jitter : stable
L = torch.linalg.cholesky(Kd + 1e-6 * torch.eye(4))
alpha_chol = torch.cholesky_solve(y_dup[:, None], L)[:, 0]
print(f"via Cholesky + jitter   : α = {np.round(alpha_chol.numpy(), 2)}  (stable et raisonnable)")

**Lecture.** Avec un conditionnement gigantesque, l'inverse explicite amplifie les erreurs
d'arrondi et produit des poids $\alpha$ aberrants ; Cholesky + jitter reste **stable**. C'est
pourquoi `GP._chol` factorise au lieu d'inverser, et ajoute un jitter de sécurité. Cette robustesse
sera vitale dans la boucle PILCO, où le GP est réajusté des dizaines de fois sur des données qui
peuvent contenir des points très proches.

## I.11 — Plusieurs sorties : un GP par dimension

La dynamique d'un système renvoie un **vecteur** (pour le pendule, $\Delta x \in \mathbb{R}^3$).
PILCO fait le choix simple et efficace de modéliser **chaque dimension de sortie par un GP
indépendant**, partageant les mêmes entrées $X$ mais avec ses propres hyperparamètres.

Pourquoi ce choix ? Modéliser la covariance **entre** sorties coûterait beaucoup plus cher
($O(n^3 E^3)$ au lieu de $E \times O(n^3)$) pour un gain souvent marginal sur ces systèmes. On
suppose donc les sorties **conditionnellement indépendantes** sachant l'entrée — une approximation
raisonnable et très répandue.

In [ ]:
class MultiOutputGP:
    # Un GP indépendant par dimension de sortie ; entrées X partagées.
    def __init__(self, input_dim, output_dim):
        self.gps = [GP(input_dim) for _ in range(output_dim)]
    def fit(self, X, Y, steps=60):
        for e, gp in enumerate(self.gps):
            gp.set_data(X, Y[:, e]); gp.fit(steps=steps)
        return self
    def predict(self, Xs):
        outs = [gp.predict(Xs) for gp in self.gps]
        mean = torch.stack([m for m, _ in outs], dim=1)
        var = torch.stack([v for _, v in outs], dim=1)
        return mean, var

# Mini-test : deux sorties indépendantes apprises correctement
Xm = torch.rand(20, 2) * 4 - 2
Ym = torch.stack([torch.sin(Xm[:, 0]), torch.cos(Xm[:, 1])], dim=1)
mo = MultiOutputGP(2, 2).fit(Xm, Ym, steps=50)
mm, vv = mo.predict(Xm)
print("RMSE par sortie :", [round(float(((mm[:, e] - Ym[:, e]) ** 2).mean().sqrt()), 4) for e in range(2)])
assert mm.shape == (20, 2) and vv.shape == (20, 2)
print("✓ MultiOutputGP OK : chaque dimension interpolée indépendamment")

---
# Partie II — Apprendre la dynamique d'un vrai système

On tient notre modèle probabiliste. Branchons-le maintenant sur un environnement réel de contrôle continu :
**`InvertedPendulum-v5`**.

Ce choix est important. Après CartPole, on garde une intuition physique familière — un chariot qui doit
maintenir un pendule vertical — mais on change la nature de l'action. Dans CartPole, l'agent choisissait
entre deux actions discrètes : pousser à gauche ou pousser à droite. Ici, l'action est une **force continue**
dans l'intervalle $[-3, 3]$.

C'est exactement le territoire naturel de PILCO : apprendre un modèle de dynamique continu

$$
(x_t, u_t) \longmapsto x_{t+1}
$$

avec très peu de transitions réelles, puis optimiser une politique continue directement dans ce modèle.

| Propriété | Valeur |
|-----------|--------|
| Observation brute $x$ | $[p,\ \theta,\ \dot p,\ \dot\theta]$ — position du chariot, angle du pendule, vitesses |
| Observation interne | $[p,\ \sin\theta,\ \cos\theta,\ \dot p,\ \dot\theta]$ |
| Action $u$ | force continue dans $[-3, 3]$ |
| Récompense réelle | $+1$ par pas tant que le pendule reste debout |
| Terminaison réelle | chute si $|\theta| > 0.2$ rad ou état non fini |
| Horizon d'évaluation | jusqu'à 1000 pas |

## Pourquoi encoder l'angle avec $\sin\theta$ et $\cos\theta$ ?

L'observation brute contient directement l'angle $\theta$. Mathématiquement, c'est pratique ; pour un
modèle de régression, c'est moins idéal. Les angles vivent sur un cercle : $\theta = \pi$ et
$\theta = -\pi$ sont très proches physiquement, mais très éloignés numériquement si on les donne tels quels.

On encode donc l'angle par deux coordonnées :

$$
\theta
\quad\longmapsto\quad
(\sin\theta,\ \cos\theta)
$$

L'état interne devient :

$$
[p,\ \sin\theta,\ \cos\theta,\ \dot p,\ \dot\theta]
$$

Cela rend la géométrie de l'angle plus douce pour le GP. Mais cela introduit aussi une contrainte :

$$
\sin^2\theta + \cos^2\theta = 1
$$

Pendant les rollouts imaginés, le modèle prédit une variation $\Delta x$. Rien ne garantit que
$(\sin\theta, \cos\theta)$ reste exactement sur le cercle unité après plusieurs pas. On ajoute donc une
petite fonction de projection `project_ip_encoded` pour remettre cette paire sur le cercle. Sans cette
projection, le modèle pourrait inventer des états impossibles, par exemple une représentation avec
$\sin^2\theta + \cos^2\theta = 0.4$ ou $1.8$. La politique optimiserait alors une physique fictive.

## Pourquoi introduire `FixedHorizonNoEarlyTermination` ?

Il faut distinguer deux usages de l'environnement.

### 1. L'évaluation réelle

Pour mesurer la performance de la politique, on utilise l'environnement Gymnasium normal :

$$
\text{retour} \approx \text{nombre de pas avant la chute}
$$

Si le pendule tombe après 37 pas, l'épisode s'arrête et le retour vaut environ 37. C'est la vraie
métrique. On ne triche pas avec elle.

### 2. La collecte pour apprendre le modèle

Pour entraîner un GP de dynamique, on a besoin de transitions variées :

$$
(x_t, u_t, x_{t+1})
$$

Le problème est qu'au début, une politique aléatoire ou mal entraînée fait tomber le pendule très vite.
Si l'environnement s'arrête immédiatement à chaque chute, le dataset contient surtout les premiers pas
près du reset, puis plus rien. Le modèle apprend alors très mal ce qui se passe **pendant** et **après**
la chute.

Le wrapper `FixedHorizonNoEarlyTermination` sert uniquement à cette collecte modèle. Il ne remet pas
le pendule debout, ne change pas l'état initial, et ne donne pas de meilleur score. Il dit simplement :

> continue la même simulation pendant quelques pas, même si Gymnasium aurait déclaré `terminated=True`.

On obtient donc des trajectoires plus longues et plus informatives pour le GP. Le modèle voit des états
proches de la chute, des vitesses plus grandes, et des effets d'action plus variés. C'est utile parce que
PILCO optimise ensuite dans son modèle : si le GP ne connaît que la minuscule zone autour du reset, la
politique peut facilement exploiter des extrapolations absurdes.

En résumé :

| Usage | Environnement | Pourquoi |
|-------|---------------|----------|
| Collecte modèle | `FixedHorizonNoEarlyTermination` | obtenir des transitions plus riches pour apprendre la dynamique |
| Évaluation | Gymnasium normal | mesurer honnêtement combien de temps la politique reste debout |

Cette séparation est importante : le wrapper aide le **modèle** à apprendre, mais ne rend pas la
politique meilleure dans la métrique finale. La performance réelle reste toujours évaluée avec les
terminaisons normales.

In [ ]:
def encode_ip_obs(obs):
    """[x, theta, x_dot, theta_dot] -> [x, sin(theta), cos(theta), x_dot, theta_dot]."""
    obs = np.asarray(obs, dtype=np.float64)
    return np.array([obs[0], np.sin(obs[1]), np.cos(obs[1]), obs[2], obs[3]], dtype=np.float64)

def project_ip_encoded(x):
    """Projection différentiable/NumPy-friendly sur sin²+cos²=1."""
    if isinstance(x, torch.Tensor):
        s, c = x[..., 1], x[..., 2]
        n = torch.sqrt((s * s + c * c).clamp_min(1e-8))
        return torch.cat([x[..., :1], (s / n).unsqueeze(-1), (c / n).unsqueeze(-1), x[..., 3:]], dim=-1)
    arr = np.asarray(x).copy()
    s, c = arr[..., 1], arr[..., 2]
    n = np.maximum(np.sqrt(s * s + c * c), 1e-8)
    arr[..., 1], arr[..., 2] = s / n, c / n
    return arr

class FixedHorizonNoEarlyTermination(gym.Wrapper):
    """Continue la même simulation après la chute, puis tronque à horizon fixe."""
    def __init__(self, env, horizon):
        super().__init__(env)
        self.horizon = int(horizon)
        self.steps = 0

    def reset(self, **kwargs):
        self.steps = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.steps += 1
        if not np.isfinite(np.asarray(obs)).all():
            return obs, reward, True, truncated, info
        terminated = False
        if self.steps >= self.horizon:
            truncated = True
        return obs, reward, terminated, truncated, info

env = gym.make("InvertedPendulum-v5")
obs, _ = env.reset(seed=0)
print("observation brute      :", np.round(obs, 4), "  [x, theta, x_dot, theta_dot]")
print("observation encodée    :", np.round(encode_ip_obs(obs), 4), "  [x, sinθ, cosθ, x_dot, θ_dot]")
print("espace d'action        :", env.action_space, "→ force continue dans [-3, 3]")
print("sin²+cos² encodé       :", round(float(encode_ip_obs(obs)[1] ** 2 + encode_ip_obs(obs)[2] ** 2), 6))
print("\n3 transitions aléatoires avec terminaison naturelle :")
for t in range(3):
    u = env.action_space.sample()
    nobs, r, term, trunc, _ = env.step(u)
    print(f"  pas {t}: u={u[0]:+.2f}  r={r:+.1f}  done={term or trunc}  obs'={np.round(nobs,4)}")
    obs = nobs

env.close()

# Mini-test : la projection corrige un état encodé impossible.
bad = torch.tensor([[0.0, 0.3, 0.4, 0.0, 0.0]], dtype=torch.float64)
fixed = project_ip_encoded(bad)
assert abs(float(fixed[0, 1] ** 2 + fixed[0, 2] ** 2) - 1.0) < 1e-8
print("✓ encodage/projection OK : les rollouts imaginés restent sur le cercle unité")


## II.1 — Prédire la *variation* $\Delta x$, pas l'état absolu

Un détail qui n'en est pas un : PILCO n'apprend pas $x' = f(x, u)$ directement, mais la **variation encodée**

$$\Delta x = \operatorname{encode}(x') - \operatorname{encode}(x).$$

Le GP modélise donc $(x, u) \mapsto \Delta x$, et on reconstruit l'état imaginé par

$$x_{t+1}^{\text{enc}} = \operatorname{project}\bigl(x_t^{\text{enc}} + \Delta x_t\bigr).$$

La projection est indispensable ici : le modèle prédit deux composantes séparées pour $(\sin\theta, \cos\theta)$. Sans correction, une somme résiduelle peut donner une paire qui ne correspond à aucun angle réel. C'est exactement le genre de petit bug géométrique qui produit une belle baisse de coût imaginé mais une politique médiocre dans l'environnement.

In [ ]:
# Collecte de transitions aléatoires, puis entraînement d'un MultiOutputGP sur Δx encodé.
def collect_transitions(env_id="InvertedPendulum-v5", n_steps=120, seed=0, fixed_horizon=None):
    base = gym.make(env_id)
    e = FixedHorizonNoEarlyTermination(base, fixed_horizon) if fixed_horizon else base
    o_raw, _ = e.reset(seed=seed)
    e.action_space.seed(seed)
    o = encode_ip_obs(o_raw)
    X, Y = [], []
    for _ in range(n_steps):
        u = e.action_space.sample()
        no_raw, _, term, trunc, _ = e.step(u)
        no = encode_ip_obs(no_raw)
        X.append(np.concatenate([o, u]))
        Y.append(project_ip_encoded(no) - o)      # cible = Δx encodé
        o = no
        if term or trunc:
            o_raw, _ = e.reset()
            o = encode_ip_obs(o_raw)
    e.close()
    return torch.tensor(np.asarray(X), dtype=torch.float64), torch.tensor(np.asarray(Y), dtype=torch.float64)

Xdyn, Ydyn = collect_transitions(n_steps=160, seed=0, fixed_horizon=80)
dyn_gp = MultiOutputGP(input_dim=6, output_dim=5).fit(Xdyn, Ydyn, steps=40)

Xtest, Ytest = collect_transitions(n_steps=80, seed=99, fixed_horizon=80)
pred, _ = dyn_gp.predict(Xtest)
rmse = [float(((pred[:, e] - Ytest[:, e]) ** 2).mean().sqrt()) for e in range(5)]
print("RMSE de prédiction de Δx encodé par dimension :", [round(r, 5) for r in rmse])
print("length-scales apprises (GP dim 0) :", np.round(torch.exp(dyn_gp.gps[0].log_ell).detach().numpy(), 2))
assert pred.shape == Ytest.shape == (80, 5)
print("✓ dynamique GP OK : entrée 6D [état encodé, action] -> delta 5D")

**Lecture.** Le GP reçoit maintenant exactement ce que la version packagée reçoit : un état encodé 5D, une action continue, et une cible $\Delta x$ encodée. La RMSE est lue par dimension, car $p$, $\sin\theta$, $\cos\theta$, $\dot p$ et $\dot\theta$ n'ont pas la même échelle. Si les dimensions angle dérivent mal, la projection protège la géométrie, mais elle ne remplace pas un bon modèle : une RMSE élevée sur $\sin\theta$ ou $\cos\theta$ indique que les rollouts imaginés vont vite devenir trompeurs.

Ce mini-test n'est pas encore un agent. Il vérifie seulement que la brique “modèle de dynamique” apprend le bon type de signal avant de l'utiliser dans le moment matching.

---

# Partie III — Propager l'incertitude : le moment matching

## III.1 — Pourquoi une *distribution* doit voyager dans le temps

PILCO cherche à optimiser une politique sur un horizon de $T$ pas. Il part d'un état initial
incertain $x_0 \sim \mathcal{N}(\mu_0, \Sigma_0)$ — incertain parce que les capteurs sont
bruités, parce que l'état initial lui-même peut varier entre essais. À chaque pas de temps,
**deux sources d'incertitude s'ajoutent** :

**Source 1 — l'incertitude sur l'état courant** $x_t$. On ne sait pas exactement où on se trouve
dans l'espace d'état, et cette ignorance ne se réduit pas toute seule : à chaque transition, elle
se propage et peut s'amplifier.

**Source 2 — l'incertitude du modèle GP** sur la dynamique. Même si on connaissait $x_t$
exactement, le GP ne connaît pas parfaitement $f$ — surtout dans les régions de l'espace d'état
qui n'ont pas été visitées pendant l'apprentissage. Cette incertitude est *aleatoire en x* : elle
est faible près des données d'entraînement, grande ailleurs.

Ces deux sources ne s'additionnent pas simplement ; elles **se composent** : l'incertitude sur
$x_t$ fait en sorte qu'on évalue le GP dans des régions variées, ce qui module l'incertitude de
modèle elle-même. Le résultat : l'incertitude totale **grossit** avec l'horizon $T$.

> **Analogie météo.** Si la prévision de lundi est déjà incertaine, celle de mardi — qui en
> dépend — l'est encore plus, et celle de vendredi est presque inutilisable. La météo est
> un problème chaotique ; la dynamique robotique non linéaire se comporte de façon similaire.

**Pourquoi ne pas simplement faire du Monte-Carlo ?** L'approche la plus naturelle serait
d'échantillonner des trajectoires : tirer $x_0$, appliquer la dynamique, tirer $x_1$, etc. Ça
marche, mais c'est **trop lent pour l'optimisation**. Pour estimer le gradient de la politique
$\nabla_\theta J(\pi_\theta)$, il faut des milliers de trajectoires *et* leurs gradients, à chaque
itération de l'optimiseur. Monte-Carlo est exact mais pas différentiable analytiquement.

PILCO fait quelque chose d'élégant : **propager analytiquement une gaussienne à travers le GP**,
pas à pas. La question devient alors : comment passer une gaussienne à travers une fonction non
linéaire ?

---

## III.2 — Le problème central : une gaussienne traverse une non-linéarité

Formalisons. À l'étape $t$, on maintient l'approximation :

$$p(x_t) \approx \mathcal{N}(x_t \mid \mu_t,\, \Sigma_t).$$

La dynamique apprise par le GP est $\Delta x \approx f(x_t, u_t)$, ce qui donne
$x_{t+1} = x_t + f(x_t, u_t)$. Rigoureusement, la distribution de l'état suivant s'obtient en
marginalisant sur $x_t$ :

$$p(x_{t+1}) = \int \underbrace{p(x_{t+1} \mid x_t, u_t)}_{\text{transition GP}} \; \underbrace{p(x_t)}_{\mathcal{N}(\mu_t, \Sigma_t)} \; \mathrm{d}x_t.$$

Le problème est structurel : le GP est une fonction **non linéaire** de $x_t$, donc même si
$p(x_t)$ est gaussienne, la vraie $p(x_{t+1})$ peut être **multimodale, asymétrique, à queues
lourdes**. Elle n'est pas gaussienne en général.

```
  p(x_t)         f(x_t) non linéaire      p(x_{t+1}) réelle
  gaussienne   ─────────────────────────►  non gaussienne
  N(μ, Σ)                                 (peut avoir 2 modes, asymétrie, etc.)
```

Calculer exactement cette intégrale est **intractable**. La sortie du GP élevée à la moyenne
implique des intégrales de la forme $\int k(x, x_i)\, k(x, x_j)\, \mathcal{N}(x; \mu, \Sigma)\,
\mathrm{d}x$, qui n'admettent pas de forme close pour un noyau général.

**Le moment matching tranche ce nœud** : plutôt que de chercher la vraie distribution (impossible),
on calcule ses deux premiers moments — moyenne et variance — et on **réapproxime le résultat par
une gaussienne** de même moyenne et variance. On perd la forme exacte, mais :

1. on reste dans l'espace des gaussiennes → la propagation sur $T$ pas est entièrement analytique,
2. la gaussienne est paramétrée par $(\mu, \Sigma)$ → le tout est **différentiable** par rapport
   aux paramètres de la politique.

C'est le principe fondamental : **approcher la propagation, mais de façon exacte sur les moments**.

---

## III.3 — Pourquoi le noyau RBF rend les intégrales tractables

L'astuce technique de PILCO repose sur une propriété remarquable : **l'intégrale d'un noyau RBF
contre une gaussienne est elle-même une gaussienne**. Démontrons cela pour le cas 1D avant de
généraliser.

### La clé : intégrer le noyau contre la prior

Le noyau RBF de longueur de corrélation $\ell$ et variance $\sigma_f^2$ est :

$$k(x, x') = \sigma_f^2 \exp\!\left(-\frac{(x - x')^2}{2\ell^2}\right).$$

Calculons $E_{x \sim \mathcal{N}(\mu, \sigma^2)}[k(x, x_i)]$ — c'est-à-dire, la valeur moyenne du
noyau quand l'entrée est gaussienne :

$$q_i \;=\; \int k(x, x_i)\,\mathcal{N}(x;\,\mu,\,\sigma^2)\,\mathrm{d}x
         \;=\; \sigma_f^2 \int \exp\!\left(-\frac{(x-x_i)^2}{2\ell^2}\right)
               \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right) \mathrm{d}x.$$

C'est le produit de deux exponentielles quadratiques en $x$ : son intégrale est encore une
exponentielle quadratique (propriété des gaussiennes). Le résultat est :

$$\boxed{q_i = \sigma_f^2 \sqrt{\frac{\sigma^2}{\sigma^2 + \ell^2}}\exp\!\left(-\frac{(\mu - x_i)^2}{2(\sigma^2 + \ell^2)}\right).}$$

**Interprétation** : c'est une fonction gaussienne de $(\mu - x_i)$ avec variance $\sigma^2 +
\ell^2$. Autrement dit, la « distance effective » entre le point de requête $\mu$ et le point
d'entraînement $x_i$ est élargie par l'incertitude $\sigma^2$ sur l'entrée. Quand l'état est
très incertain ($\sigma^2 \gg \ell^2$), tous les points d'entraînement contribuent de façon
presque uniforme — la prédiction est vague. Quand l'état est bien connu ($\sigma^2 \ll \ell^2$),
on retrouve le noyau standard.

### La moyenne prédite $\mu_{t+1}$

La moyenne du GP en $x$ est $m(x) = k(x, X)\,\boldsymbol\alpha$ où $\boldsymbol\alpha = (K +
\sigma_n^2 I)^{-1}\mathbf{y}$. La moyenne prédite pour $x_{t+1}$ est donc :

$$\mu_{t+1} = \mathbb{E}_{x_t}[m(x_t)] = \sum_{i=1}^n \alpha_i\,\mathbb{E}_{x_t}[k(x_t, x_i)]
            = \boldsymbol\alpha^\top \mathbf{q},$$

où $\mathbf{q} = (q_1, \ldots, q_n)^\top$ et chaque $q_i$ est calculé par la formule ci-dessus.
**C'est une forme close** : un produit scalaire entre les poids GP $\boldsymbol\alpha$ et les
intégrales noyau-gaussienne $q_i$.

### La variance prédite $\Sigma_{t+1}$ : décomposition de la variance totale

La variance de $x_{t+1}$ se décompose en deux termes distincts via la **loi de la variance
totale** :

$$\Sigma_{t+1} = \underbrace{\mathbb{V}_{x_t}\!\left[\mathbb{E}_f[f(x_t)]\right]}_{\text{(A) incertitude de l'état}} + \underbrace{\mathbb{E}_{x_t}\!\left[\mathbb{V}_f[f(x_t)]\right]}_{\text{(B) incertitude du modèle}}.$$

**Terme (A) — combien la moyenne du GP varie quand $x_t$ varie.** C'est la variance due au fait
qu'on ne sait pas exactement où on se trouve. Elle est calculée comme :

$$\mathbb{V}_{x_t}[m(x_t)] = \mathbb{E}_{x_t}[m(x_t)^2] - \left(\mathbb{E}_{x_t}[m(x_t)]\right)^2
= \boldsymbol\alpha^\top Q\,\boldsymbol\alpha - \mu_{t+1}^2,$$

où $Q_{ij} = \mathbb{E}_{x_t}[k(x_t, x_i)\,k(x_t, x_j)]$ — une intégrale de produit de deux
noyaux contre une gaussienne, qui admet aussi une forme close avec le noyau RBF.

**Terme (B) — l'incertitude moyenne du GP lui-même.** Même si on connaissait $x_t$ exactement,
le GP serait incertain sur la dynamique dans les régions peu visitées. Ce terme intègre cette
incertitude du modèle sur la distribution de $x_t$ :

$$\mathbb{E}_{x_t}[v(x_t)] = \mathbb{E}_{x_t}\!\left[k(x_t, x_t) - k(x_t, X)(K+\sigma_n^2 I)^{-1}k(X, x_t)\right],$$

qui se calcule également en forme close avec le noyau RBF.

### Ce qu'on obtient et ce qu'on perd

| | Monte-Carlo | Moment matching (PILCO) |
|---|---|---|
| Exactitude | Distribution exacte (asymptotique) | Exacte sur $\mu$ et $\Sigma$ uniquement |
| Coût | $O(N_{\text{samples}} \cdot T)$ trajectoires | $O(T \cdot n^2)$ (n = données GP) |
| Différentiabilité | Gradients bruités (REINFORCE) | Gradients analytiques exacts |
| Représentation | Particules | $(\mu_t, \Sigma_t)$ par pas |
| Limite | Lent pour optimiser | Perd l'information multimodale |

> **À retenir.** Le moment matching propage analytiquement $(\mu_t, \Sigma_t) \to (\mu_{t+1},
> \Sigma_{t+1})$ à chaque pas, en exploitant que $\int k_{\text{RBF}}(x, x_i)\,
> \mathcal{N}(x;\mu,\Sigma)\,\mathrm{d}x$ est une gaussienne en $(\mu - x_i)$ avec covariance
> élargie $\Sigma + \Lambda$. Le prix payé : on suppose que $p(x_{t+1})$ reste gaussienne — ce
> qui est faux en général, mais reste une bonne approximation tant que $\Sigma_t$ reste modérée.
> Si la politique envoie l'agent dans des régions où la dynamique est très non linéaire,
> l'approximation se dégrade.


In [ ]:
np.random.seed(42)

def f(x):
    """Dynamique jouet : x' = x + 0.5 sin(x)."""
    return x + 0.5 * np.sin(x)

N = 20_000

# ─── Un pas : distribution d'entrée gaussienne -> sortie déformée ─────────
mu_in, sig_in = 1.6, 0.55
x_in  = np.random.normal(mu_in, sig_in, N)
x_out = f(x_in) + np.random.normal(0, 0.04, N)
mu_mm, sig_mm = x_out.mean(), x_out.std()      # les moments « matchés »

# ─── Propagation sur T pas ────────────────────────────────────────────────
pts = np.random.normal(0.3, 0.08, N)
pm, ps, pl, ph = [], [], [], []
for _ in range(16):
    pm.append(pts.mean()); ps.append(pts.std())
    pl.append(np.percentile(pts, 2.5)); ph.append(np.percentile(pts, 97.5))
    pts = f(pts) + np.random.normal(0, 0.04, N)
ts = np.arange(16)
pm, ps, pl, ph = map(np.array, (pm, ps, pl, ph))


In [ ]:
# ─── Figure ───────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(13, 9.5))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.55, wspace=0.30,
                        height_ratios=[1.25, 0.80])
ax_fn   = fig.add_subplot(gs[0, 0])
ax_cmp  = fig.add_subplot(gs[0, 1])
ax_prop = fig.add_subplot(gs[1, :])

BLUE, ORANGE, RED = "royalblue", "darkorange", "crimson"

# ════════════════════════════════════════════════════════════════
# ① La non-linéarité distord la gaussienne d'entrée
# ════════════════════════════════════════════════════════════════
x_line = np.linspace(-0.6, 3.8, 500)
YBASE  = -1.15           # ligne de base de la densité d'entrée (en bas)
XBASE  = -0.55           # ligne de base de la densité de sortie (à gauche)
DENS_H = 0.70           # hauteur visuelle allouée aux densités

ax_fn.plot(x_line, f(x_line), color="steelblue", lw=2.6, zorder=3,
           label=r"$f(x)=x+0.5\sin x$")
ax_fn.plot(x_line, x_line, color="grey", lw=1, ls=":", alpha=0.6,
           label="identité (référence)")

# Densité d'ENTRÉE, dessinée horizontalement sous la courbe
xs = np.linspace(mu_in - 3.2*sig_in, mu_in + 3.2*sig_in, 300)
pin = norm.pdf(xs, mu_in, sig_in); pin = pin / pin.max() * DENS_H
ax_fn.fill_between(xs, YBASE, YBASE + pin, color=BLUE, alpha=0.5, zorder=2)
ax_fn.plot(xs, YBASE + pin, color=BLUE, lw=1.4, zorder=2)
ax_fn.text(mu_in, YBASE - 0.10, r"$p(x_t)=\mathcal{N}(\mu_t,\Sigma_t)$",
           ha="center", va="top", color=BLUE, fontsize=9.5)

# Densité de SORTIE (vraie), dessinée verticalement à gauche
kde = gaussian_kde(x_out)
ys  = np.linspace(np.percentile(x_out, 0.3), np.percentile(x_out, 99.7), 300)
pout_true  = kde(ys)
pout_gauss = norm.pdf(ys, mu_mm, sig_mm)
SC = DENS_H / pout_true.max()
ax_fn.fill_betweenx(ys, XBASE, XBASE + pout_true * SC,
                    color=ORANGE, alpha=0.45, zorder=2,
                    label=r"vraie $p(x_{t+1})$")
ax_fn.plot(XBASE + pout_gauss * SC, ys, color=RED, lw=2, ls="--", zorder=3,
           label=r"approx. $\mathcal{N}(\hat\mu,\hat\Sigma)$")

# Flèches : ±1σ et la moyenne, projetés à travers f (espacement inégal !)
for xi, c, lw in [(mu_in - sig_in, "#88a", 1.0),
                  (mu_in,          "#444", 1.6),
                  (mu_in + sig_in, "#88a", 1.0)]:
    yi = f(xi)
    ax_fn.plot([xi, xi], [YBASE + 0.05, yi], color=c, lw=lw, ls="-", alpha=0.5, zorder=1)
    ax_fn.annotate("", xy=(XBASE + 0.02, yi), xytext=(xi, yi),
                   arrowprops=dict(arrowstyle="->", color=c, lw=lw, alpha=0.6))

# f(μ) vs μ̂ : montrer que propager la moyenne seule est faux
ax_fn.plot(mu_in, f(mu_in), "o", ms=8, color="darkred", zorder=5)
ax_fn.annotate(r"$f(\mu_t)$", (mu_in, f(mu_in)), (mu_in + 0.25, f(mu_in) - 0.55),
               fontsize=9, color="darkred",
               arrowprops=dict(arrowstyle="->", color="darkred", lw=1))
ax_fn.axhline(mu_mm, color=RED, lw=1, ls=":", alpha=0.6)
ax_fn.text(3.55, mu_mm + 0.08, r"$\hat\mu_{t+1}$", color=RED, fontsize=9, ha="right")

ax_fn.set_xlim(XBASE - 0.15, 3.9)
ax_fn.set_ylim(YBASE - 0.55, 4.3)
ax_fn.set_xlabel(r"$x_t$  (état courant)", fontsize=10)
ax_fn.set_ylabel(r"$x_{t+1}$  (état suivant)", fontsize=10)
ax_fn.set_title(r"① $f$ non-linéaire $\Rightarrow$ la gaussienne se déforme"
                "\n" r"$\pm1\sigma$ atterrit à des hauteurs inégales $\to$ sortie asymétrique",
                fontweight="bold", fontsize=9.5)
ax_fn.legend(loc="upper left", fontsize=8.3, framealpha=0.92)

# ════════════════════════════════════════════════════════════════
# ② Vraie distribution vs gaussienne « matchée »
# ════════════════════════════════════════════════════════════════
y2 = np.linspace(mu_mm - 4.5*sig_mm, mu_mm + 4.5*sig_mm, 400)
ax_cmp.fill_between(y2, 0, kde(y2), color=ORANGE, alpha=0.45,
                    label="vraie $p(x_{t+1})$  (Monte-Carlo)")
ax_cmp.plot(y2, norm.pdf(y2, mu_mm, sig_mm), color=RED, lw=2.5, ls="--",
            label=r"$\mathcal{N}(\hat\mu,\hat\Sigma)$  (moment matching)")
ax_cmp.axvline(mu_mm, color="darkred", lw=1.4, ls=":")
for k, a in ((1, 0.10), (2, 0.05)):
    ax_cmp.axvspan(mu_mm - k*sig_mm, mu_mm + k*sig_mm, color=RED, alpha=a)

ax_cmp.text(0.035, 0.97,
            "On garde (exact, analytique) :\n"
            r"   $\hat\mu_{t+1},\ \hat\Sigma_{t+1}$" "\n"
            "On sacrifie :\n"
            "   asymétrie, queues, modes\n"
            "Gain :\n"
            "   propagation différentiable\n"
            "   sur T pas, sans échantillonner",
            transform=ax_cmp.transAxes, ha="left", va="top", fontsize=8.2,
            color="#333",
            bbox=dict(boxstyle="round,pad=0.45", fc="lightyellow",
                      ec="#cca", alpha=0.92))
ax_cmp.set_xlabel(r"$x_{t+1}$", fontsize=10)
ax_cmp.set_ylabel("densité", fontsize=10)
ax_cmp.set_yticks([])
ax_cmp.set_title("② Ré-approximer par une gaussienne de mêmes moments",
                 fontweight="bold", fontsize=9.5)
ax_cmp.legend(fontsize=8.3, loc="upper right")

# ════════════════════════════════════════════════════════════════
# ③ Propagation sur T pas : l'incertitude cumule deux sources
# ════════════════════════════════════════════════════════════════
ax_prop.fill_between(ts, pl, ph, color=ORANGE, alpha=0.15, label="IC 95 %")
ax_prop.fill_between(ts, pm - ps, pm + ps, color=ORANGE, alpha=0.40,
                     label=r"$\pm1\sigma$")
ax_prop.plot(ts, pm, "o-", color="darkred", lw=2, ms=4, label=r"moyenne $\mu_t$")

t0 = 8
ax_prop.annotate(
    r"$\Sigma_{t+1}=\mathbb{V}_{x_t}[\mathbb{E}_f\,f]+\mathbb{E}_{x_t}[\mathbb{V}_f\,f]$"
    "\n"
    r"$\qquad\;\;$(état courant)$\quad\;$(modèle GP)",
    xy=(t0, pm[t0] + ps[t0]),
    xytext=(t0 + 1.7, pm[t0] - 4.6*ps[t0]),
    arrowprops=dict(arrowstyle="->", color="#555", lw=1.3),
    fontsize=10.5, ha="left", va="center",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#bbb", alpha=0.95))

ax_prop.set_xlabel("pas de temps $t$", fontsize=10)
ax_prop.set_ylabel("état", fontsize=10)
ax_prop.set_title(r"③ Répéter ①+② sur $T$ pas : $\Sigma_t$ croît "
                  "(deux sources d'incertitude s'accumulent)",
                  fontweight="bold", fontsize=9.5)
ax_prop.legend(fontsize=9, loc="upper left")

fig.suptitle(r"Moment matching : $(\mu_t,\Sigma_t)\ \longrightarrow\ "
             r"(\mu_{t+1},\Sigma_{t+1})$  —  le cœur de PILCO",
             fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


Lecture. Trois étapes dans l'ordre de lecture :

(haut gauche) — La croyance gaussienne sur $x_t$ (bleu, sous la courbe) est projetée à travers la dynamique non-linéaire $f$. Les flèches montrent comment trois quantiles ${\mu_t - \sigma, \mu_t, \mu_t + \sigma}$ atterrissent à des hauteurs inégalement espacées dans $f$ (la courbe est plus raide à gauche qu'à droite). La sortie vraie (orange, à droite de la courbe) est asymétrique : ce n'est plus une gaussienne. Notez que $f(\mu_t) \neq \hat\mu_{t+1}$ (point rouge) — propager seulement la moyenne est insuffisant.

(haut droite) — On compare la vraie distribution de $x_{t+1}$ (orange, Monte-Carlo) à l'approximation gaussienne $\mathcal{N}(\hat\mu, \hat\Sigma)$ (rouge 
tirets). Le moment matching calcule $\hat\mu$ et $\hat\Sigma$ exactement puis oublie le reste (asymétrie, queues lourdes). Le gain : on reste dans l'espace gaussien, les calculs restent analytiques et différentiables.

(bas) — On répète 1+2 sur $T$ pas. L'incertitude totale ($\pm 1\sigma$, orange foncé) grossit régulièrement parce que deux sources s'accumulent à chaque transition : $\mathbb{V}[\mathbb{E}[f(x_t)]]$ (on ne sait pas où on est) et $\mathbb{E}[\mathbb{V}[f(x_t)]]$ (le GP est incertain sur la dynamique). PILCO propage analytiquement cette décomposition, pas à pas, sans jamais échantillonner de trajectoires.

## III.2 — Le problème : entrée gaussienne, sortie *non* gaussienne

Si $x \sim \mathcal{N}(\mu, \Sigma)$ et qu'on calcule $y = f(x)$ avec $f$ **non-linéaire**, alors
$y$ **n'est plus gaussien** en général. Sa densité est l'intégrale

$$p(y) = \int p(y \mid x)\, p(x)\, dx,$$

qui n'a aucune raison d'être une cloche. Visualisons-le : un nuage gaussien (donc « elliptique »)
poussé à travers une non-linéarité devient une forme tordue, « en banane ».

In [ ]:
def cov_ellipse(ax, mean, cov, color, label=None, n_std=2.0):
    vals, vecs = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(vecs[1, np.argmax(vals)], vecs[0, np.argmax(vals)]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(mean, w, h, angle=ang, fill=False, edgecolor=color, lw=2.5, label=label))

inp = np.random.multivariate_normal([0, 0], [[0.35, 0], [0, 0.35]], size=3000)   # entrée gaussienne
out = np.stack([inp[:, 0], inp[:, 1] + 0.8 * inp[:, 0] ** 2], axis=1)            # non-linéarité
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(inp[:, 0], inp[:, 1], s=4, alpha=0.3, color="steelblue")
cov_ellipse(axes[0], inp.mean(0), np.cov(inp.T), "navy", "ellipse à 2σ")
axes[0].set_title("Entrée : nuage gaussien (elliptique)"); axes[0].legend(); axes[0].set_aspect("equal")
axes[1].scatter(out[:, 0], out[:, 1], s=4, alpha=0.3, color="darkorange")
cov_ellipse(axes[1], out.mean(0), np.cov(out.T), "darkred", "gaussienne moment-matchée")
axes[1].set_title("Sortie : nuage « banane » (NON gaussien)"); axes[1].legend(); axes[1].set_aspect("equal")
plt.tight_layout(); plt.show()


**Lecture.** À gauche, l'entrée est un nuage elliptique parfait. À droite, après la
non-linéarité, la sortie est une **banane** : asymétrique, incurvée — clairement pas une gaussienne.
L'ellipse rouge est la **meilleure gaussienne** qui résume ce nuage (même moyenne, même
covariance). C'est tout l'esprit du **moment matching** : on renonce à représenter exactement la
banane, on la **remplace par sa gaussienne de mêmes deux premiers moments**, et on continue. C'est
une approximation — d'autant meilleure que la non-linéarité est douce et l'incertitude modérée.

## III.3 — Le moment matching analytique pour un GP

Bonne nouvelle : pour un GP à kernel SE, on n'a **pas besoin** d'échantillonner pour calculer ces
deux moments — ils ont une **forme analytique fermée**. Soit une entrée $x \sim \mathcal{N}(\mu,
\Sigma)$ et un GP (poids $\boldsymbol{\beta}$, centres $x_i$, métrique $\Lambda = \mathrm{diag}(\ell_d^2)$).

**Moyenne de sortie.** On définit pour chaque centre $i$ un poids d'intégration $q_i$ — la
covariance du kernel entre le centre $x_i$ et l'entrée incertaine, *moyennée* sur $\mathcal{N}(\mu,
\Sigma)$ :

$$q_i = \sigma_f^2\,\sqrt{\frac{|\Lambda|}{|\Sigma + \Lambda|}}\;
\exp\!\left(-\tfrac12 (x_i - \mu)^\top (\Sigma + \Lambda)^{-1}(x_i - \mu)\right),
\qquad M = \boldsymbol{\beta}^\top \mathbf{q}.$$

| Terme | Lecture |
|-------|---------|
| $(x_i-\mu)^\top(\Sigma+\Lambda)^{-1}(x_i-\mu)$ | distance centre↔entrée, **élargie** par l'incertitude $\Sigma$ |
| $\sqrt{|\Lambda|/|\Sigma+\Lambda|}$ | facteur d'**atténuation** : plus $\Sigma$ est grand, plus les bumps s'aplatissent |
| $M = \boldsymbol\beta^\top\mathbf q$ | la moyenne de sortie reste une **somme pondérée de bumps**, comme la prédiction GP, mais lissée par $\Sigma$ |

**Covariance entrée–sortie.** On a aussi besoin de la corrélation entre $x$ et la sortie (pour
chaîner les pas) :

$$C = \Sigma\,(\Sigma + \Lambda)^{-1} \sum_i \beta_i\, q_i\, (x_i - \mu).$$

**Covariance de sortie.** Pour deux sorties $a, b$, on introduit la matrice
$R = \Sigma(\Lambda_a^{-1} + \Lambda_b^{-1}) + I$ et

$$Q_{ij} = \frac{\sigma_{f,a}^2\,\sigma_{f,b}^2}{\sqrt{|R|}}\,
\exp\!\Big(\!-\tfrac12 z_i^\top\!\Lambda_a^{-1} z_i -\tfrac12 z_j^\top\!\Lambda_b^{-1} z_j
+ \tfrac12 (\ldots)^\top R^{-1}\Sigma(\ldots)\Big),
\qquad S_{ab} = \boldsymbol\beta_a^\top Q\, \boldsymbol\beta_b - M_a M_b\,(+\text{var. GP}).$$

Pas besoin de retenir chaque symbole par cœur ; l'**intuition** suffit :

- **si $\Sigma \to 0$** (entrée quasi certaine), $q_i \to k(x_i, \mu)$ : on retombe **exactement**
  sur la prédiction GP ponctuelle. Le moment matching est alors *exact*.
- **si $\Sigma$ grandit**, les bumps s'aplatissent, $M$ se lisse, et la variance de sortie $S$
  **augmente** — l'incertitude d'entrée se propage bien vers la sortie.
- sur la diagonale, $S_{aa}$ ajoute la **variance propre du GP** (son incertitude épistémique),
  moyennée sur l'entrée : *l'ignorance du modèle est, elle aussi, propagée*.

Ces moments sont **exacts** pour le kernel SE (aucune approximation MC) ; la seule approximation est
de *re-gaussianiser* la sortie. Codons-les.

In [ ]:
def gaussian_moments(mo, mu, sigma):
    # Propage N(mu, sigma) à travers un MultiOutputGP. Retourne M [E], S [E,E], C [D,E].
    gps = mo.gps; centers = gps[0].X
    N, D = centers.shape; E = len(gps); eye = torch.eye(D)
    zeta = centers - mu                                   # x_i - mu  [N, D]
    betas, kinvs, ells, sf2s = [], [], [], []
    for gp in gps:
        L = gp._chol()
        betas.append(torch.cholesky_solve(gp.y[:, None], L)[:, 0])   # (K+σ²I)⁻¹ y
        kinvs.append(torch.cholesky_inverse(L))
        ells.append(torch.exp(gp.log_ell)); sf2s.append(torch.exp(2 * gp.log_sf))

    M = torch.zeros(E); C = torch.zeros(D, E); qs = []
    for a in range(E):
        lam = torch.diag(ells[a] ** 2)                    # Λ_a
        SpL = sigma + lam; SpL_inv = torch.linalg.inv(SpL)
        log_c = torch.log(sf2s[a]) + 0.5 * torch.logdet(lam) - 0.5 * torch.logdet(SpL)
        q = torch.exp(log_c - 0.5 * ((zeta @ SpL_inv) * zeta).sum(1))   # q_i  [N]
        qs.append(q); M[a] = betas[a] @ q
        C[:, a] = sigma @ SpL_inv @ ((betas[a] * q).unsqueeze(1) * zeta).sum(0)

    S = torch.zeros(E, E)
    for a in range(E):
        for b in range(a, E):
            ia = torch.diag(1.0 / ells[a] ** 2); ib = torch.diag(1.0 / ells[b] ** 2)
            R = sigma @ (ia + ib) + eye; R_inv = torch.linalg.inv(R)
            za = zeta @ ia; zb = zeta @ ib
            na = (za * zeta).sum(1); nb = (zb * zeta).sum(1)
            rs = R_inv @ sigma
            quad = ((za @ rs) * za).sum(1)[:, None] + 2 * (za @ rs @ zb.t()) + ((zb @ rs) * zb).sum(1)[None, :]
            log_Q = (torch.log(sf2s[a]) + torch.log(sf2s[b]) - 0.5 * torch.logdet(R)
                     - 0.5 * na[:, None] - 0.5 * nb[None, :] + 0.5 * quad)
            Qm = torch.exp(log_Q)
            val = betas[a] @ Qm @ betas[b] - M[a] * M[b]
            if a == b:                                    # + variance épistémique du GP
                val = val + sf2s[a] - torch.trace(kinvs[a] @ Qm)
            S[a, b] = S[b, a] = val
    return M, S, C

print("gaussian_moments défini (moment matching analytique)")

**Lecture.** La fonction retourne trois objets : **M** (moyenne de la sortie), **S** (sa
covariance, incertitude d'entrée *plus* incertitude du GP), et **C** (covariance entrée–sortie,
indispensable pour enchaîner les pas — on y reviendra en III.7). Vérifions que ce calcul analytique
dit bien la vérité, en le comparant à une simulation Monte-Carlo.

In [ ]:
# Viz décisive : moment matching analytique vs vérité Monte-Carlo, pour σ petit puis grand
gp1 = MultiOutputGP(1, 1).fit(torch.linspace(-3, 3, 14).unsqueeze(1),
                              torch.sin(1.4 * torch.linspace(-3, 3, 14)).unsqueeze(1), steps=80)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, sig in zip(axes, [0.15, 0.9]):
    mu0 = torch.tensor([0.4]); Sig = torch.tensor([[sig]])
    M, S, _ = gaussian_moments(gp1, mu0, Sig)

    # Monte-Carlo de la VRAIE loi de sortie : on échantillonne (1) l'entrée, puis
    # (2) la loi prédictive complète du GP f ~ N(m(x), v(x)) — les DEUX sources
    # d'incertitude, exactement ce que gaussian_moments propage analytiquement.
    xs_mc = mu0 + torch.randn(20000, 1) * math.sqrt(sig)
    mean_mc, var_mc = gp1.predict(xs_mc)                          # moyenne ET variance GP
    mean_mc = mean_mc[:, 0].detach()
    std_mc  = var_mc[:, 0].clamp_min(0.0).sqrt().detach()
    f_mc = (mean_mc + std_mc * torch.randn_like(mean_mc)).numpy()  # tirage prédictif complet

    ax.hist(f_mc, bins=60, density=True, alpha=0.5, color="steelblue", label="MC (vérité)")
    zz = np.linspace(f_mc.min(), f_mc.max(), 200)
    ax.plot(zz, gaussian_pdf(zz, float(M), float(S.sqrt())), "r", lw=2,
            label="gaussienne moment-matchée")
    ax.axvline(float(M), color="r", ls=":", lw=1)
    ax.set_title(f"σ² = {sig}"); ax.set_xlabel("sortie f(x)"); ax.legend()
plt.suptitle("Moment matching analytique vs Monte-Carlo (loi prédictive complète)")
plt.tight_layout(); plt.show()

Lecture — le compromis central. À gauche (faible incertitude d'entrée), la sortie réelle (histogramme) est quasi gaussienne : la courbe analytique rouge épouse à la fois sa position et sa largeur — le moment matching est quasi exact. À droite (forte incertitude), la même loi devient asymétrique ; la gaussienne rouge en restitue correctement la moyenne (trait pointillé) et la variance, mais ne peut pas rendre l'asymétrie ni les queues. C'est le prix de la tractabilité : on échange la forme exacte contre des équations fermées et différentiables, donc une optimisation rapide de la politique. En pratique, PILCO garde $\Sigma$ assez petit pour rester dans le régime de gauche.

In [ ]:
# Mini-test : en régime de faible incertitude, M et S analytiques collent au Monte-Carlo
mu0 = torch.tensor([0.4]); Sig = torch.tensor([[0.05]])
M, S, _ = gaussian_moments(gp1, mu0, Sig)
xs_mc = mu0 + torch.randn(60000, 1) * math.sqrt(0.05)
fm = gp1.predict(xs_mc)[0][:, 0]; vm = gp1.predict(xs_mc)[1][:, 0]
M_mc = fm.mean(); S_mc = fm.var() + vm.mean()                 # var(moyenne) + E[variance GP]
print(f"moyenne : analytique {float(M):.4f} | MC {float(M_mc):.4f}")
print(f"variance: analytique {float(S):.4f} | MC {float(S_mc):.4f}")
assert abs(float(M) - float(M_mc)) < 2e-2 and abs(float(S) - float(S_mc)) < 2e-2
print("✓ le moment matching analytique reproduit le Monte-Carlo (à l'erreur d'échantillonnage près)")

## III.4 — Le coût saturant : ne récompenser que le fait de *se rapprocher*

### Ce qu'on cherche à minimiser

PILCO optimise la politique en minimisant le **coût attendu sur tout l'horizon** :

$$J(\pi) = \sum_{t=0}^{T} \mathbb{E}_{x_t \sim \mathcal{N}(\mu_t, \Sigma_t)}\big[c(x_t)\big].$$

Point crucial : à chaque pas, $x_t$ n'est pas un point mais **une gaussienne** (c'est tout
l'objet de III.1–III.3). On ne paie donc jamais $c(\text{un état})$, mais l'**espérance de
$c$ sous la distribution propagée**. Le choix de la fonction $c$ doit donc être jugé non pas sur
sa valeur en un point, mais sur la façon dont son **espérance se comporte quand $\Sigma_t$
grandit**. C'est là que le quadratique et le saturant divergent radicalement.

### Le coût saturant

$$c(x) = 1 - \exp\!\left(-\tfrac12 (x - x^*)^\top W (x - x^*)\right) \in [0, 1]$$

| Terme | Rôle |
|-------|------|
| $x^*$ | état cible (pour le pendule : la verticale $[\cos\theta{=}1,\ \sin\theta{=}0,\ \dot\theta{=}0]$) |
| $W$ | matrice de **précision** : quelles dimensions comptent, et à quelle échelle (l'inverse d'une largeur²) |
| $1 - \exp(-\cdot)$ | coût **borné** dans $[0,1]$, valant $0$ à la cible et **saturant à $1$** au loin |

La forme : une cuvette inversée. Tout près de $x^*$, $c \approx \tfrac12 (x-x^*)^\top W (x-x^*)$
— il *imite* le quadratique. Mais loin de $x^*$, l'exponentielle s'éteint et $c$ **plafonne à
1** : passé une certaine distance, « assez loin » et « très très loin » coûtent pareil.

### Pourquoi pas le quadratique ? — le vrai argument est l'incertitude

Le défaut du quadratique $\|x-x^*\|^2$ n'est pas seulement qu'il « explose ». C'est sa façon
d'interagir avec l'incertitude propagée. Calculons l'espérance sous $x \sim \mathcal{N}(\mu,
\Sigma)$ :

$$\mathbb{E}\big[\|x - x^*\|^2\big] = \underbrace{\|\mu - x^*\|^2}_{\text{erreur de la moyenne}} + \underbrace{\operatorname{tr}(\Sigma)}_{\text{toujours} \ \geq 0}.$$

Le terme de variance **s'ajoute toujours**, quel que soit l'endroit. Conséquence : une politique
qui réduit son coût attendu a *toujours* intérêt à réduire $\Sigma$. Le quadratique **punit
l'incertitude partout** — il décourage l'agent d'explorer, même quand il est loin du but et que
l'exploration serait précisément ce qu'il faut.

Le coût saturant fait l'inverse, et c'est sa propriété décisive. Son espérance sous une gaussienne
a aussi une **forme close** (encore grâce à exp × gaussienne, comme en III.3) :

$$\mathbb{E}[c(x)] = 1 - \frac{1}{\sqrt{\det(I + \Sigma W)}}\,\exp\!\left(-\tfrac12 (\mu - x^*)^\top W (I + \Sigma W)^{-1} (\mu - x^*)\right).$$

Ce qui compte, c'est l'effet de $\Sigma$ selon où l'on est :

- **Loin de la cible** ($\mu$ éloigné de $x^*$) : **plus** d'incertitude **diminue** le coût
  attendu. Pourquoi ? Parce qu'une distribution large a un peu de masse qui « déborde » vers la
  cible, et le coût étant plafonné, ce gain n'est pas annulé par la masse partie de l'autre côté.
  → le coût saturant **encourage l'exploration** quand la politique est encore mauvaise.
- **Près de la cible** ($\mu \approx x^*$) : **plus** d'incertitude **augmente** le coût. Une
  fois proche du but, l'agent a intérêt à **resserrer** $\Sigma$, donc à être précis et fiable.
  → le coût saturant **récompense la précision** une fois le but en vue.

C'est exactement le comportement qu'on veut d'un apprenti : explorer largement tant qu'on est
loin, devenir précis une fois proche. Le quadratique fait le contraire (il pénalise l'exploration
dès le début), ce qui, combiné à un modèle encore imparfait, mène à des politiques timides et à
des gradients dominés par les rares états très éloignés.

> **À retenir.** Le coût saturant n'est pas choisi pour « borner » le coût par cosmétique. Il est
> choisi parce que son **espérance sous la gaussienne propagée** couple intelligemment incertitude
> et distance : exploration loin du but, exactitude près du but — et cette espérance reste
> **analytique et différentiable**, donc compatible avec l'optimisation par gradient de PILCO.



In [ ]:
def saturating_cost(x, target, W):
    d = x - target
    return 1.0 - torch.exp(-0.5 * (d @ W * d).sum(-1))

xx = torch.linspace(-4, 4, 300)
target1 = torch.tensor([0.0]); W1 = torch.tensor([[1.0]])
quad = 0.5 * xx ** 2
sat = saturating_cost(xx.unsqueeze(1), target1, W1)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(xx, quad, "--", color="gray", label="coût quadratique $\\frac{1}{2}x^2$ (non borné)")
ax.plot(xx, sat, "r", lw=2, label="coût saturant $1-e^{-x^2/2}$ (borné, plat au loin)")
ax.set_ylim(-0.1, 4); ax.set_xlabel("écart à la cible"); ax.set_ylabel("coût")
ax.set_title("Quadratique vs saturant"); ax.legend(); plt.show()

**Lecture.** Le quadratique (gris) monte sans limite : une politique qui hésite entre deux
mauvaises options très éloignées sera surtout guidée par « éviter l'option la plus lointaine ». Le
saturant (rouge) sature à 1 : passé une certaine distance, tout est « également mauvais », et le
seul moyen de baisser le coût est de **vraiment se rapprocher** de la cible. Comme PILCO porte une
*distribution* d'états, il lui faut le **coût espéré** $\mathbb{E}[c(x)]$ pour $x \sim
\mathcal{N}(\mu, \Sigma)$ — qui, lui aussi, est analytique :

$$\mathbb{E}[c(x)] = 1 - |I + \Sigma W|^{-1/2}\,
\exp\!\left(-\tfrac12 (\mu - x^*)^\top (\Sigma + W^{-1})^{-1}(\mu - x^*)\right).$$

In [ ]:
def expected_cost(mu, sigma, target, W):
    D = mu.shape[0]
    eye = torch.eye(D, dtype=mu.dtype, device=mu.device)
    sigma = _psd_project(sigma) if "_psd_project" in globals() else 0.5 * (sigma + sigma.T)
    d = mu - target

    # Forme symétrique de E[exp(-1/2 (x-target)^T W (x-target))].
    # Elle évite que l'optimiseur exploite des logdet négatifs dus à I+ΣW non symétrique.
    sqrt_w = torch.diag(torch.sqrt(torch.clamp(torch.diag(W), min=0.0))).to(dtype=mu.dtype, device=mu.device)
    A = eye + sqrt_w @ sigma @ sqrt_w
    A = 0.5 * (A + A.T) + 1e-8 * eye
    logdet = torch.linalg.slogdet(A).logabsdet
    inner = sqrt_w @ torch.linalg.solve(A, sqrt_w)
    quad = d @ inner @ d
    cost = 1.0 - torch.exp(-0.5 * logdet - 0.5 * quad)
    return torch.clamp(cost, 0.0, 1.0)

# Mini-test : coût espéré ∈ [0,1], nul à la cible sans incertitude, et = au Monte-Carlo
mu_t = torch.tensor([0.3, -0.2]); Sig_t = torch.tensor([[0.2, 0.05], [0.05, 0.15]])
tgt = torch.tensor([0.0, 0.0]); Wt = torch.eye(2)
ec = expected_cost(mu_t, Sig_t, tgt, Wt)
mc = saturating_cost(torch.distributions.MultivariateNormal(mu_t, Sig_t).sample((40000,)), tgt, Wt).mean()
print(f"coût espéré : analytique {float(ec):.4f} | MC {float(mc):.4f}")
assert 0 <= float(ec) <= 1 and abs(float(ec) - float(mc)) < 1e-2
assert float(expected_cost(tgt, 1e-9 * torch.eye(2), tgt, Wt)) < 1e-6
print("✓ coût saturant espéré OK (borné, nul à la cible, = Monte-Carlo)")

## III.5 — La politique : un réseau RBF (et pourquoi surtout pas un MLP)

### La contrainte qui décide de tout

Récapitulons la chaîne de propagation. À chaque pas, PILCO transforme une gaussienne d'états en
la gaussienne d'états suivante :

$$\mathcal{N}(\mu_t, \Sigma_t)\;\xrightarrow{\ \pi\ }\;\text{(état, action)}\;\xrightarrow{\ \text{GP}\ }\;\mathcal{N}(\mu_{t+1}, \Sigma_{t+1}).$$

La **dynamique** (le GP) sait faire ce passage analytiquement — c'est tout III.3. Mais regarde le
premier maillon : avant d'atteindre le GP, il faut **passer la gaussienne d'états à travers la
politique** $u = \pi(x)$ pour obtenir la distribution jointe $(x, u)$. Si ce maillon n'est pas
analytique, toute la chaîne casse.

Autrement dit, la politique a une **contrainte que les RL classiques n'ont pas** : il ne suffit
pas qu'elle calcule une action pour un état donné, il faut qu'on sache calculer
**analytiquement les moments de $\pi(x)$ quand $x$ est gaussien**. Cette exigence élimine
d'emblée la plupart des architectures.

### Pourquoi un MLP est disqualifié

Un MLP standard enchaîne des non-linéarités — `tanh`, `ReLU`, `sigmoid`. Demande-toi :
quelle est la moyenne de `tanh(x)` quand $x \sim \mathcal{N}(\mu, \sigma^2)$ ?

$$\mathbb{E}[\tanh(x)] = \int \tanh(x)\,\mathcal{N}(x;\mu,\sigma^2)\,\mathrm{d}x \;=\; ?$$

**Cette intégrale n'a pas de forme close.** Idem pour `ReLU`, `sigmoid`, etc. On serait obligé de
revenir au Monte-Carlo (échantillonner, faire passer dans le réseau, moyenner) — exactement ce
que PILCO veut éviter, parce que ça détruit la différentiabilité analytique et ralentit
l'optimisation. Un MLP rend la politique expressive mais **non intégrable**.

### Le réseau RBF : non-linéaire *et* intégrable

PILCO choisit donc une politique faite de **bosses gaussiennes** :

$$\pi_{\text{brut}}(x) = \sum_{j=1}^{m} w_j\,\exp\!\left(-\tfrac12 (x - c_j)^\top \Lambda_\pi^{-1}(x - c_j)\right).$$

| Terme | Rôle |
|-------|------|
| $c_j$ | **centres** des bosses, répartis dans l'espace d'état |
| $w_j$ | **poids** (amplitude, signée) de chaque bosse |
| $\Lambda_\pi$ | largeurs des bosses (mêmes longueurs de corrélation que dans un noyau RBF) |
| $m$ | nombre de bosses (≈ 50 pour le pendule) — contrôle la flexibilité |

L'observation clé : **cette forme est structurellement identique à la moyenne d'un GP**. Compare
avec la prédiction GP $m(x) = \sum_i \beta_i\, k(x, x_i)$ : les centres $c_j$ jouent le rôle des
points d'entraînement $x_i$, les poids $w_j$ celui de $\boldsymbol\beta$, et la bosse gaussienne
celui du noyau RBF. **La politique est, mathématiquement, une moyenne de GP fictif.**

Conséquence directe et décisive : propager une gaussienne à travers la politique utilise
**exactement les mêmes formules de moment matching** que la dynamique (les intégrales RBF ×
gaussienne de III.3). On réutilise le même outil. La politique est donc **non-linéaire** (somme
de bosses → elle peut approximer des contrôles complexes) **mais intégrable** (chaque morceau a
des moments gaussiens en forme close).

### Le dernier détail : borner l'action

Une somme de gaussiennes produit une sortie non bornée, alors que les actionneurs réels saturent
(couple max du moteur). PILCO ajoute donc un **écrasement** final différentiable, typiquement
$u = u_{\max}\,\sin(\pi_{\text{brut}}(x))$. Le `sin` est lui aussi choisi exprès : son espérance
sous une gaussienne **a une forme close** (contrairement à `tanh`), donc il préserve la
propriété d'intégrabilité de bout en bout.

> **À retenir.** La politique de PILCO n'est pas un réseau de neurones « parce que c'est à la
> mode en moins bien » : c'est un choix dicté par la propagation analytique. Tout, de la
> dynamique (GP) à la politique (RBF) jusqu'à l'écrasement (`sin`), est construit pour qu'une
> gaussienne traverse la chaîne entière en restant gaussienne par moment matching — et donc pour
> que $\nabla_\theta J$ se calcule **analytiquement**. L'expressivité d'un MLP n'a aucune valeur
> si elle brise cette chaîne.

### Politique canonique vs recette locale du notebook

Le **PILCO canonique** garde donc une politique **RBF** : c'est la bonne famille quand il faut un
contrôle franchement non linéaire, par exemple un **swing-up** depuis le bas puis une
stabilisation finale. Le notebook montre cette forme parce qu'elle reste la référence théorique du
papier de 2011 et de la version packagée.

Mais la recette qui marche ici sur `InvertedPendulum-v5` est plus locale : le reset officiel est
déjà près de l'upright, donc le problème empirique n'est pas de balancer le pendule depuis le bas
mais de **stabiliser proprement une petite perturbation**. Dans ce régime, on garde le moteur
PILCO intact et on simplifie seulement la famille de contrôleurs vers `LinearSinePolicy`. La RBF
reste indispensable pour un vrai swing-up ; la version linéaire bornée est simplement la recette
la plus stable et la plus sample-efficient pour ce notebook précis.


In [ ]:
class RBFPolicy(torch.nn.Module):
    def __init__(self, state_dim, action_dim, n_basis=20, action_high=2.0):
        super().__init__()
        self.centers = torch.nn.Parameter(torch.randn(n_basis, state_dim))
        self.weights = torch.nn.Parameter(0.5 * torch.randn(n_basis, action_dim))
        self.log_ell = torch.nn.Parameter(torch.zeros(state_dim))
        self.action_high = torch.as_tensor(action_high, dtype=torch.float64).reshape(-1)

    def raw_control(self, x):                       # x: [B, Dx] -> [B, Du]
        single = (x.dim() == 1); xb = x.unsqueeze(0) if single else x
        d = (xb[:, None, :] - self.centers[None, :, :]) / torch.exp(self.log_ell)
        phi = torch.exp(-0.5 * (d ** 2).sum(-1))     # [B, m] bosses
        out = phi @ self.weights                     # [B, Du]
        return out.squeeze(0) if single else out

    def forward(self, x):                            # action bornée par saturation sinus
        return self.action_high * torch.sin(self.raw_control(x))

# Illustration 1D : la politique brute est une somme de bosses ; la version finale est saturée
pol1d = RBFPolicy(state_dim=1, action_dim=1, n_basis=6, action_high=2.0)
with torch.no_grad():
    pol1d.centers.copy_(torch.linspace(-3, 3, 6).unsqueeze(1)); pol1d.weights.copy_(torch.tensor([[1.5],[-2.],[1.],[2.5],[-1.5],[1.]])); pol1d.log_ell.zero_()
xs = torch.linspace(-4, 4, 300).unsqueeze(1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for c, w in zip(pol1d.centers, pol1d.weights):
    axes[0].plot(xs, (w.item() * torch.exp(-0.5 * ((xs - c) ** 2).sum(-1))).detach(), "--", alpha=0.5)
axes[0].plot(xs, pol1d.raw_control(xs).detach(), "k", lw=2, label="$\\pi_{brut}$ = somme des bosses")
axes[0].set_title("Politique RBF brute = somme de bosses gaussiennes"); axes[0].legend()
axes[1].plot(xs, pol1d.raw_control(xs).detach(), "gray", label="brut (non borné)")
axes[1].plot(xs, pol1d.forward(xs).detach(), "r", lw=2, label="saturé dans [-2, 2]")
axes[1].axhline(2, color="k", ls=":"); axes[1].axhline(-2, color="k", ls=":")
axes[1].set_title("Saturation sinus → action bornée"); axes[1].legend(); plt.tight_layout(); plt.show()

**Lecture.** À gauche : chaque bosse (pointillés) couvre une région de l'espace d'état ;
leur somme pondérée (noir) forme une politique non-linéaire flexible. À droite : on **borne**
l'action via $u = u_{\max}\sin(\pi_{\text{brut}})$ — pour le pendule, le couple doit rester dans
$[-2, 2]$.

## III.6 — Pourquoi le sinus pour borner

On borne l'action avec un **sinus**, $u = u_{\max}\sin(v)$ où $v = \pi_{\text{brut}}(x)$, plutôt
qu'avec un `tanh`. Ce n'est pas un hasard : c'est le **dernier maillon** de la chaîne analytique
(III.5), et il doit lui aussi laisser passer une gaussienne en forme close. Pour une entrée
$v \sim \mathcal{N}(m, s)$, les moments de $\sin(v)$ s'écrivent exactement :

$$\mathbb{E}[\sin v] = e^{-s/2}\sin m, \quad
\operatorname{Var}[\sin v] = \tfrac12\bigl(1 - e^{-2s}\cos 2m\bigr) - e^{-s}\sin^2 m, \quad
\operatorname{Cov}[v, \sin v] = e^{-s/2}\, s\, \cos m.$$

Trois quantités, trois rôles dans la propagation :

- $\mathbb{E}[\sin v]$ donne la **moyenne de l'action** — ce qui alimente $\mu$ de l'entrée du GP de dynamique ;
- $\operatorname{Var}[\sin v]$ donne sa **variance** — la diagonale « action » de $\Sigma$ ;
- $\operatorname{Cov}[v, \sin v]$ relie l'action **brute** à l'action **bornée**, indispensable pour reconstruire la covariance jointe $(x, u)$ correctement (l'action est corrélée à l'état dont elle est issue).

Ces trois formules permettent de propager l'incertitude **à travers la saturation** sans rien
échantillonner : la politique reste entièrement analytique et différentiable, **même bornée**. Un
`tanh` n'offre pas d'expressions fermées comparables — c'est pour ça qu'il est écarté, exactement
comme dans le MLP (III.5).

### Le facteur $e^{-s/2}$ : une prudence automatique

Le terme le plus parlant est le facteur d'atténuation dans la moyenne :

$$\mathbb{E}[\sin v] = \underbrace{e^{-s/2}}_{\in\,(0,1]}\,\sin m.$$

Lis-le ainsi : $\sin m$ serait l'action si l'entrée était **certaine** ($s = 0$, donc
$e^{-s/2}=1$). Dès que l'entrée devient incertaine ($s > 0$), le facteur $e^{-s/2} < 1$ **tire
l'action moyenne vers zéro**, et d'autant plus fort que $s$ est grand.

L'intuition est saine : quand le réseau RBF hésite — parce que l'**état** $x$ est lui-même
incertain, ce qui rend $v = \pi_{\text{brut}}(x)$ incertain — PILCO **n'applique pas un couple
agressif au hasard**. Il modère l'action. C'est une **prudence sous incertitude qui émerge
gratuitement** de la forme $\sin$, sans qu'on l'ait codée à la main : plus on est dans le flou,
plus on agit doucement.

> **À retenir.** Le `sin` n'est pas qu'un « clip lisse » : ses moments fermés ferment la chaîne
> analytique de bout en bout (dynamique → politique → saturation), et son facteur $e^{-s/2}$
> couple naturellement l'amplitude de l'action à la confiance qu'on a dans l'état — une
> régularisation implicite, alignée avec l'esprit de tout PILCO : *agir prudemment tant qu'on
> n'est pas sûr*.


## III.7 — La distribution jointe état–contrôle

### Le piège qu'on évite

Reprends la chaîne d'un pas : pour prédire $x_{t+1}$, le GP de dynamique a besoin de l'entrée
complète $\tilde{x} = (x, u)$ — l'état **et** l'action. On connaît la gaussienne sur l'état,
$x \sim \mathcal{N}(\mu_x, \Sigma_x)$. La question est : quelle est la gaussienne sur
$\tilde{x} = (x, u)$ ?

La tentation naïve serait de calculer séparément une gaussienne sur $x$ et une gaussienne sur $u$,
puis de les **empiler comme si elles étaient indépendantes** :

$$\begin{bmatrix} x \\ u \end{bmatrix} \overset{?}{\sim} \mathcal{N}\!\left(
\begin{bmatrix} \mu_x \\ \mu_u \end{bmatrix},\;
\begin{bmatrix} \Sigma_x & \mathbf{0} \\ \mathbf{0} & \Sigma_u \end{bmatrix}\right).$$

**C'est faux**, et c'est l'erreur que PILCO prend soin d'éviter. L'action n'est pas tirée
indépendamment de l'état : elle en est une **fonction déterministe**, $u = \pi(x)$. Toute
l'incertitude sur $u$ vient *de* l'incertitude sur $x$. Les deux sont donc **corrélées** — mettre
des zéros hors-diagonale, c'est jeter cette corrélation à la poubelle.

### Pourquoi la corrélation existe — concrètement

Imagine un pendule dont l'angle est mal connu : $x \sim \mathcal{N}(\mu_x, \Sigma_x)$ avec une
variance non négligeable. La politique $\pi$ dit « si le pendule penche à droite, pousse à
gauche ». Alors :

- les réalisations de $x$ « penché à droite » produisent des $u$ « pousse fort à gauche » ;
- les réalisations de $x$ « penché à gauche » produisent des $u$ « pousse fort à droite ».

L'action **co-varie** avec l'état de façon systématique. Connaître $x$ d'un tirage, c'est
**connaître $u$** de ce même tirage. C'est précisément ce que mesure le bloc croisé $C_{xu}$ :

$$C_{xu} = \operatorname{Cov}(x, u) = \mathbb{E}\big[(x - \mu_x)(u - \mu_u)^\top\big].$$

> **Analogie de la route floue.** Si ta position sur la route est incertaine, ta décision de
> braquer l'est aussi — mais pas indépendamment : tu braques *parce que* tu te crois à tel
> endroit. Les deux flous sont attelés l'un à l'autre. Modéliser le volant comme un bruit
> indépendant de la position serait absurde ; modéliser $u$ indépendamment de $x$ l'est tout
> autant.

### La gaussienne jointe que PILCO construit

PILCO assemble donc le bloc complet, **corrélations incluses** :

$$\begin{bmatrix} x \\ u \end{bmatrix} \sim \mathcal{N}\!\left(
\begin{bmatrix} \mu_x \\ \mu_u \end{bmatrix},\;
\begin{bmatrix} \Sigma_x & C_{xu} \\ C_{xu}^\top & \Sigma_u \end{bmatrix}\right).$$

Chaque bloc se calcule analytiquement, en réutilisant les outils déjà en place :

| Bloc | D'où il vient |
|------|---------------|
| $\mu_u$ | moyenne de l'action : moment matching à travers le RBF (III.5) puis $\mathbb{E}[\sin v]$ (III.6) |
| $\Sigma_u$ | variance de l'action : $\operatorname{Var}[\sin v]$ propagée à travers le RBF |
| $C_{xu}$ | covariance croisée : via $\operatorname{Cov}[v, \sin v]$ et la covariance état–sortie-RBF |

C'est pour ça que le $\operatorname{Cov}[v, \sin v]$ de III.6 n'était pas un détail décoratif :
c'est exactement la pièce qui permet de remplir $C_{xu}$. Sans elle, on n'aurait que la diagonale.

### Pourquoi $C_{xu}$ change vraiment le résultat

Quand on injecte $\tilde{x} = (x, u)$ dans le GP de dynamique (III.3), le moment matching utilise
la covariance **complète** $\Sigma_{\tilde{x}}$, y compris $C_{xu}$. Le GP « voit » donc que
certaines combinaisons $(x, u)$ sont plausibles et d'autres non. Reprends le pendule :

- **Avec $C_{xu}$** : le GP sait que « penché à droite » va avec « poussée à gauche ». Ces paires
  cohérentes décrivent un contrôle qui **corrige** — la dynamique propagée prédit un retour vers
  l'équilibre, avec une incertitude réaliste.
- **Sans $C_{xu}$** (en mettant $\mathbf 0$) : le GP considère comme également probables des paires
  **incohérentes** — « penché à droite » *et* « poussée à droite ». Ces combinaisons que la
  politique ne produirait jamais gonflent artificiellement la variance prédite et **biaisent** la
  trajectoire d'incertitude. L'estimation du coût attendu $J(\pi)$ devient fausse, et donc son
  gradient aussi.

Autrement dit : ignorer $C_{xu}$ ne fait pas qu'« approximer un peu plus » — ça injecte des
scénarios physiquement impossibles dans la prédiction, ce qui peut rendre l'optimisation de la
politique instable ou la pousser vers de mauvais optima.

> **À retenir.** L'état incertain rend l'action incertaine **et corrélée** à lui ; $u = \pi(x)$
> est déterministe, donc tout le flou de $u$ est hérité de $x$. PILCO construit la gaussienne
> jointe $(x, u)$ avec son bloc croisé $C_{xu}$ — bâti à partir du $\operatorname{Cov}[v,\sin v]$
> de III.6 — pour que le GP de dynamique ne raisonne **que sur des paires état–action que la
> politique produirait réellement**. C'est ce qui garde la propagation d'incertitude honnête de
> bout en bout.


## III.7 bis — Pour la stabilisation locale : simplifier la politique, pas PILCO

La politique RBF précédente est utile quand le contrôle doit être franchement **non linéaire**, par
exemple pour balancer un pendule depuis le bas puis le stabiliser. `InvertedPendulum-v5` pose une tâche
différente : le reset est déjà proche de la verticale et il faut appliquer un **retour d'état local**
avant que la petite perturbation ne diverge.

Optimiser ici une RBF complète reviendrait à ajuster près de deux cents paramètres avec une centaine de
transitions, alors que six paramètres suffisent. PILCO n'impose pas une architecture précise : il exige
surtout que les moments de la politique sous une entrée gaussienne soient calculables. On choisit donc

$$
v = Kx + b,
\qquad
u = u_{\max}\sin(v).
$$

La partie affine reste exactement gaussienne :

$$
\mu_v = K\mu_x+b,
\qquad
\Sigma_v = K\Sigma_xK^\top,
\qquad
\operatorname{cov}(x,v)=\Sigma_xK^\top.
$$

On applique ensuite les mêmes moments fermés du sinus que pour la RBF. Le **moteur PILCO ne change
pas** : GP probabiliste, propagation de croyance, coût attendu et L-BFGS restent identiques. Seule la
famille de contrôleurs est adaptée à la géométrie locale de la tâche. La RBF reste le choix naturel pour
un futur cart-pole *swing-up*.

In [ ]:
class LinearSinePolicy(torch.nn.Module):
    """Contrôleur local borné u = u_max sin(Kx + b), analytique sous x gaussien."""
    def __init__(self, state_dim, action_dim, action_high=3.0):
        super().__init__()
        self.gain = torch.nn.Parameter(torch.zeros(action_dim, state_dim))
        self.bias = torch.nn.Parameter(torch.zeros(action_dim))
        self.action_high = torch.as_tensor(action_high, dtype=torch.float64).reshape(-1)

    def raw_control(self, x):
        return x @ self.gain.T + self.bias

    def forward(self, x):
        return self.action_high * torch.sin(self.raw_control(x))

    def propagate_gaussian(self, mu_x, sigma_x):
        # Une transformation affine d'une gaussienne reste exactement gaussienne.
        m_raw = self.gain @ mu_x + self.bias
        s_raw = self.gain @ sigma_x @ self.gain.T
        c_raw = sigma_x @ self.gain.T

        # Moments fermés de sin(v) : même saturation bornée que la politique RBF.
        var_raw = torch.diagonal(s_raw)
        exp_half = torch.exp(-0.5 * var_raw)
        mu_u = self.action_high * exp_half * torch.sin(m_raw)
        var_u = self.action_high ** 2 * (
            0.5 * (1.0 - torch.exp(-2.0 * var_raw) * torch.cos(2.0 * m_raw))
            - torch.exp(-var_raw) * torch.sin(m_raw).square()
        )
        local_gain = self.action_high * exp_half * torch.cos(m_raw)
        sigma_u = torch.outer(local_gain, local_gain) * s_raw
        sigma_u = sigma_u - torch.diag(torch.diagonal(sigma_u)) + torch.diag(var_u)
        return mu_u, sigma_u, c_raw * local_gain.unsqueeze(0)


def policy_propagate(policy, mu_x, sigma_x):
    # InvertedPendulum est localement linéaire : sa politique finale a des moments exacts.
    if isinstance(policy, LinearSinePolicy):
        return policy.propagate_gaussian(mu_x, sigma_x)

    # Cas général enseigné plus haut : moments d'une politique RBF saturée.
    C, W, ell = policy.centers, policy.weights, torch.exp(policy.log_ell)
    Dx, Du = mu_x.shape[0], W.shape[1]; eye = torch.eye(Dx)
    zeta = C - mu_x; lam = torch.diag(ell ** 2)
    SpL_inv = torch.linalg.inv(sigma_x + lam)
    q = torch.exp(0.5 * torch.logdet(lam) - 0.5 * torch.logdet(sigma_x + lam)
                  - 0.5 * ((zeta @ SpL_inv) * zeta).sum(1))
    m_raw = W.t() @ q
    Cxr = torch.stack([sigma_x @ SpL_inv @ ((W[:, k] * q).unsqueeze(1) * zeta).sum(0)
                       for k in range(Du)], dim=1)
    ia = torch.diag(1.0 / ell ** 2); R_inv = torch.linalg.inv(sigma_x @ (2 * ia) + eye)
    za = zeta @ ia; na = (za * zeta).sum(1); rs = R_inv @ sigma_x
    quad = ((za @ rs) * za).sum(1)[:, None] + 2 * (za @ rs @ za.t()) + ((za @ rs) * za).sum(1)[None, :]
    Qm = torch.exp(-0.5 * torch.logdet(sigma_x @ (2 * ia) + eye) - 0.5 * na[:, None] - 0.5 * na[None, :] + 0.5 * quad)
    s_raw = torch.zeros(Du, Du)
    for k in range(Du):
        for l in range(k, Du):
            s_raw[k, l] = s_raw[l, k] = W[:, k] @ Qm @ W[:, l] - m_raw[k] * m_raw[l]
    umax = policy.action_high; v = torch.diagonal(s_raw); eh = torch.exp(-0.5 * v)
    mu_u = umax * eh * torch.sin(m_raw)
    var_u = umax ** 2 * (0.5 * (1 - torch.exp(-2 * v) * torch.cos(2 * m_raw)) - torch.exp(-v) * torch.sin(m_raw) ** 2)
    gain = umax * eh * torch.cos(m_raw)
    sigma_u = torch.outer(gain, gain) * s_raw
    sigma_u = sigma_u - torch.diag(torch.diagonal(sigma_u)) + torch.diag(var_u)
    return mu_u, sigma_u, Cxr * gain.unsqueeze(0)


linear_demo = LinearSinePolicy(2, 1, action_high=3.0)
with torch.no_grad():
    linear_demo.gain.copy_(torch.tensor([[1.2, -0.7]]))
states_demo = torch.tensor([[-1.0, 0.0], [0.0, 0.0], [1.0, 0.0]])
actions_demo = linear_demo(states_demo).detach().squeeze()
print("actions du contrôleur local :", np.round(actions_demo.numpy(), 3))
assert actions_demo[0] < 0 < actions_demo[2]
assert bool((actions_demo.abs() <= 3.0 + 1e-12).all())
print("✓ politique linéaire bornée : feedback visible et moments gaussiens exacts")

**Lecture.** Le nuage $(x, u)$ n'est pas un blob isotrope : il suit la forme de la politique
— état et action sont **corrélés**. C'est cette corrélation $C_{xu}$ que la gaussienne jointe
capture, et qu'il faut transmettre au GP de dynamique. On a maintenant **toutes les pièces** pour
faire avancer la croyance d'**un pas**.

## III.8 — Un pas de propagation : assembler le tout

Un pas de PILCO enchaîne trois opérations analytiques :

1. **Politique** : $\mathcal{N}(\mu_x, \Sigma_x) \to (\mu_u, \Sigma_u, C_{xu})$ *(policy_propagate)*.
2. **Jointe** : assembler $\mathcal{N}(\mu_{(x,u)}, \Sigma_{(x,u)})$.
3. **Dynamique** : moment matching du GP sur cette jointe $\to (M_{\Delta}, S_{\Delta}, C_{(x,u),\Delta})$,
   puis $x' = x + \Delta$ :

$$\mu_{x'} = \mu_x + M_\Delta, \qquad
\Sigma_{x'} = \Sigma_x + S_\Delta + C_{x\Delta} + C_{x\Delta}^\top$$

(le terme $C_{x\Delta}$, bloc « état » de la cross-covariance, vient de la corrélation entre $x$ et
son propre incrément $\Delta$). Vérifions ce pas contre Monte-Carlo.

In [ ]:
def _psd_project(sigma, floor=1e-6):
    # Symétriser puis ajouter le jitter minimal pour rester semi-définie positive.
    sigma = 0.5 * (sigma + sigma.t())
    sigma = torch.nan_to_num(sigma, nan=0.0, posinf=1e6, neginf=-1e6)
    eye = torch.eye(sigma.shape[-1], dtype=sigma.dtype, device=sigma.device)
    with torch.no_grad():
        try:
            lam_min = torch.linalg.eigvalsh(sigma).min()
        except Exception:
            lam_min = torch.tensor(-1.0, dtype=sigma.dtype)
    return sigma + torch.clamp(floor - lam_min, min=0.0) * eye


def project_ip_belief(mu, sigma):
    """Projette la moyenne ET la covariance sur sin²(theta)+cos²(theta)=1."""
    s, c = mu[1], mu[2]
    radius_sq = (s.square() + c.square()).clamp_min(1e-8)
    radius = torch.sqrt(radius_sq)
    mu_projected = torch.cat([mu[:1], (s / radius).reshape(1), (c / radius).reshape(1), mu[3:]])

    # Jacobienne de (s,c) -> (s/r,c/r), pour transporter aussi la covariance.
    radius_cubed = radius_sq * radius
    block = torch.stack([
        torch.stack([c.square() / radius_cubed, -s * c / radius_cubed]),
        torch.stack([-s * c / radius_cubed, s.square() / radius_cubed]),
    ])
    jacobian = torch.eye(mu.shape[0], dtype=mu.dtype, device=mu.device)
    jacobian[1:3, 1:3] = block
    sigma_projected = jacobian @ sigma @ jacobian.T
    return mu_projected, _psd_project(sigma_projected)


def propagate_step(dyn_gp, policy, mu_x, sigma_x):
    Dx = mu_x.shape[0]
    mu_u, sigma_u, c_xu = policy_propagate(policy, mu_x, sigma_x)
    mu_j = torch.cat([mu_x, mu_u])
    sigma_j = torch.cat([torch.cat([sigma_x, c_xu], 1), torch.cat([c_xu.t(), sigma_u], 1)], 0)
    sigma_j = _psd_project(sigma_j)
    M, S, Cjd = gaussian_moments(dyn_gp, mu_j, sigma_j)
    c_xd = Cjd[:Dx, :]
    mu_next = mu_x + M
    sigma_next = sigma_x + S + c_xd + c_xd.t()
    return mu_next, _psd_project(sigma_next)


mu_bad = torch.tensor([0.0, 0.3, 0.4, 0.0, 0.0])
sigma_bad = torch.diag(torch.tensor([0.01, 0.03, 0.02, 0.01, 0.01]))
mu_fixed, sigma_fixed = project_ip_belief(mu_bad, sigma_bad)
assert abs(float(mu_fixed[1].square() + mu_fixed[2].square()) - 1.0) < 1e-10
assert torch.linalg.eigvalsh(sigma_fixed).min() > 0
print("✓ projection de croyance OK : moyenne sur le cercle et covariance transportée")

**Lecture.** Le pas analytique reproduit le Monte-Carlo (à l'erreur d'échantillonnage près),
**sans jamais simuler** : tout est calculé en forme fermée et reste différentiable par rapport aux
paramètres de la politique. C'est ce qui rend PILCO si efficace. Il ne reste qu'à **enchaîner**
ces pas sur tout l'horizon, sommer le coût espéré, et **optimiser la politique** — la boucle
complète.

# Partie IV — La boucle PILCO complète

Toutes les pièces sont en place : un GP de dynamique entraîné par vraisemblance marginale (I),
propagé analytiquement par moment matching (III.1–III.3), un coût saturant dont l'espérance est en
forme close (III.4), une politique RBF intégrable (III.5) bornée par un sinus (III.6), et la
gaussienne jointe état–action qui garde la propagation honnête (III.7). Il reste à les **assembler
en une boucle**.

## IV.1 — Les trois verbes : Observer → Imaginer → Ajuster

PILCO alterne entre le monde réel et un monde imaginé. Trois verbes résument tout :

- **Observer** — exécuter la politique courante **une fois** dans l'environnement réel, et
  collecter le ruban de transitions $(x_t, u_t, x_{t+1})$. C'est la **seule** interaction physique
  de l'itération. C'est rare, donc précieux.
- **Imaginer** — sans plus toucher au monde, *rêver* ce que ferait la politique : partir de
  $\mathcal{N}(\mu_0, \Sigma_0)$ et propager la croyance sur $H$ pas via le GP, en accumulant le
  coût attendu $J(\pi)$.
- **Ajuster** — améliorer la politique en descendant le gradient de ce coût imaginé, $\nabla_\pi
  J$, obtenu par autograd à travers toute la propagation. Puis retourner observer.

$$
\boxed{
    \begin{aligned}
    &\textbf{répéter pour chaque itération :}\\
    &\quad 1.\ \textbf{Observer : exécuter } \pi \text{ dans l'environnement réel} \to \text{nouvelles transitions}\\
    &\quad 2.\ \text{ré-entraîner le GP de dynamique sur } \textit{toutes} \text{ les données (vraisemblance marginale)}\\
    &\quad 3.\ \textbf{Imaginer} : J(\pi) = \sum_{t=1}^{H} \mathbb{E}[c(x_t)] \text{ par propagation analytique de la croyance}\\
    &\quad 4.\ \textbf{Ajuster} : \min_\pi J(\pi) \text{ par L-BFGS (gradient via autograd à travers la propagation)}
    \end{aligned}
}
$$

## IV.2 — Le pseudo-code, étape par étape

<!-- ```text
Entrées : coût c(·), horizon H, n_itérations
Initialiser π (poids RBF aléatoires), D ← ∅          # aucune donnée au départ

# — Amorçage : une trajectoire sous politique aléatoire pour démarrer le GP —
D ← D ∪ rollout_réel(π_aléatoire)

répéter n_itérations fois :
    # ===== (2) APPRENTISSAGE DU MODÈLE =====
    pour chaque dimension de sortie e :
        optimiser (ℓ, σ_f, σ_n)_e  en maximisant la NLML sur TOUT D   # I.9
    pré-calculer β = (K+σ²I)⁻¹y  et la Cholesky L   # I.10, réutilisés en boucle interne

    # ===== (3+4) OPTIMISATION DE LA POLITIQUE (100 % imaginée) =====
    répéter jusqu'à convergence de L-BFGS :          # BOUCLE INTERNE
        (μ, Σ) ← (μ₀, Σ₀)
        J ← 0
        pour t = 1 … H :
            (μ_u, Σ_u, C_xu) ← politique_RBF + sin(μ, Σ)        # III.5, III.6, III.7
            (μ̃, Σ̃)          ← joint_état_action(μ, Σ, μ_u, Σ_u, C_xu)
            (μ, Σ)            ← moment_matching_GP(μ̃, Σ̃)        # III.3 — un pas de dynamique
            J ← J + E[c(x_t)]  sous N(μ, Σ)                      # III.4 — coût attendu
        g ← ∇_π J   via autograd à travers TOUTE la boucle t
        π ← pas_LBFGS(π, J, g)

    # ===== (1) OBSERVATION RÉELLE =====
    D ← D ∪ rollout_réel(π)        # une seule interaction physique par itération

renvoyer π
``` -->
$$
\boxed{
\begin{array}{ll}
\textbf{Algorithme — PILCO} & \\[2pt]
\textbf{Entrées : } \text{coût } c(\cdot),\ \text{horizon } H,\ n_{\text{iter}} & \\
\textbf{Initialiser : } \pi\ (\text{poids RBF aléatoires}),\ \ \mathcal{D} \leftarrow \varnothing & \\[4pt]
\triangleright\ \textit{Amorçage : trajectoire sous politique aléatoire} & \\
\mathcal{D} \leftarrow \mathcal{D}\,\cup\,\text{rollout}_{\text{réel}}(\pi_{\text{alea}}) & \\[4pt]
\textbf{répéter } n_{\text{iter}} \textbf{ fois :} & \\[2pt]
\quad \triangleright\ \textbf{(2) Apprentissage du modèle} & \\
\quad \textbf{pour } \text{chaque dimension de sortie } e: & \\
\quad\quad \text{optimiser } (\ell,\sigma_f,\sigma_n)_e \text{ : maximiser la NLML sur } \mathcal{D} & \text{[I.9]} \\
\quad \text{pré-calculer } \boldsymbol\beta=(K+\sigma_n^2 I)^{-1}\mathbf{y},\ \text{Cholesky } L & \text{[I.10]} \\[4pt]
\quad \triangleright\ \textbf{(3+4) Optimisation politique } (100\%\ \text{imaginée}) & \\
\quad \textbf{répéter } \text{jusqu'à convergence L-BFGS :} & \\
\quad\quad (\mu,\Sigma) \leftarrow (\mu_0,\Sigma_0),\ \ J \leftarrow 0 & \\
\quad\quad \textbf{pour } t = 1 \ldots H: & \\
\quad\quad\quad (\mu_u,\Sigma_u,C_{xu}) \leftarrow \text{RBF}+\sin\,(\mu,\Sigma) & \text{[III.5–7]} \\
\quad\quad\quad (\tilde\mu,\tilde\Sigma) \leftarrow \text{joint}_{x,u}(\mu,\Sigma,\mu_u,\Sigma_u,C_{xu}) & \text{[III.7]} \\
\quad\quad\quad (\mu,\Sigma) \leftarrow \text{moment-matching}_{\text{GP}}(\tilde\mu,\tilde\Sigma) & \text{[III.3]} \\
\quad\quad\quad J \leftarrow J + \mathbb{E}_{\,x_t\sim\mathcal{N}(\mu,\Sigma)}[\,c(x_t)\,] & \text{[III.4]} \\
\quad\quad g \leftarrow \nabla_\pi J \ \ (\text{autograd sur toute la boucle } t) & \\
\quad\quad \pi \leftarrow \text{pas-LBFGS}(\pi, J, g) & \\[4pt]
\quad \triangleright\ \textbf{(1) Observation réelle} & \\
\quad \mathcal{D} \leftarrow \mathcal{D}\,\cup\,\text{rollout}_{\text{réel}}(\pi)\ \ (\text{1 interaction}) & \\[4pt]
\textbf{renvoyer } \pi & \\
\end{array}
}
$$

Le détail à ne pas manquer : il y a **deux boucles imbriquées de natures différentes**.

| | Boucle **externe** (itérations) | Boucle **interne** (L-BFGS) |
|---|---|---|
| Ce qu'elle fait | améliore le **modèle** | améliore la **politique** |
| Touche au monde réel ? | oui, **une fois** par tour (étape 1) | **jamais** — tout est imaginé |
| Coût dominant | ré-entraînement du GP | propagation sur $H$ pas + autograd |
| Nombre de tours | ~10–20 essais réels suffisent | des dizaines/centaines d'itérations L-BFGS |

C'est cette structure — **beaucoup** d'optimisation imaginée par **une seule** interaction réelle —
qui donne à PILCO son efficacité-données légendaire.

## IV.3 — Pourquoi le gradient est « gratuit »

Le point qui fait tout fonctionner : **chaque** opération de la boucle interne est une fonction
*lisse et fermée* des paramètres de la politique $\theta$.

$$\theta \;\to\; (\mu_u, \Sigma_u, C_{xu}) \;\to\; (\tilde\mu, \tilde\Sigma) \;\to\; (\mu_{t+1}, \Sigma_{t+1}) \;\to\; \mathbb{E}[c(x_{t+1})] \;\to\; J.$$

Aucune de ces flèches ne contient d'échantillonnage, de `if`, ni d'intégrale non résolue : que
des produits matriciels, des exponentielles, des déterminants. Le graphe de calcul reliant
$\theta$ à $J$ est donc **entièrement différentiable**. On obtient

$$\nabla_\theta J = \frac{\partial J}{\partial \theta}$$

par simple rétropropagation à travers les $H$ pas — exactement comme on rétropropage à travers les
couches d'un réseau, sauf qu'ici les « couches » sont les pas de temps de la trajectoire imaginée.
C'est ce qui autorise L-BFGS (un optimiseur **du second ordre**, défini en I.9) à
converger en peu d'itérations, sans gradients bruités.

> **Contraste avec le RL classique.** Un policy-gradient à la REINFORCE estime $\nabla_\theta J$
> par échantillonnage de trajectoires réelles → gradients **à forte variance**, des milliers
> d'épisodes nécessaires. PILCO calcule $\nabla_\theta J$ **analytiquement sur un modèle** →
> gradient **exact** (vis-à-vis du modèle), une poignée d'essais réels suffisent. Le prix : tout
> repose sur la qualité du GP, et l'approximation gaussienne (III.2) doit rester valide.

## IV.4 — Ce que ça coûte, et où ça casse

**Efficacité-données — la grande force.** PILCO a résolu le pendule et le cart-pole en
**quelques dizaines de secondes d'interaction réelle**, là où un policy-gradient sans modèle en
demande des heures. La raison est structurelle : on **réutilise chaque transition à l'infini**
dans la boucle interne, au lieu de la jeter après un gradient.

**Coût de calcul — la rançon.** Tout n'est pas gratuit :

- Le moment matching coûte $O(n^2)$ par pas et par dimension ($n$ = taille de $D$), et $D$
  **grossit** à chaque itération externe. Les GP exacts scalent en $O(n^3)$ à l'entraînement →
  PILCO devient lourd au-delà de quelques milliers de transitions.
- La boucle interne réévalue toute la propagation à **chaque** pas L-BFGS.

C'est pourquoi PILCO vise des problèmes **à faible dimension et peu de données** (pendule,
cart-pole), pas des espaces d'état de grande dimension.

**Où l'approximation casse.** Deux hypothèses peuvent lâcher :

1. **La gaussianité** (III.2). Si la politique envoie l'agent dans des régions très non linéaires,
   $p(x_t)$ devient franchement multimodale et le moment matching, qui ne garde que deux moments,
   sous-estime la réalité.
2. **La confiance dans le GP**. Si le modèle est sur-confiant dans une région peu visitée,
   l'optimiseur peut **exploiter cette erreur de modèle** — trouver une politique magnifique *en
   imagination* mais nulle dans le monde réel. (C'est précisément ce que la prise en compte
   honnête de l'incertitude — variance épistémique en III.3, prudence du coût saturant en III.4 —
   cherche à atténuer.)

> **À retenir.** PILCO = *un modèle probabiliste honnête + une propagation analytique
> différentiable + une optimisation par gradient sur le modèle imaginé*. La boucle externe collecte
> très peu de données réelles ; la boucle interne en extrait le maximum en optimisant la politique
> entièrement dans la tête. L'efficacité-données vient de là, son coût cubique et sa fragilité face
> à la non-gaussianité aussi.


## IV.5 — Aligner l'imagination sur la vraie fin d'épisode

Le coût saturant PILCO mesure la proximité de l'upright, mais Gymnasium impose en plus une règle dure :
l'épisode se termine dès que $|\theta|>0.2$. Une imagination qui ignorerait cette frontière pourrait
accepter de la franchir puis de « récupérer » plus tard, alors que le vrai simulateur aurait déjà
arrêté l'épisode.

Autour de la verticale, $\sin\theta$ est monotone et presque égal à $\theta$. Pour une croyance
gaussienne sur la coordonnée $s=\sin\theta$, on calcule donc le risque

$$
p_{\mathrm{fail}}
=
1-\left[
\Phi\!\left(\frac{\sin(0.2)-\mu_s}{\sigma_s}\right)
-
\Phi\!\left(\frac{-\sin(0.2)-\mu_s}{\sigma_s}\right)
\right].
$$

L'objectif final additionne le coût saturant attendu, ce risque terminal et une très petite pénalité de
commande :

$$
J(\pi)=\sum_{t=1}^{H}
\left(
\mathbb{E}[c(x_t)]
+10\,p_{\mathrm{fail},t}
+10^{-4}\,\mathbb{E}[u_t^2]
\right).
$$

Ce terme n'apprend aucune dynamique au contrôleur et ne lui donne pas la solution. Il traduit simplement
dans l'imagination une règle déjà connue de l'environnement. Ainsi, une baisse de $J$ doit correspondre
à une probabilité plus faible de chute réelle, pas à une récupération fictive après terminaison.

In [ ]:
def termination_probability(mu, sigma, angle_limit=0.2):
    """Approxime P(|theta| > limite) via sin(theta), valide autour de l'upright."""
    threshold = math.sin(angle_limit)
    mean_sin = mu[1]
    std_sin = torch.sqrt(sigma[1, 1].clamp_min(1e-10))
    normal = torch.distributions.Normal(
        torch.zeros((), dtype=mu.dtype, device=mu.device),
        torch.ones((), dtype=mu.dtype, device=mu.device),
    )
    p_safe = normal.cdf((threshold - mean_sin) / std_sin) - normal.cdf((-threshold - mean_sin) / std_sin)
    return (1.0 - p_safe).clamp(0.0, 1.0)


def inverted_pendulum_expected_cost(mu, sigma, mu_u=None, sigma_u=None):
    # Cœur PILCO : coût saturant attendu, moyenne et incertitude incluses analytiquement.
    task_cost = expected_cost(mu, sigma, TARGET, W_COST)
    # Adaptation Gymnasium : franchir |theta|=0.2 termine réellement l'épisode.
    failure_risk = termination_probability(mu, sigma)
    torque_cost = torch.zeros((), dtype=mu.dtype, device=mu.device)
    if mu_u is not None and sigma_u is not None:
        torque_cost = mu_u.square().sum() + torch.diagonal(sigma_u).sum().clamp_min(0.0)
    return task_cost + 10.0 * failure_risk + 1e-4 * torque_cost


def propagate_step_projected(dyn_gp, policy, mu, sigma):
    mu_next, sigma_next = propagate_step(dyn_gp, policy, mu, sigma)
    return project_ip_belief(mu_next, sigma_next)


def predict_trajectory(dyn_gp, policy, mu0, sigma0, horizon, target=None, W=None):
    # Chaque action paie le coût de l'état suivant qu'elle produit.
    del target, W
    mu, sig = project_ip_belief(mu0, sigma0)
    total = torch.zeros((), dtype=mu0.dtype)
    means = [mu]
    for _ in range(horizon):
        mu_u, sigma_u, _ = policy_propagate(policy, mu, sig)
        mu, sig = propagate_step_projected(dyn_gp, policy, mu, sig)
        total = total + inverted_pendulum_expected_cost(mu, sig, mu_u, sigma_u)
        means.append(mu)
    return total, means


def sample_ip_reset_beliefs(n_starts, *, seed, dtype=torch.float64):
    # Reset officiel : chaque qpos/qvel est uniforme dans [-0.01, 0.01].
    rng = np.random.default_rng(seed)
    reset_variance = (0.02 ** 2) / 12.0
    beliefs = []
    for _ in range(n_starts):
        raw = rng.uniform(low=-0.01, high=0.01, size=4)
        obs = torch.tensor(encode_ip_obs(raw), dtype=dtype)
        covariance = torch.diag(torch.tensor(
            [reset_variance, reset_variance, 1e-8, reset_variance, reset_variance], dtype=dtype
        ))
        beliefs.append((obs, covariance))
    return beliefs


def predict_reset_distribution_cost(dyn_gp, policy, beliefs, horizon):
    return torch.stack([predict_trajectory(dyn_gp, policy, mu, sig, horizon)[0] for mu, sig in beliefs]).mean()


TARGET = torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0])
W_COST = torch.diag(torch.tensor([0.01, 100.0, 1.0, 0.01, 2.0]))
_demo_beliefs = sample_ip_reset_beliefs(6, seed=0)
J0 = predict_reset_distribution_cost(dyn_gp, LinearSinePolicy(5, 1, 3.0), _demo_beliefs, horizon=20)
print(f"coût prévu du contrôleur nul, horizon 20 : {float(J0):.3f}")

## IV.6 — Pourquoi L-BFGS plutôt que SGD ?

### Deux familles d'optimiseurs

Pour minimiser $J(\pi)$, on a le choix entre deux grandes familles, et elles ne visent pas le même
régime de problèmes.

- **Premier ordre** (SGD, Adam, RMSProp). Elles n'utilisent que le **gradient** $\nabla_\theta J$
  — la pente locale. Elles font un pas dans la direction de plus forte descente : $\theta \leftarrow
  \theta - \eta\,\nabla_\theta J$. Simples, peu de mémoire, robustes au bruit, mais **aveugles à la
  courbure** : elles ne savent pas si le terrain devant elles est une vallée étroite ou une plaine,
  donc elles avancent à pas prudents et zigzaguent dans les ravins.
- **Second ordre / quasi-Newton** (Newton, BFGS, **L-BFGS**). Elles utilisent en plus une
  information de **courbure** — la Hessienne $\nabla^2_\theta J$, qui décrit comment la pente
  *change*. Connaître la courbure permet de calculer non seulement *où* descendre mais *de combien*,
  et d'aligner le pas sur la géométrie locale. Le pas de Newton idéal est :
  $$\theta \leftarrow \theta - \big[\nabla^2_\theta J\big]^{-1}\,\nabla_\theta J.$$

### Ce qu'est L-BFGS, précisément

Le pas de Newton exact exige de former et d'inverser la Hessienne — $O(d^2)$ en mémoire, $O(d^3)$
à inverser pour $d$ paramètres. Prohibitif en général. **BFGS** contourne cela : il ne calcule
**jamais** la vraie Hessienne ; il en **reconstruit une approximation** $\hat H$ progressivement,
à partir de la suite des gradients et des pas déjà effectués (chaque couple
$(\theta_{k+1}-\theta_k,\ \nabla J_{k+1}-\nabla J_k)$ révèle un morceau de courbure le long d'une
direction). C'est l'esprit « **quasi**-Newton » : approcher la courbure sans la calculer.

**L-BFGS** = *Limited-memory* BFGS. Plutôt que de stocker la matrice $\hat H$ entière (encore
$O(d^2)$), il ne garde que les $m$ derniers couples (typiquement $m \approx 10$) et reconstruit
l'action de $\hat H^{-1}$ sur le gradient par un produit implicite — coût **linéaire** en $d$. Il
combine cela avec une **recherche linéaire** (line search) qui ajuste la longueur du pas pour
garantir une décroissance suffisante, sans avoir à régler un taux d'apprentissage à la main.

### Pourquoi c'est *exactement* le bon outil pour PILCO

Le régime de PILCO coche toutes les cases qui font gagner L-BFGS et perdre SGD :

| Propriété de $J(\pi)$ dans PILCO | Conséquence | Qui gagne |
|---|---|---|
| **Lisse et déterministe** — pas de mini-batch, $J$ vient d'un rollout analytique exact | gradient **non bruité** → la courbure estimée est fiable | L-BFGS |
| **Peu de paramètres** — quelques dizaines de poids RBF | la mémoire de courbure est bon marché et très informative | L-BFGS |
| **Chaque évaluation de $J$ est chère** — propager $(\mu,\Sigma)$ sur $H$ pas + autograd | on veut **minimiser le nombre d'évaluations**, pas le coût par pas | L-BFGS |
| **Gradient exact disponible** (autograd, IV.3) | inutile de moyenner sur des tirages bruités | L-BFGS |

Mets ça en regard du terrain où SGD/Adam brillent : **réseaux à des millions de paramètres**,
**gradients stochastiques** (mini-batchs), où la Hessienne est trop grosse pour être approchée et
où le bruit rendrait toute estimation de courbure inutile. PILCO est l'**exact opposé** :
petit, lisse, déterministe, évaluations coûteuses. Utiliser SGD ici, ce serait faire des centaines
de pas bruités et mal calibrés là où L-BFGS converge en **quelques dizaines d'évaluations** bien
dirigées.

> **Analogie.** SGD descend la montagne dans le brouillard, un petit pas à la fois, en ne sentant
> que la pente sous ses pieds — sûr mais lent, et il zigzague dans les couloirs étroits. L-BFGS
> garde une **carte mentale du relief** construite à partir des pas précédents : il sait que la
> vallée tourne, et coupe droit vers le fond. Quand le terrain est lisse et qu'on ne peut se
> permettre que peu de pas (chaque pas = un rollout analytique coûteux), c'est la carte qui gagne,
> pas la prudence.

### La nuance honnête

Une précision pour ne pas survendre : L-BFGS suppose un objectif **raisonnablement convexe
localement** et un gradient **propre**. Si $J(\pi)$ devient très non convexe (modèle GP irrégulier,
approximation gaussienne qui se dégrade), l'approximation de courbure peut pointer dans une
mauvaise direction ; la recherche linéaire rattrape généralement le coup, mais L-BFGS reste un
optimiseur **local** — il trouve un bon minimum proche de l'initialisation, pas l'optimum global.
En pratique, sur les problèmes basse dimension de PILCO (pendule, cart-pole) avec gradient exact,
ce compromis est largement gagnant.


## La Hessienne : ce qu'elle encode, et pourquoi elle change tout

### Définition — la dérivée seconde, en plusieurs dimensions

Pour une fonction d'une variable, on connaît la dérivée seconde $J''(\theta)$ : elle dit si la
courbe se creuse (convexe, $J''>0$) ou bombe (concave), et **à quel point**. La **Hessienne** est
sa généralisation en dimension $d$ : c'est la matrice $d\times d$ de toutes les dérivées secondes
croisées,

$$H = \nabla^2_\theta J, \qquad H_{ij} = \frac{\partial^2 J}{\partial \theta_i\,\partial \theta_j}.$$

- La **diagonale** $H_{ii} = \partial^2 J/\partial\theta_i^2$ dit à quelle vitesse la pente change
  quand on bouge le paramètre $i$ tout seul — la courbure « propre » de chaque axe.
- Les **termes croisés** $H_{ij}$ ($i\neq j$) disent comment bouger $\theta_j$ **modifie la pente
  selon $\theta_i$** — c'est-à-dire à quel point les paramètres sont **couplés**. S'ils sont nuls,
  les paramètres s'optimisent indépendamment ; s'ils sont grands, bouger l'un dérègle l'autre.

Localement, $J$ se lit comme une **parabole multidimensionnelle** (développement de Taylor à
l'ordre 2 autour de $\theta_0$) :

$$J(\theta) \approx J(\theta_0) + \nabla J(\theta_0)^\top (\theta-\theta_0) + \tfrac12 (\theta-\theta_0)^\top H\, (\theta-\theta_0).$$

Le gradient donne la **pente** ; la Hessienne donne la **forme du bol** dans lequel on descend.

### Ce que ses valeurs propres racontent : la géométrie du paysage

Diagonalise $H$ : ses **vecteurs propres** sont les axes principaux du bol, ses **valeurs propres**
$\lambda_1, \dots, \lambda_d$ les courbures le long de ces axes.

- **Grande $\lambda$** → direction à **forte courbure** : la pente change vite, le bol est *raide
  et étroit* dans cette direction. Il faut y faire de **petits** pas, sinon on sur-saute et on
  oscille.
- **Petite $\lambda$** → direction **plate** : le bol est *large et mou*. Il faut y faire de
  **grands** pas, sinon on rampe.

Le drame de la descente de gradient pure, c'est qu'elle applique **le même** taux $\eta$ à toutes
les directions. Si les courbures sont très inégales, aucun $\eta$ ne convient : trop grand, ça
diverge dans la direction raide ; trop petit, ça stagne dans la direction plate. Résultat — le
fameux **zigzag dans le ravin**.

### Le conditionnement : un seul nombre qui prédit la souffrance

Le rapport entre la plus grande et la plus petite courbure,

$$\kappa = \frac{\lambda_{\max}}{\lambda_{\min}},$$

est le **conditionnement** de la Hessienne. C'est lui qui gouverne la vitesse de la descente de
gradient :

- $\kappa \approx 1$ — bol presque sphérique, toutes les directions équivalentes : SGD descend
  droit, peu de pas.
- $\kappa \gg 1$ — vallée très allongée : SGD zigzague, et le nombre d'itérations explose
  (grossièrement $\propto \kappa$).

### Pourquoi le pas de Newton règle ça

Le pas de Newton multiplie le gradient par $H^{-1}$ :

$$\theta \leftarrow \theta - H^{-1}\,\nabla J.$$

Géométriquement, $H^{-1}$ **réétalonne chaque direction par sa propre courbure** : il divise la
composante du gradient selon l'axe $k$ par $\lambda_k$. Les directions raides sont raccourcies, les
directions plates allongées — exactement la correction qui manquait à SGD. Dans le repère ainsi
transformé, **le ravin allongé redevient un bol rond** ($\kappa$ effectif $\to 1$), et sur une
parabole parfaite, Newton atteint le minimum **en un seul pas**. C'est *pour ça* que connaître la
courbure vaut tant : elle dit non seulement où descendre, mais **comment dilater chaque axe** pour
que la descente soit droite.

### Le lien avec PILCO et L-BFGS

Dans PILCO, les paramètres de la politique sont des **poids RBF couplés** : bouger l'amplitude
d'une bosse modifie l'effet des bosses voisines. La Hessienne de $J(\pi)$ est donc **loin d'être
diagonale** et souvent mal conditionnée — précisément le cas où SGD rame et où l'information de
courbure est précieuse.

Mais former $H$ explicitement coûterait $O(d^2)$ en mémoire et $O(d^3)$ à inverser. **L-BFGS ne
calcule jamais $H$** : il en **devine l'action** à partir de l'histoire des gradients. L'intuition
de la **condition sécante** : entre deux itérations, la variation du gradient $\,y_k = \nabla
J_{k+1} - \nabla J_k\,$ par rapport au déplacement $\,s_k = \theta_{k+1}-\theta_k\,$ donne une
mesure *empirique* de la courbure le long de $s_k$, puisque par Taylor $\,y_k \approx H\,s_k$.
Accumuler quelques couples $(s_k, y_k)$ suffit à reconstruire une bonne approximation de $H^{-1}$
**sans jamais l'écrire** — et donc à approcher le pas de Newton au coût d'un pas de gradient.

> **À retenir.** Le gradient répond à « *dans quelle direction descendre ?* ». La Hessienne répond
> à « *de combien, dans chaque direction, vu la forme du terrain ?* ». Ses valeurs propres sont les
> courbures, leur rapport $\kappa$ prédit la lenteur de SGD, et son inverse $H^{-1}$ est le facteur
> qui redresse les ravins. L-BFGS récolte cette information gratuitement le long du chemin
> ($y_k \approx H s_k$) — d'où sa convergence rapide sur l'objectif lisse, bas-dimensionnel et
> coûteux de PILCO.

In [ ]:
def cost_curve(make_opt, lbfgs=False, n=4, seed=0):
    """Retourne la liste des coûts J(π) à CHAQUE évaluation (closure incluse)."""
    torch.manual_seed(seed)
    pol = RBFPolicy(5, 1, 20, 3.0)
    costs = []

    beliefs = sample_ip_reset_beliefs(1, seed=10_000 + seed)

    def eval_J():
        return predict_reset_distribution_cost(dyn_gp, pol, beliefs, horizon=4)

    if lbfgs:
        opt = torch.optim.LBFGS(pol.parameters(), max_iter=n,
                                line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad()
            J = eval_J()
            costs.append(float(J.detach()))      # une entrée par appel de closure
            J.backward()
            return J
        opt.step(closure)
    else:
        opt = make_opt(pol.parameters())
        for _ in range(n):
            opt.zero_grad()
            J = eval_J()
            costs.append(float(J.detach()))
            J.backward()
            opt.step()
    return costs


c_lbfgs = cost_curve(None, lbfgs=True)
c_adam  = cost_curve(lambda p: torch.optim.Adam(p, lr=0.05))
c_sgd   = cost_curve(lambda p: torch.optim.SGD(p, lr=0.02, momentum=0.9))

print(f"L-BFGS : {len(c_lbfgs)} évaluations, J {c_lbfgs[0]:.3f} -> {c_lbfgs[-1]:.3f}")
print(f"Adam   : {len(c_adam)} évaluations, J {c_adam[0]:.3f} -> {c_adam[-1]:.3f}")
print(f"SGD    : {len(c_sgd)} évaluations, J {c_sgd[0]:.3f} -> {c_sgd[-1]:.3f}")

fig, ax = plt.subplots(figsize=(8, 4.2))
for costs, style, col, lab in [
    (c_lbfgs, "o-", "crimson",    f"L-BFGS ({len(c_lbfgs)} éval.)"),
    (c_adam,  "s-", "steelblue",  f"Adam ({len(c_adam)} éval.)"),
    (c_sgd,   "^-", "seagreen",   f"SGD+momentum ({len(c_sgd)} éval.)"),
]:
    xs = range(1, len(costs) + 1)
    ax.plot(xs, costs, style, ms=4, lw=1.6, color=col, label=lab)
    ax.annotate(f"{costs[-1]:.3f}", (len(costs), costs[-1]),
                textcoords="offset points", xytext=(6, 0),
                fontsize=8, color=col, va="center")

ax.set_yscale("log")                       # L-BFGS chute de plusieurs ordres -> log indispensable
ax.set_xlabel("évaluations de $J(\\pi)$")
ax.set_ylabel("coût prévu $J(\\pi)$  (échelle log)")
ax.set_title("Convergence : L-BFGS exploite la courbure, "
             "SGD/Adam avancent à l'aveugle")
ax.grid(True, which="both", ls=":", alpha=0.4)
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** À nombre d'évaluations comparable, L-BFGS plonge le coût bien plus vite et plus
bas qu'Adam : en exploitant la courbure, il fait d'énormes progrès par évaluation — exactement ce
qu'on veut quand chaque évaluation est un rollout analytique coûteux. Le nombre d'évaluations peut
être légèrement supérieur à `max_iter`, car la *line search* de L-BFGS réappelle la `closure` pour
tester plusieurs tailles de pas avant d'accepter une mise à jour. C'est pour cela que PILCO, qui mise
tout sur l'efficacité, choisit un optimiseur quasi-Newton.

## IV.7 — Une expérience dimensionnée en temps physique

Le pas MuJoCo vaut $\Delta t = 0.04$ s. Un horizon de $12$ pas ne voit que $0.48$ s de futur :
la politique nulle a souvent le temps de *commencer* à dériver, pas encore de tomber. On planifie
donc sur **$40$ pas** ($1.6$ s), ce qui suffit à rendre la chute visible dans l'imaginaire tout en
gardant un notebook exécutable.

La recette gagnante du notebook est volontairement simple à énumérer, car chaque choix répond à un
problème empirique précis :

1. **Politique.** La politique **canonique** de PILCO reste la **RBF** pour les tâches de
   *swing-up*. Ici, la recette qui tient sur `InvertedPendulum-v5` est un
   **`LinearSinePolicy`** : feedback local pour la stabilisation, même saturation sinus et mêmes
   moments fermés que la version RBF.
2. **Objectif.** On minimise un **coût saturant attendu** pour l'upright, auquel on ajoute
   $10\,P(|\theta|>0.2)$ pour refléter la vraie terminaison Gymnasium, plus un très faible coût
   d'action $10^{-4}\,\mathbb{E}[u^2]$. Le coût principal reste donc la proximité de la verticale ;
   la barrière de chute sert seulement à empêcher l'imagination de « récupérer » après une fin
   d'épisode qui serait déjà réelle.
3. **Propagation honnête.** Après chaque pas analytique, on **reprojette la moyenne et la
   covariance** sur l'état encodé valide afin de garder $(\sin\theta, \cos\theta)$ sur le cercle
   unité au niveau des croyances, pas seulement des échantillons.
4. **Dynamique GP.** Le GP exact n'utilise jamais plus de **90 transitions** à la fois :
   **80 %** viennent de la zone « sûre » proche de l'upright, **20 %** gardent volontairement des
   dynamiques de chute. À la première itération, les hyperparamètres sont **initialisés à
   l'échelle des données** ; ensuite on les **warm-start** au lieu de repartir de zéro.
5. **Budgets.** Le mini-training tient avec **15 updates GP**, **8 itérations L-BFGS** par tour,
   **10 tours externes**, et une collecte réelle de **50 pas** à horizon fixe.
6. **Deux croyances de départ.** La politique n'est pas optimisée depuis un seul reset moyen, mais
   sur **deux croyances** tirées du reset officiel. On évite ainsi d'apprendre un correcteur trop
   spécialisé à un unique état initial.
7. **Sélection et restauration.** La sélection du meilleur contrôleur se fait sur des **seeds de
   validation fixes** ; si une politique améliore le coût imaginé mais baisse l'évaluation réelle,
   on **restore** la meilleure politique validée jusque-là.
8. **Score honnête.** Le score final n'est pas pris sur la collecte à horizon fixe. Il vient de
   **20 seeds held-out disjointes** dans l'environnement Gymnasium normal, jusqu'à $1000$ pas, avec
   terminaison naturelle.

La nuance la plus importante est la dernière : la **collecte fixed-horizon** continue la physique
après une chute pour apprendre les transitions de bord du GP ; elle n'est **jamais** utilisée comme
mesure de performance. Dans ce notebook, *collecter* et *scorer* sont deux protocoles différents,
et c'est précisément ce qui rend la lecture empirique honnête.


In [ ]:
# Garde-fou numérique pour les GP du notebook.
def _safe_gp_kernel(self, X1, X2):
    log_ell = torch.clamp(self.log_ell, -5.0, 5.0)
    log_sf = torch.clamp(self.log_sf, -5.0, 5.0)
    ell = torch.exp(log_ell)
    d = (X1[:, None, :] - X2[None, :, :]) / ell
    sqdist = torch.clamp((d ** 2).sum(-1), max=1e6)
    return torch.exp(2 * log_sf) * torch.exp(-0.5 * sqdist)

def _robust_gp_chol(self):
    n = self.X.shape[0]
    eye = torch.eye(n, dtype=self.X.dtype, device=self.X.device)
    base_noise = torch.exp(2 * torch.clamp(self.log_sn, -6.0, 2.0)) + 1e-6
    K = self.kernel(self.X, self.X)
    K = torch.nan_to_num(K, nan=0.0, posinf=1e6, neginf=0.0)
    K = 0.5 * (K + K.T)
    for jitter in [1e-6, 1e-4, 1e-2, 1.0]:
        try:
            return torch.linalg.cholesky(K + (base_noise + jitter) * eye)
        except torch.linalg.LinAlgError:
            continue
    return torch.linalg.cholesky(K + (base_noise + 10.0) * eye)

GP.kernel = _safe_gp_kernel
GP._chol = _robust_gp_chol

def clone_policy_params(policy):
    return [p.detach().clone() for p in policy.parameters()]

def restore_policy_params(policy, backup):
    with torch.no_grad():
        for param, saved in zip(policy.parameters(), backup):
            param.copy_(saved)

def policy_action(policy, encoded_obs, dtype=torch.float64, noise_std=0.0, rng=None):
    with torch.no_grad():
        action = policy(torch.tensor(encoded_obs, dtype=dtype)).detach().cpu().numpy()
    if noise_std > 0.0:
        assert rng is not None
        action = action + rng.normal(0.0, noise_std, size=action.shape)
    return np.clip(action, -3.0, 3.0)


In [ ]:
# Configuration complète du mini-training PILCO.
ENV_ID = "InvertedPendulum-v5"
PILCO_ITERATIONS = 10
INIT_RANDOM_ROLLOUTS = 3
COLLECT_HORIZON = 50
EVAL_HORIZON = 1000
MAX_GP_POINTS = 100
GP_FIT_STEPS = 20
POLICY_OPT_STEPS = 10
POLICY_HORIZON = 60
EVAL_EVERY = 1
EVAL_EPISODES = 10
EXPLORATION_NOISE_START = 0.05

torch.manual_seed(0)
np.random.seed(0)


In [ ]:
def collect_rollout(env, policy, *, seed, random=False, noise_std=0.0, max_steps=1000, dtype=torch.float64):
    raw_obs, _ = env.reset(seed=seed)
    env.action_space.seed(seed)
    obs = encode_ip_obs(raw_obs)
    X, Y, rewards, angles, actions = [], [], [], [], []
    rng = np.random.default_rng(seed)
    for _ in range(max_steps):
        if random:
            action = env.action_space.sample()
        else:
            action = policy_action(policy, obs, dtype=dtype, noise_std=noise_std, rng=rng)
        next_raw, reward, terminated, truncated, _ = env.step(action)
        next_obs = encode_ip_obs(next_raw)
        X.append(np.concatenate([obs, action]))
        Y.append(project_ip_encoded(next_obs) - obs)
        rewards.append(float(reward))
        angles.append(float(np.degrees(next_raw[1])))
        actions.append(float(action[0]))
        obs = project_ip_encoded(next_obs)
        if terminated or truncated:
            break
    return {
        "X": torch.tensor(np.asarray(X), dtype=dtype),
        "Y": torch.tensor(np.asarray(Y), dtype=dtype),
        "reward": float(np.sum(rewards)),
        "length": len(rewards),
        "angles": np.asarray(angles, dtype=np.float32),
        "actions": np.asarray(actions, dtype=np.float32),
    }

def evaluate_policy(policy, *, seeds, max_steps=1000, dtype=torch.float64):
    env_eval = gym.make(ENV_ID)
    returns, lengths, max_abs_angle = [], [], []
    try:
        for seed in seeds:
            out = collect_rollout(env_eval, policy, seed=seed, max_steps=max_steps, dtype=dtype)
            returns.append(out["reward"])
            lengths.append(out["length"])
            max_abs_angle.append(float(np.max(np.abs(out["angles"]))) if len(out["angles"]) else 0.0)
    finally:
        env_eval.close()
    return {
        "mean_reward": float(np.mean(returns)),
        "std_reward": float(np.std(returns)),
        "mean_length": float(np.mean(lengths)),
        "max_abs_angle": float(np.mean(max_abs_angle)),
    }

def evaluate_random_policy(*, seeds, max_steps=1000):
    env_eval = gym.make(ENV_ID)
    returns = []
    try:
        for seed in seeds:
            out = collect_rollout(env_eval, None, seed=seed, random=True, max_steps=max_steps)
            returns.append(out["reward"])
    finally:
        env_eval.close()
    return float(np.mean(returns)), float(np.std(returns))


In [ ]:
def fit_gp_with_cap(
    data_X, data_Y, *, max_points, fit_steps, val_X, val_Y, gp=None
):
    # 80 % des points décrivent la zone utile, 20 % gardent les dynamiques de chute.
    rng = np.random.default_rng(0)
    safe = np.flatnonzero(np.abs(data_X[:, 1].cpu().numpy()) <= np.sin(0.25))
    outside = np.setdiff1d(np.arange(len(data_X)), safe, assume_unique=True)
    n_safe = min(len(safe), int(0.8 * max_points))
    n_out = min(len(outside), max_points - n_safe)
    idx_safe = rng.choice(safe, size=n_safe, replace=False) if n_safe else np.array([], dtype=int)
    idx_out = rng.choice(outside, size=n_out, replace=False) if n_out else np.array([], dtype=int)
    selected = np.concatenate([idx_safe, idx_out])
    wanted = min(max_points, len(data_X))
    if len(selected) < wanted:
        remaining = np.setdiff1d(np.arange(len(data_X)), selected, assume_unique=False)
        selected = np.concatenate([
            selected, rng.choice(remaining, size=wanted - len(selected), replace=False)
        ])
    selected = np.sort(selected)
    X_fit, Y_fit = data_X[selected], data_Y[selected]
    finite = torch.isfinite(X_fit).all(dim=1) & torch.isfinite(Y_fit).all(dim=1)
    X_fit, Y_fit = X_fit[finite], Y_fit[finite]

    first_fit = gp is None
    gp = MultiOutputGP(6, 5) if gp is None else gp
    for dim, gp_dim in enumerate(gp.gps):
        gp_dim.set_data(X_fit, Y_fit[:, dim])
        if first_fit:
            x_scale = X_fit.std(dim=0).clamp_min(0.05)
            y_scale = Y_fit[:, dim].std().clamp_min(1e-3)
            with torch.no_grad():
                gp_dim.log_ell.copy_(torch.log(x_scale))
                gp_dim.log_sf.copy_(torch.log(y_scale))
                gp_dim.log_sn.copy_(torch.log((0.05 * y_scale).clamp_min(1e-4)))
        try:
            gp_dim.fit(steps=fit_steps if first_fit else max(6, fit_steps // 3))
        except Exception:
            gp_dim.set_data(X_fit, Y_fit[:, dim])

    # Diagnostic honnête sur le même petit holdout à toutes les itérations.
    X_val, Y_val = val_X, val_Y
    with torch.no_grad():
        pred, _ = gp.predict(X_val)
        rmse_dim = torch.sqrt(((pred - Y_val) ** 2).mean(0))
        nrmse_dim = rmse_dim / Y_val.std(0).clamp_min(1e-6)
    return gp, {
        "rmse": float(rmse_dim.mean()),
        "nrmse": float(nrmse_dim.mean()),
        "nrmse_dim": nrmse_dim.cpu().numpy(),
        "fit_points": len(X_fit),
        "val_points": len(X_val),
    }


In [ ]:
def record_ip_policy_video(policy, *, label="pilco_inverted_pendulum", seed=7, max_steps=1000, video_root="videos/10_pilco"):
    video_dir = Path(video_root) / label
    video_dir.mkdir(parents=True, exist_ok=True)
    env_video = gym.make(ENV_ID, render_mode="rgb_array")
    env_video = RecordVideo(
        env_video,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == 0,
        name_prefix=label,
    )
    try:
        rewards = collect_rollout(env_video, policy, seed=seed, max_steps=max_steps)
    finally:
        env_video.close()
    return sorted(video_dir.glob("*.mp4")), rewards


## IV.8 — Assembler PILCO sans cacher sa boucle


In [ ]:
class PILCOAgent:
    def __init__(self, policy, initial_eval_reward, optimization_beliefs):
        self.policy = policy
        self.gp = None
        self.optimization_beliefs = optimization_beliefs
        self.initial_policy = clone_policy_params(policy)
        self.best_policy = clone_policy_params(policy)
        self.best_eval_reward = float(initial_eval_reward)

    def fit_dynamics(self, data_X, data_Y, val_X, val_Y):
        self.gp, diagnostics = fit_gp_with_cap(
            data_X, data_Y,
            max_points=MAX_GP_POINTS,
            fit_steps=GP_FIT_STEPS,
            val_X=val_X,
            val_Y=val_Y,
            gp=self.gp,
        )
        return diagnostics

    def optimize_policy(self):
        backup = clone_policy_params(self.policy)
        with torch.no_grad():
            before = predict_reset_distribution_cost(
                self.gp, self.policy, self.optimization_beliefs,
                horizon=POLICY_HORIZON,
            )
        optimizer = torch.optim.LBFGS(
            self.policy.parameters(), lr=0.3, max_iter=POLICY_OPT_STEPS,
            line_search_fn="strong_wolfe",
        )

        def closure():
            optimizer.zero_grad()
            cost = predict_reset_distribution_cost(
                self.gp, self.policy, self.optimization_beliefs,
                horizon=POLICY_HORIZON,
            )
            cost.backward()
            return cost

        try:
            optimizer.step(closure)
        except Exception:
            restore_policy_params(self.policy, backup)

        with torch.no_grad():
            after = predict_reset_distribution_cost(
                self.gp, self.policy, self.optimization_beliefs,
                horizon=POLICY_HORIZON,
            )
        finite = all(torch.isfinite(parameter).all() for parameter in self.policy.parameters())
        if not finite or not torch.isfinite(after) or after > before + 1e-8:
            restore_policy_params(self.policy, backup)
            after = before
        return before, after

    def collect_rollout(self, env, **kwargs):
        return collect_rollout(env, self.policy, **kwargs)

    def evaluate(self, seeds, max_steps=EVAL_HORIZON):
        return evaluate_policy(self.policy, seeds=seeds, max_steps=max_steps)

    def keep_best_real_policy(self, evaluation):
        if evaluation["mean_reward"] > self.best_eval_reward:
            self.best_eval_reward = evaluation["mean_reward"]
            self.best_policy = clone_policy_params(self.policy)
        else:
            restore_policy_params(self.policy, self.best_policy)

    def restore_initial(self):
        restore_policy_params(self.policy, self.initial_policy)

    def restore_best(self):
        restore_policy_params(self.policy, self.best_policy)


In [ ]:
torch.manual_seed(0)
np.random.seed(0)

env = FixedHorizonNoEarlyTermination(gym.make(ENV_ID), COLLECT_HORIZON)
policy = LinearSinePolicy(5, 1, action_high=3.0)

validation_seeds = tuple(range(400, 400 + EVAL_EPISODES))
random_mean, random_std = evaluate_random_policy(seeds=validation_seeds)
initial_eval = evaluate_policy(policy, seeds=validation_seeds)
print(f"Baseline random      : {random_mean:8.1f} ± {random_std:.1f}")
print(f"Action nulle initiale: {initial_eval['mean_reward']:8.1f} ± {initial_eval['std_reward']:.1f}")

rollouts = [
    collect_rollout(env, policy, seed=10 + ep, random=True, max_steps=COLLECT_HORIZON)
    for ep in range(INIT_RANDOM_ROLLOUTS)
]
data_X = torch.cat([r["X"] for r in rollouts])
data_Y = torch.cat([r["Y"] for r in rollouts])

# Petit holdout fixe : jamais vu par le GP, identique à toutes les itérations.
holdout_rng = np.random.default_rng(123)
holdout_idx = np.sort(holdout_rng.choice(len(data_X), size=30, replace=False))
train_mask = np.ones(len(data_X), dtype=bool)
train_mask[holdout_idx] = False
model_val_X, model_val_Y = data_X[holdout_idx], data_Y[holdout_idx]
data_X, data_Y = data_X[train_mask], data_Y[train_mask]

hist = {
    "reward": [], "pred_cost_before": [], "pred_cost": [],
    "gp_val_rmse": [], "gp_val_nrmse": [], "data_size": [], "fit_points": [],
    "eval_step": [], "eval_mean": [], "eval_std": [], "eval_length": [], "gain_norm": [],
}
optimization_beliefs = sample_ip_reset_beliefs(2, seed=5000)
agent = PILCOAgent(policy, initial_eval["mean_reward"], optimization_beliefs)

pbar = tqdm(range(1, PILCO_ITERATIONS + 1), desc="Mini PILCO")
for it in pbar:
    # 1. Réajuster le même GP : hyperparamètres warm-startés, validation hors fit.
    gp_diag = agent.fit_dynamics(data_X, data_Y, model_val_X, model_val_Y)

    # 2. Optimiser puis valider la politique dans le modèle probabiliste.
    cost_before, cost_after = agent.optimize_policy()

    # 3. Une seule trajectoire réelle supplémentaire, avec bruit décroissant.
    noise = EXPLORATION_NOISE_START * max(
        0.0, 1.0 - (it - 1) / max(1, PILCO_ITERATIONS - 1)
    )
    trace = agent.collect_rollout(
        env, seed=100 + it, noise_std=noise, max_steps=COLLECT_HORIZON
    )
    data_X = torch.cat([data_X, trace["X"]])
    data_Y = torch.cat([data_Y, trace["Y"]])

    hist["reward"].append(trace["reward"])
    hist["pred_cost_before"].append(float(cost_before))
    hist["pred_cost"].append(float(cost_after))
    hist["gp_val_rmse"].append(gp_diag["rmse"])
    hist["gp_val_nrmse"].append(gp_diag["nrmse"])
    hist["data_size"].append(len(data_X) + len(model_val_X))
    hist["fit_points"].append(gp_diag["fit_points"])
    hist["gain_norm"].append(float(policy.gain.detach().norm()))

    ev = agent.evaluate(validation_seeds, max_steps=EVAL_HORIZON)
    hist["eval_step"].append(it)
    hist["eval_mean"].append(ev["mean_reward"])
    hist["eval_std"].append(ev["std_reward"])
    hist["eval_length"].append(ev["mean_length"])
    agent.keep_best_real_policy(ev)

    pbar.set_postfix(
        eval=f"{ev['mean_reward']:.0f}", J=f"{float(cost_after):.1f}",
        nrmse=f"{gp_diag['nrmse']:.2f}", gain=f"{float(policy.gain.norm()):.2f}",
    )


env.close()
heldout_seeds = tuple(range(900, 920))
agent.restore_initial()
initial_heldout_eval = evaluate_policy(policy, seeds=heldout_seeds, max_steps=EVAL_HORIZON)
agent.restore_best()
final_eval = evaluate_policy(policy, seeds=heldout_seeds, max_steps=EVAL_HORIZON)
_env_trace = gym.make(ENV_ID)
try:
    final_trace = collect_rollout(_env_trace, policy, seed=999, max_steps=EVAL_HORIZON)
finally:
    _env_trace.close()

print(
    f"Meilleure politique : held-out {final_eval['mean_reward']:.1f} ± {final_eval['std_reward']:.1f} "
    f"vs action nulle {initial_heldout_eval['mean_reward']:.1f} ± {initial_heldout_eval['std_reward']:.1f}"
)
print("Gain appris K sur [x, sinθ, cosθ, x_dot, θ_dot] :", np.round(policy.gain.detach().numpy(), 3))
print(f"Replay final seed 999 : longueur={final_trace['length']}  retour={final_trace['reward']:.1f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))

steps = np.asarray(hist["eval_step"])
means = np.asarray(hist["eval_mean"])
stds = np.asarray(hist["eval_std"])
axes[0, 0].plot(steps, means, "s-", color="seagreen", label="validation fixe")
axes[0, 0].fill_between(steps, means - stds, means + stds, color="seagreen", alpha=0.2)
axes[0, 0].axhline(initial_eval["mean_reward"], color="orange", ls=":", label="action nulle")
axes[0, 0].axhline(random_mean, color="gray", ls="--", label="random")
axes[0, 0].axhline(950, color="black", ls=":", lw=1, label="seuil 950")
axes[0, 0].set_title("Évaluation réelle avec terminaison")
axes[0, 0].set_xlabel("itération PILCO"); axes[0, 0].set_ylabel("pas avant chute")
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(hist["pred_cost_before"], "o--", color="salmon", label="avant optimisation")
axes[0, 1].plot(hist["pred_cost"], "o-", color="darkred", label="après optimisation")
axes[0, 1].set_title("Coût imaginé terminal-aware")
axes[0, 1].set_xlabel("itération"); axes[0, 1].set_ylabel(r"$J(\pi)$")
axes[0, 1].legend(fontsize=8)

axes[0, 2].plot(hist["gp_val_nrmse"], "o-", color="steelblue")
axes[0, 2].set_title("Erreur GP hors points de fit")
axes[0, 2].set_xlabel("itération"); axes[0, 2].set_ylabel("NRMSE moyenne")

axes[1, 0].plot(hist["data_size"], "o-", label="dataset réel")
axes[1, 0].plot(hist["fit_points"], "s--", label="points GP")
axes[1, 0].set_title("Budget de données")
axes[1, 0].set_xlabel("itération"); axes[1, 0].set_ylabel("transitions")
axes[1, 0].legend(fontsize=8)

axes[1, 1].plot(hist["gain_norm"], "o-", color="darkorange")
axes[1, 1].set_title("Le feedback devient non nul")
axes[1, 1].set_xlabel("itération"); axes[1, 1].set_ylabel(r"$\|K\|_2$")

axes[1, 2].plot(final_trace["angles"], color="purple", label="angle final")
axes[1, 2].axhline(0, color="green", ls="--", label="upright")
axes[1, 2].axhline(11.46, color="red", ls=":", label=r"limite $\pm0.2$ rad")
axes[1, 2].axhline(-11.46, color="red", ls=":")
axes[1, 2].set_title("Replay final : angle du pendule")
axes[1, 2].set_xlabel("pas de temps"); axes[1, 2].set_ylabel(r"angle $\theta$ (degrés)")
axes[1, 2].legend(fontsize=8)

plt.tight_layout(); plt.show()

# Une trajectoire peut être chanceuse : la carte u(theta, theta_dot) montre le feedback appris.
theta_grid = np.linspace(-0.2, 0.2, 81)
theta_dot_grid = np.linspace(-1.5, 1.5, 81)
TT, VV = np.meshgrid(theta_grid, theta_dot_grid)
encoded = np.stack([
    np.zeros_like(TT), np.sin(TT), np.cos(TT), np.zeros_like(TT), VV
], axis=-1)
with torch.no_grad():
    action_map = policy(torch.tensor(encoded, dtype=torch.float64)).squeeze(-1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].contourf(np.degrees(TT), VV, action_map, levels=25, cmap="coolwarm")
axes[0].contour(np.degrees(TT), VV, action_map, levels=[0], colors="black", linewidths=1)
axes[0].set_title(r"Feedback appris $u(\theta, \dot\theta)$")
axes[0].set_xlabel(r"$\theta$ (degrés)"); axes[0].set_ylabel(r"$\dot\theta$")
fig.colorbar(im, ax=axes[0], label="force")
axes[1].plot(final_trace["actions"], color="teal")
axes[1].axhline(3, color="black", ls=":", lw=1); axes[1].axhline(-3, color="black", ls=":", lw=1)
axes[1].set_title("Actions du replay final")
axes[1].set_xlabel("pas de temps"); axes[1].set_ylabel("force")
plt.tight_layout(); plt.show()

**Lecture des diagnostics.** Les panneaux répondent à des questions différentes :

- **Évaluation réelle** : seule preuve de contrôle. Elle doit dépasser nettement l'action nulle et la
  politique aléatoire sur les **mêmes seeds fixes** ; c'est elle qui sert à sélectionner puis
  restaurer la meilleure politique.
- **Coût imaginé avant/après** : vérifie que L-BFGS améliore bien la politique *dans le modèle*. Une
  baisse sans hausse de l'évaluation signalerait encore un biais de modèle.
- **NRMSE hors fit** : mesure la généralisation du GP, pas son interpolation des points
  d'entraînement.
- **Budget de données** : distingue toutes les transitions réelles des **90 points** effectivement
  utilisés par le GP exact.
- **Norme du gain** : prouve que l'agent apprend un feedback d'état non nul au lieu de survivre par
  hasard avec une action presque nulle.
- **Angle et carte $u(\theta,\dot\theta)$** : montrent à la fois une trajectoire concrète et la loi de
  contrôle autour de l'upright.

La lecture globale du notebook est la suivante : le mini-training n'atteint pas le plafond Gym de
$1000/1000$, mais il produit un vrai contrôleur de stabilisation, sélectionné sur validation fixe
puis confirmé sur **20 seeds held-out disjointes** à **$297.6 \pm 20.1$**. C'est assez pour prouver
l'apprentissage et pour montrer *pourquoi* cette recette marche, sans faire semblant de reproduire
la performance du PILCO papier sur une tâche différente.


In [ ]:
# Démonstration finale : replay vidéo notebook, sans fenêtre une fenêtre locale.

In [ ]:
try:
    videos, rewards = record_ip_policy_video(
        policy,
        label="pilco_inverted_pendulum",
        seed=77,
        max_steps=EVAL_HORIZON,
    )
    print(f"Démo PILCO | retour={sum(rewards):.1f} | durée={len(rewards)} pas")
    if videos:
        print("Replay vidéo PILCO :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    else:
        print("Aucune vidéo générée pour PILCO.")
except Exception as exc:
    print("Démo vidéo non disponible :", type(exc).__name__, exc)


**Lecture de la démonstration visuelle.** La vidéo part du reset officiel de `InvertedPendulum-v5`,
donc presque upright mais pas exactement stable. Ce n'est pas une tâche de swing-up : c'est une
tâche de **stabilisation continue**. La bonne lecture est simple : le chariot applique-t-il de
petites corrections pour garder $|\theta| < 0.2$ rad longtemps, ou bien la politique laisse-t-elle
l'angle diverger ?

Cette vidéo n'est qu'une **illustration** d'un seed. La preuve empirique du notebook est ailleurs :
la meilleure politique est choisie sur validation fixe, restaurée, puis mesurée sur **20 seeds
held-out disjointes**. Le résultat à retenir est donc le score multi-seed, pas l'impression laissée
par un replay isolé.


## Conclusion et ponts vers Deep PILCO / PETS

### Ce qu'on a construit

Dans ce notebook, on a construit **PILCO** comme une boucle complète de contrôle model-based :

1. **Observer** quelques trajectoires réelles ;
2. apprendre un **modèle probabiliste de dynamique** avec un GP ;
3. **imaginer** les effets d'une politique dans ce modèle en propageant l'incertitude ;
4. optimiser directement la politique avec un gradient analytique ;
5. retourner dans l'environnement réel avec une politique améliorée.

La différence centrale avec les méthodes model-free précédentes est la suivante : PILCO ne gaspille pas
les transitions réelles en ne les utilisant qu'une fois. Chaque transition nourrit le GP, puis le modèle
est réutilisé des centaines de fois pour évaluer et améliorer la politique dans l'imagination.

| Brique | Rôle |
|--------|------|
| GP de dynamique | Modélise $\Delta x = x' - x$ avec une moyenne et une incertitude |
| Kernel RBF / ARD | Encode l'hypothèse de dynamique lisse et apprend l'importance des dimensions |
| Posterior GP | Met à jour la croyance après observation des transitions |
| Moment matching | Propage une distribution d'états sans échantillonner des milliers de rollouts |
| Coût saturant | Donne un objectif différentiable et robuste près du but |
| Politique RBF | Politique continue, lisse et optimisable |
| L-BFGS | Optimise efficacement une politique quand chaque évaluation de $J(\pi)$ coûte cher |

### Ce que PILCO apporte

PILCO illustre l'idéal model-based classique : **apprendre peu dans le monde réel, réfléchir beaucoup
dans le modèle**. Sur une tâche de contrôle continu comme InvertedPendulum, même un petit nombre de
trajectoires suffit à construire un modèle local utile. Le GP ne prédit pas seulement un prochain état :
il dit aussi **où il est incertain**, et cette incertitude est propagée jusqu'au coût attendu.

C'est cette gestion explicite de l'incertitude qui distingue PILCO d'un simple modèle déterministe.
Quand le modèle ne sait pas, le coût imaginé devient plus prudent ; quand les données contraignent bien
la dynamique, la politique peut exploiter le modèle avec confiance.

### Pourquoi les résultats restent partiels ici

Le notebook montre un vrai signal d'apprentissage, mais il ne cherche pas à reproduire toute la
performance du papier original. Notre version reste volontairement pédagogique :

- le GP utilise peu de points pour garder les calculs lisibles ;
- la politique est petite ;
- l'horizon imaginé est plus court que l'horizon complet d'évaluation ;
- l'environnement Gymnasium a une terminaison abrupte, différente des setups historiques de PILCO ;
- on fait quelques itérations, pas une campagne multi-seeds longue.

Le résultat doit donc se lire comme une **preuve de mécanisme** : le modèle apprend une dynamique utile,
la politique s'améliore avec très peu d'expérience réelle, mais la résolution complète demanderait plus
de budget, plus de soin numérique et une intégration plus lourde.

### Limites structurelles de PILCO

PILCO est très efficace en échantillons, mais cette efficacité a un prix :

| Limite | Conséquence |
|--------|-------------|
| GP exact | Coût qui grandit fortement avec le nombre de données |
| Moment matching analytique | Très élégant, mais dépend de formes mathématiques particulières |
| Politique optimisée dans le modèle | Sensible aux erreurs de modèle hors distribution |
| États vectoriels compacts | Difficile à appliquer directement à des images brutes |
| Optimisation L-BFGS | Efficace, mais moins naturelle avec des modèles très stochastiques ou de grands réseaux |

Autrement dit, PILCO est excellent pour comprendre le cœur du contrôle probabiliste, mais il n'est pas
le format le plus scalable pour les grands environnements modernes.

### Pont vers la suite

La suite naturelle consiste à garder l'idée **Observer → Imaginer → Ajuster**, mais à remplacer les
briques qui ne scalent pas.

$$
\underbrace{\text{PILCO}}_{\substack{\text{GP exact} \\ \text{moment matching analytique}}}
\quad\longrightarrow\quad
\underbrace{\text{Deep PILCO}}_{\substack{\text{réseau bayésien} \\ \text{particules}}}
\quad\longrightarrow\quad
\underbrace{\text{PETS}}_{\substack{\text{ensemble probabiliste} \\ \text{MPC + CEM}}}
\quad\longrightarrow\quad
\underbrace{\text{Dreamer}}_{\substack{\text{monde latent} \\ \text{policy learning en imagination}}}
$$

Deep PILCO remplacera le GP par un réseau avec incertitude approximative et remplacera le moment
matching analytique par des particules. PETS ira encore plus loin : au lieu d'apprendre une politique
paramétrique, il planifiera à chaque pas avec MPC+CEM. Dreamer changera encore d'échelle en apprenant
un monde latent différentiable dans lequel entraîner directement un acteur et un critique.

## Références

- Deisenroth, M. P. & Rasmussen, C. E. (2011). *PILCO: A Model-Based and Data-Efficient Approach to Policy Search*. ICML. [PDF](https://icml.cc/2011/papers/323_icmlpaper.pdf).

- Rasmussen, C. E. & Williams, C. K. I. (2006). *Gaussian Processes for Machine Learning*. MIT Press. [Livre en ligne](http://www.gaussianprocess.org/gpml/).

- Pardo, F. et al. (2018). *Time Limits in Reinforcement Learning*. ICML. [arXiv:1712.00378](https://arxiv.org/abs/1712.00378).

- Gal, Y., McAllister, R. & Rasmussen, C. E. (2016). *Improving PILCO with Bayesian Neural Network Dynamics Models*. [PDF](https://www.cs.ox.ac.uk/people/yarin.gal/website/PDFs/DeepPILCO.pdf).

- Chua, K. et al. (2018). *Deep Reinforcement Learning in a Handful of Trials using Probabilistic Dynamics Models*. NeurIPS. [arXiv:1805.12114](https://arxiv.org/abs/1805.12114). (PETS)